# Application Mapping Migration — IOH Telco Data
## Schema v1 → v2: Sig App Tags + Revised Taxonomy

This notebook transforms `mis_app_category.csv` into the new schema:

| Old Column | New Column | Change |
|---|---|---|
| `app_name` | `app_name` | Unchanged — canonical display name |
| `source_app_names` | `source_app_names_old` | Renamed — BQ `nio_aggr` identifiers kept as audit reference |
| *(new)* | `sig_app_tags` | `#Application` values from Signature Library matched to this app |
| `category_1 / 2 / 3` | `category` | L1: 11-value controlled vocabulary |
| *(new)* | `subcategory` | L2: service-type principle, no role/ownership mixing |
| *(new)* | `description` | Empty — to be filled by agentic LLM pipeline |

### L2 Design Principle: **Service Type** (uniform)
All L2 values describe *what service the app provides*, not who its user is or what device it runs on.


## Section 1 — Import Libraries & Configuration

In [75]:

import pandas as pd
import numpy as np
import re
import fnmatch
from pathlib import Path
from collections import defaultdict

# ── Try to import rapidfuzz for fuzzy matching ────────────────────────────────
try:
    from rapidfuzz import fuzz, process as rfprocess
    HAS_RAPIDFUZZ = True
except ImportError:
    HAS_RAPIDFUZZ = False
    print("⚠  rapidfuzz not installed — falling back to basic substring matching.")

# ── File paths ─────────────────────────────────────────────────────────────────
BASE_DIR = Path("/Users/mac/Documents/GitHub/DS-IOH-Application-Mapping")

MIS_APP_CSV     = BASE_DIR / "mis_app_category.csv"
SIG_LIBRARY_CSV = BASE_DIR / "Signature Apps Library 20250828(Tracker v4 20240910).csv"
OUTPUT_CSV      = BASE_DIR / "rnr_app_category_v2.csv"
TAXONOMY_REF    = BASE_DIR / "taxonomy_reference.csv"

# ── Taxonomy version ───────────────────────────────────────────────────────────
TAXONOMY_VERSION = "v2.1"

# ── Label normaliser: safe snake_case ─────────────────────────────────────────
def to_safe(s: str) -> str:
    """Convert a taxonomy label to safe snake_case.
    lowercase · spaces/&// → _ · hyphens → _ · strip other specials · collapse __
    """
    s = s.lower()
    s = re.sub(r"[\s&/]+", "_", s)
    s = s.replace("-", "_")
    s = re.sub(r"[^a-z0-9_]", "", s)
    return re.sub(r"_+", "_", s).strip("_")

assert to_safe("Information & Education") == "information_education"
assert to_safe("BNPL & Pay Later")        == "bnpl_pay_later"
assert to_safe("Health & Wellness")       == "health_wellness"
assert to_safe("Productivity & Tools")    == "productivity_tools"
assert to_safe("P2P Lending")             == "p2p_lending"
assert to_safe("E-Wallet")                == "e_wallet"
assert to_safe("Telco Self-Care")         == "telco_self_care"

print(f"Taxonomy version : {TAXONOMY_VERSION}")
print(f"rapidfuzz        : {HAS_RAPIDFUZZ}")
print("to_safe() checks passed")


Taxonomy version : v2.1
rapidfuzz        : True
to_safe() checks passed


## Section 2 — Load App Tags Data

In [76]:
# ── Load mis_app_category ──────────────────────────────────────────────────────
mis = pd.read_csv(MIS_APP_CSV, dtype=str, keep_default_na=False)
mis.columns = mis.columns.str.strip()

# Fill empty strings for category columns
for col in ["category_1", "category_2", "category_3"]:
    mis[col] = mis[col].str.strip()

print(f"Loaded {len(mis):,} rows × {len(mis.columns)} columns")
print(f"\nColumns: {list(mis.columns)}")
mis.head(5)


Loaded 1,199 rows × 5 columns

Columns: ['app_name', 'source_app_names', 'category_1', 'category_2', 'category_3']


,app_name,source_app_names,category_1,category_2,category_3
0,1Cak,[1cak],Comedy Platform,Social Media,Entertainment
1,2DFire,[2DFire],Business App,,
2,360Kredi,[360KREDIT],Fintech,Payment Platform,
3,4shared,[4shared],File Sharing,,
4,7-Eleven,[7_Eleven],Grocery Platform,Shopping App,E-Commerce


In [77]:

# Category distribution analysis removed — not needed for pipeline.


In [78]:
# ── Load Signature Apps Library ────────────────────────────────────────────────
# The library has metadata rows mixed in (right-side stats columns).
# We filter to only rows where #Protocol ID is numeric.
sig_raw = pd.read_csv(
    SIG_LIBRARY_CSV,
    dtype=str,
    keep_default_na=False,
    on_bad_lines="skip",
)
sig_raw.columns = sig_raw.columns.str.strip()

# Keep only valid data rows (numeric Protocol ID)
sig = sig_raw[sig_raw["#Protocol ID"].str.match(r"^\d+$", na=False)].copy()
sig = sig[["#Protocol ID", "#Protocol", "#Application ID", "#Application", "#Application Group", "#HOT APP FLAG"]].copy()
sig["#Application"] = sig["#Application"].str.strip()
sig = sig[sig["#Application"] != ""]

# Build unique application set (avoid duplicates from multi-protocol entries)
sig_apps_unique = sig.drop_duplicates(subset=["#Application ID", "#Application"])

print(f"Signature Library: {len(sig):,} total protocol rows")
print(f"Unique #Application entries: {sig_apps_unique['#Application'].nunique():,}")
print("\nSample #Application values:")
print(sig["#Application"].value_counts().head(20).to_string())


Signature Library: 19,628 total protocol rows
Unique #Application entries: 18,481

Sample #Application values:
#Application
HTTP_Ext           12
WeChat             12
Line               12
YouKu              10
LETV                9
Vidio               8
Skype               7
MSN                 7
ICQ                 7
QQ                  7
SinaUC              7
Metacafe            7
MySpace             7
YouTube             7
Yahoo_Messenger     6
Thunder             6
Fetion              6
BlackBerry          6
Aliww               6
Kubao               6


## Section 3 — Search by Old App Tags (Exact Match)

For each row in `mis_app_category`, parse `source_app_names` into individual tags, then look up each tag against the `#Application` column of the Signature Library (case-insensitive exact match).

In [79]:
def parse_source_tags(raw: str) -> list[str]:
    """
    Parse '[TAG1,TAG2,TAG3]' or '[TAG]' into a list of stripped tag strings.
    Handles both quoted and unquoted CSV values.
    """
    if not raw or raw.strip() in ("", "[]"):
        return []
    # Remove surrounding brackets (may have been CSV-quoted)
    cleaned = raw.strip().strip('"').strip("[]")
    tags = [t.strip() for t in cleaned.split(",") if t.strip()]
    return tags


# ── Build Signature Library lookup: {tag_upper: original_application_name} ────
# Use all unique #Application values (case-insensitive key)
sig_lookup: dict[str, str] = {
    app.upper(): app
    for app in sig["#Application"].unique()
    if app.strip()
}

print(f"Signature Library lookup size: {len(sig_lookup):,} unique #Application entries")

# ── Exact-match pass ───────────────────────────────────────────────────────────
def exact_match(source_tags: list[str], lookup: dict[str, str]) -> set[str]:
    matched = set()
    for tag in source_tags:
        key = tag.upper().strip()
        if key in lookup:
            matched.add(lookup[key])
    return matched


# Parse all source tags once
mis["_parsed_tags"] = mis["source_app_names"].apply(parse_source_tags)

# Run exact match
mis["_exact_matched"] = mis["_parsed_tags"].apply(
    lambda tags: exact_match(tags, sig_lookup)
)

exact_hit = mis["_exact_matched"].apply(bool).sum()
print(f"\nExact match results:")
print(f"  Rows with ≥1 Sig Library match : {exact_hit:,} / {len(mis):,} ({exact_hit/len(mis)*100:.1f}%)")
print(f"  Rows with NO exact match        : {len(mis)-exact_hit:,}")


Signature Library lookup size: 18,466 unique #Application entries

Exact match results:
  Rows with ≥1 Sig Library match : 1,082 / 1,199 (90.2%)
  Rows with NO exact match        : 117


## Section 4 — Search by App Name Wildcard (Fallback)

For rows that had no exact match, attempt to find matching `#Application` entries using:
1. **Substring match**: `app_name.upper()` contained in any Sig Library `#Application` key (or vice-versa)
2. **Fuzzy match** (if `rapidfuzz` is available): token-set ratio ≥ 80 against all `#Application` values
3. **Source tag partial match**: check if any source tag is a prefix/suffix of a Sig Library entry

In [114]:
import re
FUZZY_THRESHOLD = 90  # raised from 80 — prevents false fuzzy matches like MOODLE↔DOODLE
COVERAGE_THRESHOLD = 0.6  # min fraction of query len a sig entry must cover to match

# Pre-build list of all sig app names for rapidfuzz
sig_app_list = list(sig_lookup.keys())   # upper-case keys
sig_app_orig  = list(sig_lookup.values()) # original-case values


def normalize_key(s: str) -> str:
    """Strip spaces, underscores, hyphens, and dots; uppercase.
    'Mobile Legends' → 'MOBILELEGENDS', 'MobileLegends' → 'MOBILELEGENDS'
    """
    return re.sub(r'[\s_\-\.]', '', s).upper()


def whole_word_in(pattern: str, text: str) -> bool:
    """True if `pattern` appears as a whole-word/whole-token sequence in `text`."""
    return bool(re.search(r'(?<!\w)' + re.escape(pattern) + r'(?!\w)', text))


def wildcard_match(app_name: str, source_tags: list[str], lookup: dict[str, str]) -> set[str]:
    """
    Fallback matching strategy:
    1a. Normalized exact: strip separators and compare (catches 'Mobile Legends' ↔ 'MobileLegends')
    1b. Forward substring: app_name contained inside a longer sig-library entry
    1c. Reverse substring (guarded): sig entry is a whole-word in app_name AND covers ≥60% of it
    2. Fuzzy (rapidfuzz): token_set_ratio >= FUZZY_THRESHOLD (90)
    3. Source-tag partial: any tag (≥5 chars) is contained inside a sig entry
    Returns a set of matched #Application original-case strings.
    """
    matched = set()
    query = app_name.upper().strip()
    query_norm = normalize_key(app_name)

    # ── Pass 1: substring / normalized matching ────────────────────────────────
    for sig_upper, sig_orig in lookup.items():
        # Skip very short sig entries to prevent noise
        if len(sig_upper) < 4:
            continue

        # 1a. Normalized exact equality — handles space/separator mismatch
        #     e.g. 'MOBILE LEGENDS' vs sig key 'MOBILELEGENDS'
        if normalize_key(sig_upper) == query_norm:
            matched.add(sig_orig)
            continue

        # 1b. Forward: app_name contained within a longer sig entry
        #     e.g. 'GRAB' found inside 'GRAB FOOD' (safe direction)
        if query in sig_upper:
            matched.add(sig_orig)
            continue

        # 1c. Reverse: sig entry is a whole-word match inside app_name,
        #     AND the sig entry is long enough relative to the app_name
        #     to rule out short word fragments like 'LEGENDS' in 'MOBILE LEGENDS'
        if (
            whole_word_in(sig_upper, query)
            and len(sig_upper) / max(len(query), 1) >= COVERAGE_THRESHOLD
        ):
            matched.add(sig_orig)

    if matched:
        return matched

    # ── Pass 2: fuzzy match ────────────────────────────────────────────────────
    if HAS_RAPIDFUZZ:
        results = rfprocess.extract(
            query,
            sig_app_list,
            scorer=fuzz.token_set_ratio,
            limit=5,
            score_cutoff=FUZZY_THRESHOLD,
        )
        for match_upper, score, idx in results:
            matched.add(sig_app_orig[idx])

    if matched:
        return matched

    # ── Pass 3: source-tag partial match ──────────────────────────────────────
    for tag in source_tags:
        tag_up = tag.upper().strip()
        if len(tag_up) < 5:   # raised from 4 — cuts short tags like 'MOKA'
            continue
        for sig_upper, sig_orig in lookup.items():
            if whole_word_in(tag_up, sig_upper):
                matched.add(sig_orig)

    return matched


# Apply wildcard fallback only to unmatched rows
unmatched_mask = ~mis["_exact_matched"].apply(bool)

print(f"Running wildcard fallback on {unmatched_mask.sum():,} unmatched rows...")

mis.loc[unmatched_mask, "_wildcard_matched"] = mis.loc[unmatched_mask].apply(
    lambda row: wildcard_match(row["app_name"], row["_parsed_tags"], sig_lookup),
    axis=1,
)
mis["_wildcard_matched"] = mis["_wildcard_matched"].apply(
    lambda x: x if isinstance(x, set) else set()
)

wc_hit = mis.loc[unmatched_mask, "_wildcard_matched"].apply(bool).sum()
print(f"Wildcard additionally matched: {wc_hit:,} rows")
print(f"Still unmatched after both passes: {unmatched_mask.sum() - wc_hit:,} rows")


Running wildcard fallback on 117 unmatched rows...
Wildcard additionally matched: 29 rows
Still unmatched after both passes: 88 rows


In [115]:
for app in ["Adidas Running", "Mobile Legends", "Moka POS", "Moodle"]:
    row = mis[mis["app_name"] == app]
    if row.empty:
        print(f"{app}: NOT FOUND"); continue
    r = row.iloc[0]
    print(f"{app}")
    print(f"  source_tags      : {r['_parsed_tags']}")
    print(f"  exact_matched    : {r['_exact_matched']}")
    print(f"  wildcard_matched : {r['_wildcard_matched']}")
    print(f"  sig_app_tags     : {r['sig_app_tags']}")
    print(f"  match_source     : {r['_match_source']}")
    print()


Adidas Running
  source_tags      : ['ADIDAS_RUNNING']
  exact_matched    : set()
  wildcard_matched : {'Adidas'}
  sig_app_tags     : Adidas
  match_source     : wildcard

Mobile Legends
  source_tags      : ['MLBangBang']
  exact_matched    : set()
  wildcard_matched : {'MobileLegends'}
  sig_app_tags     : MobileLegends
  match_source     : wildcard

Moka POS
  source_tags      : ['MOKAPOS', 'Moka']
  exact_matched    : {'MOKAPOS'}
  wildcard_matched : set()
  sig_app_tags     : MOKAPOS
  match_source     : exact

Moodle
  source_tags      : ['Moodle']
  exact_matched    : set()
  wildcard_matched : set()
  sig_app_tags     : Moodle
  match_source     : fallback_original



In [116]:
# ── Merge exact + wildcard → final sig_app_tags ───────────────────────────────
def build_sig_tags(row) -> str:
    """
    Combine exact and wildcard matches.
    If both empty, fall back to the raw parsed source tags (best-effort).
    Returns a pipe-separated string of #Application values.
    """
    combined = row["_exact_matched"] | row["_wildcard_matched"]
    if combined:
        return "|".join(sorted(combined))
    # Fallback: use original source tags as-is
    fallback = row["_parsed_tags"]
    return "|".join(sorted(fallback)) if fallback else ""


mis["sig_app_tags"] = mis.apply(build_sig_tags, axis=1)

# ── Match source metadata columns ─────────────────────────────────────────────
mis["_match_source"] = mis.apply(
    lambda r: (
        "exact"    if r["_exact_matched"]
        else "wildcard" if r["_wildcard_matched"]
        else "fallback_original"
    ),
    axis=1,
)

print("Match source distribution:")
print(mis["_match_source"].value_counts().to_string())
print(f"\nSample sig_app_tags (exact):")
print(mis[mis["_match_source"] == "exact"][["app_name", "source_app_names", "sig_app_tags"]].head(8).to_string(index=False))


Match source distribution:
_match_source
exact                1082
fallback_original      88
wildcard               29

Sample sig_app_tags (exact):
   app_name source_app_names sig_app_tags
       1Cak           [1cak]         1cak
     2DFire         [2DFire]       2DFire
   360Kredi      [360KREDIT]    360KREDIT
    4shared        [4shared]      4shared
   7-Eleven       [7_Eleven]     7_Eleven
8 Ball Pool      [8BallPool]    8BallPool
      9Apps          [9apps]        9apps
       9GAG           [9Gag]         9Gag


## Section 5 — Taxonomy Inconsistency Analysis

Audit the current `category_1/2/3` columns to surface where the classification principles mix: service type, user role, product type, ownership model, and device ecosystem.

In [83]:

# Taxonomy inconsistency audit removed — not needed for pipeline.


## Sections 6–11 — L2 Reclassification Rules

All rules use **service type** as the single L2 principle.

| Rule | Change |
|------|--------|
| **Seller Tools** | `Seller Platform` + merchant/operator apps → `Commerce > Seller Tools` |
| **Transport split** | Public transport (KAI, PELNI) → `Transportation > Public Transport & Ticketing`; couriers → `Transportation > Logistics & Delivery` |
| **Design vs Viewer** | `Editing App` → `Productivity & Tools > Design & Editing`; `Photo Viewer` → `Productivity & Tools > File Viewer & Reader` |
| **Business split** | Generic `Business App` → `Business Operations`; GitHub/VS Code/dev SDKs → `Developer Tools`; POS apps → `Business Operations` |
| **Crypto** | Crypto-first apps → `Finance > Crypto & Digital Assets` (not generic Investment) |
| **Short Video** | Social-first platforms (TikTok, Bigo) → `Communication > Short Video & Live`; operator/seller apps → `Commerce > Seller Tools` |

In [84]:
# =============================================================================
# FULL TAXONOMY DEFINITION
# L2 principle: SERVICE TYPE — what service does the app provide?
# =============================================================================

# ── Set-based overrides (app_name → (L1, L2)) — highest priority ─────────────

# Finance > Crypto & Digital Assets
CRYPTO_APPS = {
    "Binance", "Indodax", "OKX", "Gate.io", "KuCoin", "CoinMarketCap",
    "MetaMask", "Tokocrypto", "Crypto.com", "OctaFX", "IQ Option",
    "Olymp Trade", "Nanovest", "ICAP Trade",
}

# Communication > Short Video & Live  (social-first platforms)
SHORT_VIDEO_APPS = {
    "TikTok", "BIGO Live", "SnackVideo", "Likee", "Vigo Video",
    "MoboReels", "GoodShort", "ReelShort", "ShortTV", "NetShort",
    "FlickReels", "BOOYAH!", "Nimo TV",
}

# Commerce > Seller Tools  (operator/merchant-side apps)
SELLER_TOOL_APPS = {
    "TikTok Seller", "Mitra Bukalapak", "Mitra Tokopedia",
    "Big Cartel", "Shopify", "GoBiz", "iSeller",
}

# Transportation > Public Transport & Ticketing
PUBLIC_TRANSPORT_APPS = {
    "KAI Access", "KRL Access", "PELNI", "Rosalia Indah", "TractoGo",
    "Angkasa Pura I", "Angkasa Pura II",
}

# Health & Wellness > Health Wearables (device companion apps)
WEARABLE_APPS = {
    "Garmin", "Fitbit", "Huawei Health", "Google Pixel Watch",
    "Allview Watch", "Doogee Watch", "HeyTap Watch", "Huawei Watch",
    "Infinix Watch", "Lenovo Watch", "Mi Watch", "TCL Watch", "Wiko Watch",
    "Huawei Vassistant",
}

# Commerce > Fashion  (brand apparel stores)
FASHION_BRANDS = {
    "Adidas", "Nike", "Zara", "H&M", "Uniqlo", "Pull&Bear", "Bershka",
    "ASOS", "Shein", "Berrybenka", "Farfetch", "LuisaViaRoma", "Mytheresa",
    "Foot Locker", "New Balance", "Under Armour", "Louis Vuitton", "Chanel",
    "Lotte", "Bossini", "Li-Ning", "Saturdays", "Zizara", "Tuneeca",
    "Elzatta", "Zoya", "Rabbani", "BUTTONSCARVES", "Keen", "Elbina Hijab",
    "Adidas Running", "Casetify", "iStyle", "Fadkhera", "Rings",
    "Maybelline", "Guardian",
}

# Productivity & Tools > Developer Tools
DEVELOPER_TOOLS = {
    "GitHub", "Visual Studio", "Autodesk", "MathWorks", "ArcGIS",
    "Mapbox", "Xero", "Salesforce", "Zahir", "Radmin", "TeamViewer",
}

# Commerce > Automotive Market  (buy/sell vehicle marketplace)
AUTOMOTIVE_MARKET_APPS = {
    "RajaMobil", "Moladin", "Momobil", "Momotor", "Carsome", "Carmudi",
    "BCA Merchant", "Garasi.id", "QQCar", "MobilBekas", "IMOVE",
}

# Lifestyle > Automotive Owner  (brand service/companion apps)
AUTOMOTIVE_OWNER_APPS = {
    "Toyota", "Honda", "Yamaha", "Daihatsu", "Mitsubishi", "Suzuki",
    "BMW", "Audi", "Mercedes-Benz", "Ford", "Mazda", "Volkswagen",
    "Porsche", "Tesla", "BYD", "Skoda", "Wuling",
    "AUTO2000", "Astra Motor", "DAYA AUTO",
}

# Commerce > Home & Living
HOME_LIVING_APPS = {
    "IKEA", "Dekoruma", "Fabelio", "Informa", "Mitra10", "Nitori",
    "Pepperfry", "Lotte Home Shopping", "Olympic Furniture", "Philips",
}

print("Override sets loaded:")
print(f"  CRYPTO_APPS:          {len(CRYPTO_APPS)}")
print(f"  SHORT_VIDEO_APPS:     {len(SHORT_VIDEO_APPS)}")
print(f"  SELLER_TOOL_APPS:     {len(SELLER_TOOL_APPS)}")
print(f"  PUBLIC_TRANSPORT:     {len(PUBLIC_TRANSPORT_APPS)}")
print(f"  WEARABLE_APPS:        {len(WEARABLE_APPS)}")
print(f"  FASHION_BRANDS:       {len(FASHION_BRANDS)}")
print(f"  DEVELOPER_TOOLS:      {len(DEVELOPER_TOOLS)}")
print(f"  AUTOMOTIVE_MARKET:    {len(AUTOMOTIVE_MARKET_APPS)}")
print(f"  AUTOMOTIVE_OWNER:     {len(AUTOMOTIVE_OWNER_APPS)}")


Override sets loaded:
  CRYPTO_APPS:          14
  SHORT_VIDEO_APPS:     13
  SELLER_TOOL_APPS:     7
  PUBLIC_TRANSPORT:     7
  WEARABLE_APPS:        14
  FASHION_BRANDS:       37
  DEVELOPER_TOOLS:      11
  AUTOMOTIVE_MARKET:    11
  AUTOMOTIVE_OWNER:     20


In [85]:
# ── category_1 → (L1, L2) base lookup ────────────────────────────────────────
# When category_1 alone is unambiguous, this maps directly.
# Context-sensitive categories are handled in assign_taxonomy() below.

CAT1_MAP: dict[str, tuple[str, str]] = {
    # Finance
    "Banking":              ("Finance", "Mobile Banking"),
    "E-Wallet":             ("Finance", "E-Wallet"),
    "Payment Platform":     ("Finance", "Payment Gateway"),
    "Investment App":       ("Finance", "Investment"),
    "Insurance App":        ("Finance", "Insurance"),
    "Crowdfunding App":     ("Finance", "Crowdfunding"),
    "Accounting App":       ("Finance", "Accounting"),
    # Commerce
    "E-Commerce":           ("Commerce", "Marketplace"),
    "Shopping App":         ("Commerce", "Marketplace"),
    "Grocery Platform":     ("Commerce", "Grocery"),
    "Home Retail Platform": ("Commerce", "Home & Living"),
    "B2B Marketplace":      ("Commerce", "B2B"),
    "Review Platform":      ("Commerce", "Review Platform"),
    "Seller Platform":      ("Commerce", "Seller Tools"),
    "Marketplace":          ("Commerce", "Marketplace"),
    # Communication
    "Messaging App":        ("Communication", "Instant Messaging"),
    "Social Media":         ("Communication", "Social Network"),
    "Dating App":           ("Communication", "Dating & Discovery"),
    "Social Discovery Platform": ("Communication", "Dating & Discovery"),
    "Forum Platform":       ("Communication", "Forum & Community"),
    # Entertainment
    "Video Platform":       ("Entertainment", "Video Streaming"),
    "Music Platform":       ("Entertainment", "Music Streaming"),
    "Gaming":               ("Entertainment", "Mobile Games"),
    "Gaming Platform":      ("Entertainment", "Gaming Platform"),
    "Comic Platform":       ("Entertainment", "Comics & Webtoon"),
    "Comedy Platform":      ("Entertainment", "Comedy & Memes"),
    # Information & Education
    "News and Magazine":    ("Information & Education", "News & Media"),
    "Education App":        ("Information & Education", "General Education"),
    "Religious App":        ("Information & Education", "Religious"),
    "Information App":      ("Information & Education", "Reference & Wiki"),
    "Campus App":           ("Information & Education", "Campus & LMS"),
    "Cooking App":          ("Information & Education", "Cooking & Recipes"),
    # Productivity & Tools
    "Cloud Storage":        ("Productivity & Tools", "Cloud Storage & File Sharing"),
    "File Sharing":         ("Productivity & Tools", "Cloud Storage & File Sharing"),
    "Editing App":          ("Productivity & Tools", "Design & Editing"),
    "Photo Viewer":         ("Productivity & Tools", "File Viewer & Reader"),
    "AI Chatbot":           ("Productivity & Tools", "AI Assistant"),
    "Browser":              ("Productivity & Tools", "Browser & Search"),
    "Search Engine":        ("Productivity & Tools", "Browser & Search"),
    "Productivity":         ("Productivity & Tools", "Office & Collaboration"),
    "Tools":                ("Productivity & Tools", "System & Utility"),
    "Customer Service App": ("Productivity & Tools", "Business Operations"),
    # Transportation
    "Ride Hailing":         ("Transportation", "Ride Hailing"),
    "Ride Hailing Driver":  ("Transportation", "Ride Hailing Driver"),
    "Navigation":           ("Transportation", "Navigation & Maps"),
    "Travel App":           ("Transportation", "Travel Booking"),
    "Ticket Booking App":   ("Transportation", "Travel Booking"),
    "Logistic Delivery App":("Transportation", "Logistics & Delivery"),
    "Transportation":       ("Transportation", "Public Transport & Ticketing"),
    "Parking App":          ("Transportation", "Parking"),
    # Health & Wellness
    "Health App":           ("Health & Wellness", "Healthcare & Telemedicine"),
    "Sport and Wellness App":("Health & Wellness", "Fitness & Sport"),
    "Family App":           ("Health & Wellness", "Maternal & Family"),
    # Lifestyle
    "Food Delivery App":    ("Lifestyle", "Food Delivery"),
    "FnB App":              ("Lifestyle", "Dining & FnB"),
    "Coffee Shop App":      ("Lifestyle", "Dining & FnB"),
    "Beauty App":           ("Lifestyle", "Beauty & Personal Care"),
    "Smart Home Platform":  ("Lifestyle", "Smart Home"),
    "Job Marketplace":      ("Lifestyle", "Job & Freelance"),
    "Freelance Marketplace":("Lifestyle", "Job & Freelance"),
    # Platform & System
    "App Store":            ("Platform & System", "App Store"),
    "Telco Selfcare":       ("Platform & System", "Telco Self-Care"),
    "Government App":       ("Platform & System", "Government"),
    "Energy App":           ("Platform & System", "Energy & EV"),
    "Weather":              ("Platform & System", "Weather Service"),
    # Other
    "Tobacco Device App":   ("Other", "Tobacco Device"),
    "Loyalty Program":      ("Commerce", "Marketplace"),
    "Property Marketplace": ("Commerce", "Property"),
    # Context-sensitive (default; fully disambiguated inside assign_taxonomy())
    "Fintech":              ("Finance", "P2P Lending"),
    "Business App":         ("Productivity & Tools", "Business Operations"),
    "Digital Media Store":  ("Platform & System", "App Store"),
    "Automotive App":       ("Lifestyle", "Automotive Owner Service"),
}

print(f"Base CAT1_MAP covers {len(CAT1_MAP)} category_1 values")


Base CAT1_MAP covers 72 category_1 values


In [86]:
def assign_taxonomy(row: pd.Series) -> tuple[str, str]:
    """
    Return (category, subcategory) for a row using a priority cascade:
      P1 — Named app-set overrides (crypto, short video, seller tools, etc.)
      P2 — Keyword pattern overrides (POS apps, live-commerce operators)
      P3 — CAT1_MAP lookup with context disambiguation from cat2/cat3
      P4 — Try cat2 in CAT1_MAP as fallback
      P5 — Unclassified
    """
    name = str(row["app_name"]).strip()
    c1 = str(row["category_1"]).strip()
    c2 = str(row["category_2"]).strip()
    c3 = str(row["category_3"]).strip()
    all_cats = {c1, c2, c3}

    # ── P1: App-set overrides ─────────────────────────────────────────────────
    if name in CRYPTO_APPS:
        return ("Finance", "Crypto & Digital Assets")
    if name in SHORT_VIDEO_APPS:
        return ("Communication", "Short Video & Live")
    if name in SELLER_TOOL_APPS:
        return ("Commerce", "Seller Tools")
    if name in PUBLIC_TRANSPORT_APPS:
        return ("Transportation", "Public Transport & Ticketing")
    if name in WEARABLE_APPS:
        return ("Health & Wellness", "Health Wearables")
    if name in FASHION_BRANDS:
        return ("Commerce", "Fashion")
    if name in DEVELOPER_TOOLS:
        return ("Productivity & Tools", "Developer Tools")
    if name in AUTOMOTIVE_MARKET_APPS:
        return ("Commerce", "Automotive Market")
    if name in AUTOMOTIVE_OWNER_APPS:
        return ("Lifestyle", "Automotive Owner Service")
    if name in HOME_LIVING_APPS:
        return ("Commerce", "Home & Living")

    # ── P2: Keyword-pattern overrides ─────────────────────────────────────────
    # POS / cashier apps → Business Operations
    POS_KEYWORDS = ["POS", "Kasir", "Qasir", "Kasier"]
    if any(k.lower() in name.lower() for k in POS_KEYWORDS):
        return ("Productivity & Tools", "Business Operations")

    # Live-commerce seller operator app → Commerce > Seller Tools
    LIVE_COMMERCE_KEYWORDS = ["Seller", "Merchant", "Mitra", "Seller Center"]
    if any(k.lower() in name.lower() for k in LIVE_COMMERCE_KEYWORDS):
        if c1 in ("Social Media", "Video Platform", "E-Commerce", "Shopping App"):
            return ("Commerce", "Seller Tools")

    # ── P3: CAT1_MAP with context disambiguation ───────────────────────────────
    if c1 in CAT1_MAP:
        l1, l2 = CAT1_MAP[c1]

        # Fintech: refine based on cat2/cat3
        if c1 == "Fintech":
            if "Banking" in all_cats:
                return ("Finance", "Mobile Banking")
            if "E-Wallet" in all_cats:
                return ("Finance", "E-Wallet")
            if "Investment App" in all_cats:
                return ("Finance", "Investment")
            if "Payment Platform" in all_cats:
                return ("Finance", "Payment Gateway")
            return ("Finance", "P2P Lending")   # default for fintech

        # Investment App: crypto check
        if c1 == "Investment App":
            CRYPTO_KEYWORDS = ["crypto", "coin", "token", "chain", "dax", "bitcoin", "web3"]
            if any(k in name.lower() for k in CRYPTO_KEYWORDS):
                return ("Finance", "Crypto & Digital Assets")
            return ("Finance", "Investment")

        # Business App: route by context
        if c1 == "Business App":
            if "FnB App" in all_cats or "Restaurant" in name:
                return ("Lifestyle", "Dining & FnB")
            if "Seller Platform" in all_cats or "E-Commerce" in all_cats:
                return ("Commerce", "Seller Tools")
            if "Accounting App" in all_cats:
                return ("Productivity & Tools", "Accounting")
            if name in DEVELOPER_TOOLS:
                return ("Productivity & Tools", "Developer Tools")
            return ("Productivity & Tools", "Business Operations")

        # Digital Media Store: route by context
        if c1 == "Digital Media Store":
            if "Music Platform" in all_cats:
                return ("Entertainment", "Music Streaming")
            if "Gaming Platform" in all_cats:
                return ("Entertainment", "Gaming Platform")
            if "Video Platform" in all_cats:
                return ("Entertainment", "Video Streaming")
            if "Education App" in all_cats:
                return ("Information & Education", "General Education")
            return ("Platform & System", "App Store")

        # Shopping App: context narrows it
        if c1 == "Shopping App":
            if name in FASHION_BRANDS:
                return ("Commerce", "Fashion")
            if "Grocery Platform" in all_cats:
                return ("Commerce", "Grocery")
            if "Home Retail Platform" in all_cats:
                return ("Commerce", "Home & Living")
            if "B2B Marketplace" in all_cats:
                return ("Commerce", "B2B")
            if "Marketplace" in all_cats:
                return ("Commerce", "Marketplace")
            if "E-Commerce" in all_cats:
                return ("Commerce", "Marketplace")
            return ("Commerce", "Marketplace")

        # E-Commerce: refine toward specialized commerce
        if c1 == "E-Commerce":
            if "Video Platform" in all_cats:   # Bilibili etc.
                return ("Entertainment", "Video Streaming")
            if "B2B Marketplace" in all_cats:
                return ("Commerce", "B2B")
            return ("Commerce", "Marketplace")

        # Automotive App: market vs. owner
        if c1 == "Automotive App":
            if name in AUTOMOTIVE_MARKET_APPS or "E-Commerce" in all_cats:
                return ("Commerce", "Automotive Market")
            return ("Lifestyle", "Automotive Owner Service")

        # Photo Viewer used as an editing companion → still Design & Editing
        # if cat1 IS Photo Viewer but cat2 is Editing App, lean toward editing
        if c1 == "Photo Viewer" and "Editing App" in all_cats:
            return ("Productivity & Tools", "Design & Editing")

        # Social Media: short-video detection beyond the override set
        if c1 == "Social Media":
            SHORT_VIDEO_SIGNALS = ["snack", "short", "reel", "live", "stream", "video"]
            if any(s in name.lower() for s in SHORT_VIDEO_SIGNALS):
                return ("Communication", "Short Video & Live")
            return ("Communication", "Social Network")

        # Productivity: context narrowing
        if c1 == "Productivity":
            if "Messaging App" in all_cats:
                return ("Productivity & Tools", "Office & Collaboration")
            if "Editing App" in all_cats:
                return ("Productivity & Tools", "Design & Editing")
            if "Cloud Storage" in all_cats or "File Sharing" in all_cats:
                return ("Productivity & Tools", "Cloud Storage & File Sharing")
            if "Tools" in all_cats:
                return ("Productivity & Tools", "System & Utility")
            return ("Productivity & Tools", "Office & Collaboration")

        return (l1, l2)

    # ── P4: Try category_2 as fallback ────────────────────────────────────────
    if c2 in CAT1_MAP:
        return CAT1_MAP[c2]

    # ── P5: Unclassified ──────────────────────────────────────────────────────
    return ("Other", "Unclassified")


# Quick validation on known apps
checks = [
    ("BCA",              ("Finance",              "Mobile Banking")),
    ("OVO",              ("Finance",              "E-Wallet")),
    ("TikTok",           ("Communication",        "Short Video & Live")),
    ("TikTok Seller",    ("Commerce",             "Seller Tools")),
    ("Gojek",            ("Transportation",       "Ride Hailing")),
    ("KAI Access",       ("Transportation",       "Public Transport & Ticketing")),
    ("Binance",          ("Finance",              "Crypto & Digital Assets")),
    ("Moka POS",         ("Productivity & Tools", "Business Operations")),
    ("Tokopedia",        ("Commerce",             "Marketplace")),
    ("GitHub",           ("Productivity & Tools", "Developer Tools")),
    ("Zara",             ("Commerce",             "Fashion")),
    ("PELNI",            ("Transportation",       "Public Transport & Ticketing")),
]

print("Taxonomy spot checks:")
all_ok = True
for app_name, expected in checks:
    row = mis[mis["app_name"] == app_name]
    if row.empty:
        print(f"  ⚠  {app_name!r} not found in dataset")
        continue
    result = assign_taxonomy(row.iloc[0])
    status = "✅" if result == expected else "❌"
    if result != expected:
        all_ok = False
    print(f"  {status} {app_name:<30} → {result}  (expected {expected})")

print(f"\n{'All checks passed!' if all_ok else 'Some checks failed — review assign_taxonomy().'}")


Taxonomy spot checks:
  ✅ BCA                            → ('Finance', 'Mobile Banking')  (expected ('Finance', 'Mobile Banking'))
  ✅ OVO                            → ('Finance', 'E-Wallet')  (expected ('Finance', 'E-Wallet'))
  ✅ TikTok                         → ('Communication', 'Short Video & Live')  (expected ('Communication', 'Short Video & Live'))
  ✅ TikTok Seller                  → ('Commerce', 'Seller Tools')  (expected ('Commerce', 'Seller Tools'))
  ✅ Gojek                          → ('Transportation', 'Ride Hailing')  (expected ('Transportation', 'Ride Hailing'))
  ✅ KAI Access                     → ('Transportation', 'Public Transport & Ticketing')  (expected ('Transportation', 'Public Transport & Ticketing'))
  ✅ Binance                        → ('Finance', 'Crypto & Digital Assets')  (expected ('Finance', 'Crypto & Digital Assets'))
  ✅ Moka POS                       → ('Productivity & Tools', 'Business Operations')  (expected ('Productivity & Tools', 'Business Operatio

## Section 12 — Apply Unified Taxonomy & Export Results

In [117]:
# ── Apply taxonomy to all rows ────────────────────────────────────────────────
print("Applying taxonomy to all rows...")
taxonomy_results = mis.apply(assign_taxonomy, axis=1)
mis["category"]    = [r[0] for r in taxonomy_results]
mis["subcategory"] = [r[1] for r in taxonomy_results]

# ── Before/after comparison ───────────────────────────────────────────────────
print("\nOLD category_1 top distribution:")
print(mis["category_1"].value_counts().head(15).to_string())

print("\nNEW category (L1) distribution:")
print(mis["category"].value_counts().to_string())

print("\nNEW subcategory (L2) distribution:")
print(mis["subcategory"].value_counts().to_string())


Applying taxonomy to all rows...

OLD category_1 top distribution:
category_1
Fintech                   76
Gaming                    76
Travel App                56
Video Platform            56
Banking                   53
Shopping App              52
Gaming Platform           49
News and Magazine         46
Automotive App            41
Sport and Wellness App    41
Education App             38
Investment App            38
Business App              34
Productivity              29
Editing App               28

NEW category (L1) distribution:
category
Finance                    213
Entertainment              207
Productivity & Tools       157
Commerce                   119
Information & Education    116
Transportation             100
Lifestyle                   81
Health & Wellness           74
Communication               67
Platform & System           58
Other                        7

NEW subcategory (L2) distribution:
subcategory
Mobile Games                    76
Travel Booking       

In [118]:
# ── Build final output dataframe ──────────────────────────────────────────────
output = pd.DataFrame({
    "app_name":              mis["app_name"],
    "source_app_names_old":  mis["source_app_names"],
    "sig_app_tags":          mis["sig_app_tags"],
    "category":              mis["category"],
    "subcategory":           mis["subcategory"],
    "description":           "",           # blank — to be filled by LLM pipeline
})

# ── Audit: flag rows still Unclassified ───────────────────────────────────────
unclassified = output[output["subcategory"] == "Unclassified"]
print(f"Unclassified rows: {len(unclassified)}")
if not unclassified.empty:
    print("\nApps needing manual review:")
    review_df = unclassified[["app_name", "subcategory"]].copy()
    review_df["old_cat1"] = mis.loc[unclassified.index, "category_1"]
    review_df["old_cat2"] = mis.loc[unclassified.index, "category_2"]
    print(review_df.to_string(index=False))

print(f"\nFinal output: {len(output):,} rows × {len(output.columns)} columns")
output.head(10)


Unclassified rows: 0

Final output: 1,199 rows × 6 columns


,app_name,source_app_names_old,sig_app_tags,category,subcategory,description
0,1Cak,[1cak],1cak,Entertainment,Comedy & Memes,
1,2DFire,[2DFire],2DFire,Productivity & Tools,Business Operations,
2,360Kredi,[360KREDIT],360KREDIT,Finance,Payment Gateway,
3,4shared,[4shared],4shared,Productivity & Tools,Cloud Storage & File Sharing,
4,7-Eleven,[7_Eleven],7_Eleven,Commerce,Grocery,
5,8 Ball Pool,[8BallPool],8BallPool,Entertainment,Mobile Games,
6,9Apps,[9apps],9apps,Platform & System,App Store,
7,9GAG,[9Gag],9Gag,Entertainment,Comedy & Memes,
8,ABC News,[AbcNews],AbcNews,Information & Education,News & Media,
9,ABDA Insurance,[ABDA_AUTO],ABDA_AUTO,Finance,Insurance,


In [119]:

# =============================================================================
# NORMALISE TAXONOMY LABELS → safe snake_case
# Applied immediately after output is built so every downstream cell
# (corrections, CATEGORY_FIXES, saves) works with snake_case labels.
# =============================================================================
for col in ("category", "subcategory"):
    output[col] = output[col].apply(to_safe)

print("Label normalisation applied.")
print("\nL1 distribution:")
print(output["category"].value_counts().to_string())
print(f"\nDistinct subcategories : {output['subcategory'].nunique()}")
print(f"Unclassified rows      : {(output['subcategory'] == 'unclassified').sum()}")


Label normalisation applied.

L1 distribution:
category
finance                  213
entertainment            207
productivity_tools       157
commerce                 119
information_education    116
transportation           100
lifestyle                 81
health_wellness           74
communication             67
platform_system           58
other                      7

Distinct subcategories : 65
Unclassified rows      : 0


In [90]:

# Intermediate save removed — full output is written in the final save cell.


In [120]:

# =============================================================================
# VALIDATION REPORT
# =============================================================================
print("=" * 65)
print("MIGRATION VALIDATION REPORT")
print("=" * 65)

# 1. Column check
expected_cols = ["app_name", "source_app_names_old", "sig_app_tags",
                 "category", "subcategory", "description"]
assert list(output.columns) == expected_cols, f"Column mismatch! {list(output.columns)}"
print(f"Columns correct: {expected_cols}")

# 2. No null category
assert output["category"].notna().all(),    "Null category found!"
assert output["subcategory"].notna().all(), "Null subcategory found!"
print("No null category / subcategory")

# 3. L1 controlled vocabulary (safe snake_case)
L1_ALLOWED = {
    "finance", "commerce", "communication", "entertainment",
    "information_education", "productivity_tools", "transportation",
    "health_wellness", "lifestyle", "platform_system", "other",
}
unknown_l1 = set(output["category"].unique()) - L1_ALLOWED
assert not unknown_l1, f"Unknown L1 values: {unknown_l1}"
print(f"All L1 values in controlled vocabulary ({len(L1_ALLOWED)} values)")

# 4. source_app_names preserved
assert (output["source_app_names_old"] == mis["source_app_names"]).all(), \
    "source_app_names_old values do not match original!"
print("source_app_names_old unchanged from original")

# 5. sig_app_tags coverage
filled = (output["sig_app_tags"] != "").sum()
print(f"sig_app_tags: {filled:,}/{len(output):,} rows ({filled/len(output)*100:.1f}%)")

# 6. description placeholder
assert (output["description"] == "").all(), "description should be empty at this stage"
print("description column present and empty")

# 7. Spot-check reclassifications with safe labels
spot_checks = {
    "TikTok":         ("communication",      "short_video_live"),
    "TikTok Seller":  ("commerce",           "seller_tools"),
    "KAI Access":     ("transportation",     "public_transport_ticketing"),
    "PELNI":          ("transportation",     "public_transport_ticketing"),
    "AnterAja":       ("transportation",     "logistics_delivery"),
    "Adobe":          ("productivity_tools", "design_editing"),
    "Google Photos":  ("productivity_tools", "file_viewer_reader"),
    "Moka POS":       ("productivity_tools", "business_operations"),
    "GitHub":         ("productivity_tools", "developer_tools"),
    "Binance":        ("finance",            "crypto_digital_assets"),
    "Garmin":         ("health_wellness",    "health_wearables"),
    "Zara":           ("commerce",           "fashion"),
}

print("\nSpot checks:")
all_ok = True
for name, (exp_cat, exp_sub) in spot_checks.items():
    row = output[output["app_name"] == name]
    if row.empty:
        print(f"  ⚠  {name!r} not found")
        continue
    got_cat, got_sub = row["category"].iloc[0], row["subcategory"].iloc[0]
    ok = (got_cat == exp_cat) and (got_sub == exp_sub)
    if not ok:
        all_ok = False
    icon = "OK" if ok else "FAIL"
    print(f"  [{icon}] {name:<28} → {got_cat} > {got_sub}")
    if not ok:
        print(f"         expected               → {exp_cat} > {exp_sub}")

print(f"\n{'All checks passed!' if all_ok else 'Some checks failed!'}")
print(f"\nOutput: {OUTPUT_CSV}\nTaxonomy: {TAXONOMY_REF}")


MIGRATION VALIDATION REPORT
Columns correct: ['app_name', 'source_app_names_old', 'sig_app_tags', 'category', 'subcategory', 'description']
No null category / subcategory
All L1 values in controlled vocabulary (11 values)
source_app_names_old unchanged from original
sig_app_tags: 1,199/1,199 rows (100.0%)
description column present and empty

Spot checks:
  [OK] TikTok                       → communication > short_video_live
  [OK] TikTok Seller                → commerce > seller_tools
  [OK] KAI Access                   → transportation > public_transport_ticketing
  [OK] PELNI                        → transportation > public_transport_ticketing
  [OK] AnterAja                     → transportation > logistics_delivery
  [OK] Adobe                        → productivity_tools > design_editing
  [OK] Google Photos                → productivity_tools > file_viewer_reader
  [OK] Moka POS                     → productivity_tools > business_operations
  [OK] GitHub                       → pr

## Section 13 — Description Generation

Template: **"[App Name] is a [subcategory] app by [Vendor]. It [functionality]."**

Sources: verified knowledge + official websites. Apps not in `DESCRIPTIONS` keep an empty `description` field and are listed at the end for manual review.

In [121]:

# =============================================================================
# DESCRIPTIONS DICTIONARY — Finance
# All corrections, patches, and fixes are applied directly below.
# No separate CORRECTIONS or PATCH cells needed.
# =============================================================================
DESC_FINANCE = {
    # Mobile Banking
    "BCA": "BCA is a mobile banking app by Bank Central Asia. It provides account management, fund transfers, bill payments, and QR-based transactions for BCA customers.",
    "BNI": "BNI is a mobile banking app by Bank Negara Indonesia. It offers fund transfers, bill payments, e-wallet top-ups, and account management.",
    "BRI": "BRI is a mobile banking app by Bank Rakyat Indonesia. It supports transfers, savings management, loan services, and QRIS payments.",
    "BSI": "BSI is a mobile banking app by Bank Syariah Indonesia. It provides sharia-compliant banking services including transfers, zakat, and savings.",
    "BTN": "BTN is a mobile banking app by Bank Tabungan Negara. It offers mortgage management, transfers, and bill payments focused on housing finance.",
    "BTPN": "BTPN is a mobile banking app by Bank BTPN. It provides digital banking services including savings, transfers, and financial planning tools.",
    "BJB": "BJB is a mobile banking app by Bank Jabar Banten. It offers transfers, payments, and account services for BJB customers.",
    "Mandiri": "Mandiri is a mobile banking app by Bank Mandiri. It provides transfers, bill payments, investment access, and QR payments.",
    # Halo BCA is a customer-service channel, not a full mobile banking app
    "Halo BCA": "Halo BCA is a customer service app by Bank Central Asia. It provides VoIP-based call center access, live chat support, and self-service account assistance for BCA customers.",
    "CIMB Niaga": "CIMB Niaga is a mobile banking app by CIMB Niaga. It offers fund transfers, bill payments, and investment services.",
    "Allo Bank": "Allo Bank is a mobile banking app by Allo Bank Indonesia. It provides digital banking with social features and lifestyle integrations.",
    "Jago": "Jago is a mobile banking app by Bank Jago. It offers pocket-based budgeting, auto-save features, and biometric-secured transfers.",
    "Jenius": "Jenius is a mobile banking app by Bank BTPN. It provides card management, flexible savings, split bills, and e-commerce payment features.",
    "LINE Bank": "LINE Bank is a mobile banking app by KEB Hana Bank Indonesia. It integrates banking services within the LINE messaging ecosystem.",
    # Removed unverified 'formerly Bank Nationalnobu' corporate-history claim
    "Krom Bank": "Krom Bank is a mobile banking app by PT Krom Bank Indonesia Tbk. It provides digital-first banking services targeting younger users.",
    "SeaBank": "SeaBank is a mobile banking app by SeaBank Indonesia (Sea Group). It offers high-interest savings and integrates with Shopee.",
    # Added Emtek and KakaoBank shareholders
    "Superbank": "Superbank is a mobile banking app by PT Bank Superbank Indonesia (backed by Emtek, Grab, Singtel, and KakaoBank). It provides digital savings and lending products for underbanked users.",
    "NeoBank": "NeoBank is a mobile banking app by Bank Neo Commerce. It offers digital savings accounts with competitive interest rates.",
    "blu by BCA": "blu by BCA is a mobile banking app by BCA Digital. It provides budgeting pockets, transfers, and lifestyle-oriented banking for young adults.",
    # Removed incorrect 'formerly Bank Aladin Syariah' claim
    "Hijra Bank": "Hijra Bank is a mobile banking app by Bank Hijra. It offers sharia-compliant digital banking services.",
    "Bukopin": "Bukopin is a mobile banking app by Bank KB Bukopin. It provides transfers, bill payments, and account management.",
    "HSBC": "HSBC is a mobile banking app by HSBC. It offers global banking services including account management, transfers, and wealth insights.",
    "DBS": "DBS is a mobile banking app by DBS Bank. It provides digital banking services including transfers, investments, and rewards.",
    "Capital One": "Capital One is a mobile banking app by Capital One Financial. It offers credit card management, savings accounts, and spending insights.",
    "Citibank": "Citibank is a mobile banking app by Citigroup. It provides credit card management, fund transfers, and investment access.",
    "CommBank": "CommBank is a mobile banking app by Commonwealth Bank. It offers account management, transfers, and financial product access.",
    "Maybank": "Maybank is a mobile banking app by Maybank. It supports fund transfers, bill payments, and QR-based transactions.",
    "OCBC NISP": "OCBC NISP is a mobile banking app by Bank OCBC NISP. It provides transfers, bill payments, wealth management, and insurance access.",
    "Standard Chartered": "Standard Chartered is a mobile banking app by Standard Chartered Bank. It offers account management, transfers, and wealth services.",
    "UOB": "UOB is a mobile banking app by United Overseas Bank. It provides account management, credit card controls, and cross-border transfers.",
    "Panin Bank": "Panin Bank is a mobile banking app by Panin Bank. It offers transfers, payments, and account management for retail customers.",
    "PermataBank": "PermataBank is a mobile banking app by Bank Permata. It provides transfers, QR payments, and investment product access.",
    "Bank DKI": "Bank DKI is a mobile banking app by Bank DKI. It serves Jakarta-based customers with transfers, payments, and JakCard integration.",
    "Bank Jatim": "Bank Jatim is a mobile banking app by Bank Jatim. It provides banking services focused on East Java customers.",
    "Bank Muamalat": "Bank Muamalat is a mobile banking app by Bank Muamalat Indonesia. It offers sharia-compliant fund transfers and savings.",
    "Bank Muscat": "Bank Muscat is a mobile banking app by Bank Muscat (Oman). It provides account management and transfers for Omani customers.",
    "Bank of America": "Bank of America is a mobile banking app by Bank of America. It provides account management, bill pay, and investment tracking.",
    "African Bank": "African Bank is a mobile banking app by African Bank (South Africa). It offers personal loans, savings, and card management.",
    "ANB": "ANB is a mobile banking app by Arab National Bank (Saudi Arabia). It provides fund transfers, bill payments, and account services.",
    "BBVA": "BBVA is a mobile banking app by Banco Bilbao Vizcaya Argentaria. It offers digital banking across Europe and Latin America.",
    "BDO": "BDO is a mobile banking app by Banco de Oro (Philippines). It provides transfers, bills payment, and account management.",
    "BPI": "BPI is a mobile banking app by Bank of the Philippine Islands. It offers fund transfers, bill payments, and investment access.",
    "ICBC": "ICBC is a mobile banking app by Industrial and Commercial Bank of China. It provides account services and international transfers.",
    "KakaoBank": "KakaoBank is a mobile banking app by KakaoBank (South Korea). It offers digital-first banking integrated with the Kakao ecosystem.",
    "Hana Bank": "Hana Bank is a mobile banking app by KEB Hana Bank. It provides transfers, savings, and investment services.",
    "Maya Bank": "Maya Bank is a mobile banking app by Maya (Philippines). It offers digital banking with integrated e-wallet and crypto features.",
    "NCBC": "NCBC is a mobile banking app by the National Commercial Bank (Saudi Arabia). It provides account management and transfers.",
    "PNB": "PNB is a mobile banking app by Philippine National Bank. It offers transfers, bill payments, and remittance services.",
    "SMBC": "SMBC is a mobile banking app by Sumitomo Mitsui Banking Corporation. It provides account management and digital banking for Japanese customers.",
    "UBS": "UBS is a mobile banking app by UBS Group AG. It provides wealth management and private banking services.",
    "Ziraat Bank": "Ziraat Bank is a mobile banking app by T.C. Ziraat Bankasi (Turkey). It offers account management, transfers, and loan services.",
    "m-Smile": "m-Smile is a mobile banking app by Bank Sinarmas. It provides transfers, bill payments, and account management.",
    # E-Wallet
    "OVO": "OVO is an e-wallet app by PT Visionet Internasional. It provides cashless payments, bill payments, rewards, and investment access across Indonesia.",
    "DANA": "DANA is an e-wallet app by PT Espay Debit Indonesia Koe. It offers QR payments, bill splits, e-commerce checkout, and financial services.",
    "GoPay": "GoPay is an e-wallet app by Gojek (GoTo Group). It provides cashless payments across Gojek services and partner merchants via QR and in-app transactions.",
    "ShopeePay": "ShopeePay is an e-wallet app by Shopee (Sea Group). It enables seamless Shopee checkout, merchant QR payments, and bill payments.",
    "LinkAja": "LinkAja is an e-wallet app by PT Fintek Karya Digital (state-owned consortium). It offers cashless payments, toll payments, and transit integration.",
    "Kredivo": "Kredivo is an e-wallet and buy-now-pay-later app by FinAccel. It provides instant credit lines for e-commerce purchases and bill payments.",
    "Alipay": "Alipay is an e-wallet app by Ant Group (Alibaba). It provides mobile payments, wealth management, and lifestyle services globally.",
    "Atome": "Atome is an e-wallet and buy-now-pay-later app by Atome Financial. It allows split payments across retail and online merchants in Southeast Asia.",
    "Akulaku": "Akulaku is an e-wallet and digital finance app by PT Akulaku Silvrr Indonesia. It offers buy-now-pay-later, loans, and savings products.",
    # BestPay identified as merchant-acquiring product, not China Telecom e-wallet
    "BestPay": "BestPay is a payment app by BestPay Solutions. It provides merchant payment acceptance and card-acquiring services.",
    "Dompet Kilat": "Dompet Kilat is an e-wallet app by PT Indo Fin Tek. It offers quick digital payments and cash loans in Indonesia.",
    "Duitku": "Duitku is an e-wallet and payment gateway app by PT Duitku Indonesia. It provides payment processing for merchants and consumers.",
    "Huawei Wallet": "Huawei Wallet is an e-wallet app by Huawei. It enables NFC payments, loyalty card storage, and transit pass integration on Huawei devices.",
    "Indodana": "Indodana is an e-wallet and lending app by PT Artha Dana Teknologi. It offers consumer credit, pay-later services, and e-wallet features.",
    # Indosaku is a digital lending app, not an e-wallet
    "Indosaku": "Indosaku is a digital lending app. It provides online instalment loans and consumer credit services in Indonesia.",
    "MoMo": "MoMo is an e-wallet app by M_Service (Vietnam). It offers mobile payments, bill payments, and financial services in Vietnam.",
    "PayMaya": "PayMaya is an e-wallet app by Voyager Innovations (Philippines). It provides cashless payments, online shopping, and money transfers.",
    # Use exact Unicode apostrophe (U+2019) to match app_name in source data
    "Touch \u2019n Go eWallet": "Touch \u2019n Go eWallet is an e-wallet app by TNG Digital (Malaysia). It enables toll payments, transit fares, and merchant QR payments.",
    "TrueMoney": "TrueMoney is an e-wallet app by Ascend Money (CP Group). It provides money transfers, bill payments, and merchant payments in Southeast Asia.",
    # Uangku — active product uncertain; safer neutral description
    "Uangku": "Uangku is a digital finance app. It provides personal finance management and financial services via mobile.",
    "WeChat Pay": "WeChat Pay is an e-wallet app by Tencent. It provides in-app payments, merchant QR transactions, and money transfers within the WeChat ecosystem.",
    # Payment Gateway
    "PayPal": "PayPal is a payment gateway app by PayPal Holdings. It enables online payments, money transfers, and merchant checkout worldwide.",
    "Midtrans": "Midtrans is a payment gateway app by Midtrans (GoTo Group). It provides multi-channel payment processing for Indonesian merchants.",
    "Xendit": "Xendit is a payment gateway app by Xendit. It offers payment infrastructure and disbursement APIs for businesses in Southeast Asia.",
    "Mastercard": "Mastercard is a payment gateway app by Mastercard Inc. It provides card-linking, transaction tracking, and contactless payment features.",
    "UnionPay": "UnionPay is a payment gateway app by China UnionPay. It enables card payments and cross-border transactions.",
    "Braintree": "Braintree is a payment gateway app by PayPal (Braintree division). It provides payment processing SDKs for merchants and developers.",
    "Klarna": "Klarna is a payment gateway and buy-now-pay-later app by Klarna Bank AB. It offers split payments and smooth checkout for online shoppers.",
    "Home Credit": "Home Credit is a payment gateway and consumer finance app by Home Credit Group. It provides point-of-sale loans and installment plans.",
    "Tenpay": "Tenpay is a payment gateway app by Tencent. It provides online payment processing integrated with QQ and WeChat services.",
    "Xsolla": "Xsolla is a payment gateway app by Xsolla. It specializes in game commerce, providing in-game payments and distribution for game developers.",
    "360Kredi": "360Kredi is a payment gateway app by PT 360 Kredit Teknologi. It provides consumer loans and installment services in Indonesia.",
    # AdaKami is a digital lending app, not a payment gateway
    "AdaKami": "AdaKami is a digital lending app by PT Pembiayaan Digital Indonesia. It offers consumer credit and instalment loans via mobile.",
    "AdaPundi": "AdaPundi is a payment gateway app by PT Pundi Mas Berjaya. It provides working capital loans and payment services for Indonesian micro-merchants.",
    "AdiraKu": "AdiraKu is a payment gateway app by Adira Finance. It offers vehicle and consumer installment payment management.",
    "Awantunai": "Awantunai is a payment gateway app by PT SimpleFi Teknologi Indonesia. It provides working capital financing for FMCG distributors and micro-retailers via an integrated ERP system.",
    "BAF Mobile": "BAF Mobile is a payment gateway app by Bussan Auto Finance. It provides motorcycle and consumer goods installment management.",
    "BantuSaku": "BantuSaku is a payment gateway app by PT Smartek Halodata. It offers personal loans and digital payment services.",
    "CARiN": "CARiN is a payment gateway app by PT Cakra Anugerah Indonesia. It provides digital insurance and payment services.",
    "DanaBagus": "DanaBagus is a payment gateway app by PT Dana Bagus Indonesia. It offers personal microloans through a mobile application.",
    "DanaID": "DanaID is a payment gateway app by PT DanaID Pinjaman Daring. It provides consumer microloans via a digital platform.",
    "DanaKini": "DanaKini is a payment gateway app by PT DanaKini Indonesia. It offers short-term consumer loans through a digital application.",
    "DanaRupiah": "DanaRupiah is a payment gateway app by PT DanaRupiah Apps. It provides online personal loans in Indonesia.",
    "EasyCash": "EasyCash is a payment gateway app by PT Indonesia Fintopia Technology. It provides consumer microloans with fast disbursement.",
    # Findaya — current product identity unverified; marked for review
    "Findaya": "Findaya is a digital lending app. Its current product offering and corporate identity require manual verification against the current app-store listing before use.",
    "Ivoji": "Ivoji is a payment gateway app by PT Finansia Aira Teknologi. It offers personal cash loans online without collateral, licensed by OJK.",
    "JULO": "JULO is a payment gateway app by PT JULO Teknologi Finansial. It provides flexible credit lines and installment loans for Indonesian consumers.",
    "KawanCicil": "KawanCicil is a payment gateway app. It provides installment financing services for students and young professionals in Indonesia.",
    "KlikCair": "KlikCair is a payment gateway app by PT Klikcair Magna Sejahtera. It offers consumer microloans through a mobile application.",
    # KlikKami is a digital lending app, not a payment gateway
    "KlikKami": "KlikKami is a digital lending app. It provides online loan services in Indonesia.",
    "KreditMu": "KreditMu is a payment gateway app. It offers consumer microloan services through a digital platform in Indonesia.",
    "KreditPro": "KreditPro is a payment gateway app. It provides consumer loan services via a mobile application.",
    "Kredinesia": "Kredinesia is a payment gateway app by PT Kreditku Teknologi Indonesia. It offers fast consumer microloans via mobile.",
    # Kredito is a digital lending app, not a payment gateway
    "Kredito": "Kredito is a digital lending app. It provides consumer credit and instalment loan services in Indonesia.",
    # KrediFazz is a digital lending app, not a payment gateway
    "KrediFazz": "KrediFazz is a digital lending app by Fazz Financial Group. It offers working capital loans for micro and small merchants.",
    "Kredit Pintar": "Kredit Pintar is a payment gateway app by PT Kredit Pintar Indonesia. It provides OJK-licensed personal loans with fast approval.",
    # Kudo — current app-store identity uncertain; marked for review
    "Kudo": "Kudo is a digital payment app. The currently active product under this name should be manually verified against the current app-store listing before use in analysis.",
    "MauCash": "MauCash is a payment gateway app by PT Astra WeLab Digital Arta. It provides personal loans with quick disbursement.",
    # PinjamDuit is a digital lending app, not a payment gateway
    "PinjamDuit": "PinjamDuit is a digital lending app. It offers consumer personal loans via a digital application in Indonesia.",
    "PinjamWinWin": "PinjamWinWin is a payment gateway app. It provides short-term consumer loans through a mobile platform.",
    "PinjamYuk": "PinjamYuk is a payment gateway app. It offers personal microloans through a mobile digital application in Indonesia.",
    "PayTren": "PayTren is a payment gateway app by PT Veritra Sentosa Internasional. It provides digital payments, PPOB bill payments, and Islamic fintech services.",
    "Rupiah Cepat": "Rupiah Cepat is a payment gateway app by PT Kredit Utama Fintech Indonesia. It provides fast personal microloans via mobile.",
    "Taralite": "Taralite is a payment gateway app by PT Indonusa Bara Sejahtera. It offers small-ticket consumer loans and e-commerce installments.",
    "TunaiKu": "TunaiKu is a payment gateway app by PT Amar Bank Indonesia. It provides personal loans and installment credit via mobile.",
    "UATAS": "UATAS is a payment gateway app. It offers consumer lending services through a digital platform.",
    # UangMe is a digital lending app, not a payment gateway
    "UangMe": "UangMe is a digital lending app by PT UangMe Fintek Indonesia. It provides personal loans with online-only processing.",
    # UKU is a digital lending app, not a payment gateway
    "UKU": "UKU is a digital lending app. It offers consumer microloan services via a mobile application in Indonesia.",
    "Youtap": "Youtap is a payment gateway app by PT Youtap Indonesia. It provides QR-based payment acceptance and merchant management tools.",
    # YUP is a pay-later / credit access app, not a payment gateway
    "YUP": "YUP is a pay-later and credit access app. It provides buy-now-pay-later and consumer credit services.",
    # DanaCita is an education financing app, not a payment gateway
    "DanaCita": "DanaCita is an education financing app by PT Inclusive Finance Group. It provides buy-now-pay-later instalment plans for tuition, courses, and study-abroad expenses.",
    # Investment
    "Bibit": "Bibit is an investment app by PT Bibit Tumbuh Bersama (Stockbit Group). It offers robo-advisor mutual fund investing tailored for Indonesian retail investors.",
    "Bareksa": "Bareksa is an investment app by PT Bareksa Portal Investasi. It provides mutual fund and government bond investment with marketplace comparison.",
    "Ajaib": "Ajaib is an investment app by Ajaib Group. It offers stock and mutual fund trading with beginner-friendly tools for Indonesian investors.",
    "Stockbit": "Stockbit is an investment app by PT Stockbit Sekuritas. It provides stock trading, social investing features, and market analysis tools.",
    "IPOT": "IPOT is an investment app by Indo Premier Sekuritas. It offers stock, mutual fund, and bond trading with research tools.",
    "TradingView": "TradingView is an investment app by TradingView Inc. It provides advanced charting, technical analysis, and community-driven trading ideas.",
    "MetaTrader": "MetaTrader is an investment app by MetaQuotes Software. It provides forex and CFD trading with charting, expert advisors, and multi-broker access.",
    "Pluang": "Pluang is an investment app by PT Bumi Santosa Cemerlang. It offers gold, mutual fund, and crypto micro-investing in Indonesia.",
    "POEMS": "POEMS is an investment app by Phillip Securities. It provides stock, futures, and unit trust trading across Asian markets.",
    "BIONS": "BIONS is an investment app by BNI Sekuritas. It offers stock trading and market research tools for Indonesian investors.",
    "IndoPremier": "IndoPremier is an investment app by Indo Premier Sekuritas. It provides access to IPOs, bonds, and mutual funds.",
    "Mirae HOTS": "Mirae HOTS is an investment app by Mirae Asset Sekuritas Indonesia. It offers stock trading with real-time analytics and research.",
    "XE": "XE is an investment app by XE.com (Euronet Worldwide). It provides live currency exchange rates and international money transfers.",
    "RTI Business": "RTI Business is an investment app by RTI Infokom. It provides real-time Indonesian stock market data and portfolio tracking.",
    "Pegadaian": "Pegadaian is an investment app by PT Pegadaian (state-owned). It offers gold savings, pawn services, and sharia-compliant financing.",
    "KoinP2P": "KoinP2P is an investment app by KoinWorks. It provides peer-to-peer lending where users fund vetted borrower loans.",
    "KoinWorks": "KoinWorks is an investment app by PT Lunaria Annua Teknologi. It offers P2P lending and SME financing.",
    "Akseleran": "Akseleran is an investment app by PT Akseleran Keuangan Inklusif Indonesia. It provides P2P lending for SME business loans.",
    "Investree": "Investree is an investment app by PT Investree Radhika Jaya. It offers P2P lending for invoice and working-capital financing.",
    "AdaModal": "AdaModal is an investment app. It provides investment and lending services through a digital platform in Indonesia.",
    "Alami": "Alami is an investment app by PT Alami Fintek Sharia. It offers sharia-compliant P2P lending and invoice financing.",
    "Amartha": "Amartha is an investment app by PT Amartha Mikro Fintek. It provides microfinance P2P lending focused on women-owned rural businesses.",
    "Avantee": "Avantee is an investment app by PT Avantee Gadai Indonesia. It offers P2P lending and gold pawn services.",
    "Bahana DXtrade": "Bahana DXtrade is an investment app by Bahana Sekuritas. It offers stock and derivatives trading on the Indonesian exchange.",
    "Batumbu": "Batumbu is an investment app by PT Batumbu Lentera Indonesia. It provides P2P lending for agricultural sector SMEs.",
    "Dana Syariah": "Dana Syariah is an investment app by PT Dana Syariah Indonesia. It offers sharia-compliant P2P lending for property financing.",
    "DanaMerdeka": "DanaMerdeka is an investment app by PT Dana Merdeka. It provides retail investment access to small-cap mutual funds and savings bonds.",
    "Danamas": "Danamas is an investment app by PT Pasar Dana Pinjaman (Sinar Mas Group). It is Indonesia's first licensed P2P lending platform, connecting funders with MSME borrowers.",
    "Dhanapala": "Dhanapala is an investment app by PT Dhanapala Teradata Indonesia. It provides P2P lending for SME business loans.",
    "Esta Kapital": "Esta Kapital is an investment app. It offers P2P lending and investment services in Indonesia.",
    "IkiModal": "IkiModal is an investment app. It provides P2P lending investment access for Indonesian retail investors.",
    "Indofund": "Indofund is an investment app by PT Indofund Berkat Mulya. It offers P2P lending for SME working-capital loans.",
    "Investasik": "Investasik is an investment app. It provides investment and savings products for Indonesian retail users.",
    # Komunal — BPR deposit marketplace; centred on rural bank digitization
    "Komunal": "Komunal is an investment app by Komunal Group. It is a fintech platform centred on a rural bank (BPR) deposit marketplace, enabling digital savings placement in licensed BPR institutions and peer-to-peer lending for MSME borrowers.",
    "Lentera Dana Nusantara": "Lentera Dana Nusantara is an investment app. It provides P2P lending services for Indonesian borrowers and lenders.",
    "LumbungDana": "LumbungDana is an investment app by PT Lumbung Dana Indonesia. It provides OJK-licensed P2P lending with online personal loans and investor funding access.",
    "Makmur": "Makmur is an investment app by PT Inovasi Finansial Teknologi. It provides mutual fund investing with competitive fee structures.",
    "ModalRakyat": "ModalRakyat is an investment app by PT Modal Rakyat Indonesia. It provides OJK-licensed P2P lending for productive MSME loans and micro-borrowers.",
    "Modalku": "Modalku is an investment app by PT Mitrausaha Indonesia Grup (Funding Societies). It provides P2P lending for SME invoice financing, term loans, and purchase-order financing across Southeast Asia.",
    "Moduit": "Moduit is an investment app by PT Moduit Digital Indonesia. It offers mutual fund investment with personalized portfolio recommendations.",
    "Moinves": "Moinves is an investment app. It provides mutual fund investment services for Indonesian retail investors.",
    "Simas Fund": "Simas Fund is an investment app by Sinarmas Asset Management. It provides mutual fund investment and portfolio management.",
    "Tanamduit": "Tanamduit is an investment app by PT Star Mercato Capitale. It offers mutual fund, gold, and government bond micro-investing.",
    "TaniFund": "TaniFund is an investment app by PT Tani Fund Madani Indonesia. It provides P2P lending for agricultural projects.",
    "Yo! Inves": "Yo! Inves is an investment app. It provides retail investment access for Indonesian users.",
    "iGrow": "iGrow is an investment app by PT iGrow Resources Indonesia. It connects investors with agricultural and aquaculture projects for social-impact returns.",
    # Crypto & Digital Assets
    "Binance": "Binance is a crypto exchange app by Binance Holdings. It provides spot and futures trading, staking, and DeFi access for hundreds of cryptocurrencies.",
    "Indodax": "Indodax is a crypto exchange app by PT Indodax Nasional Indonesia. It offers cryptocurrency trading for Indonesian retail investors.",
    "OKX": "OKX is a crypto exchange app by OKX (formerly OKEx). It provides spot, derivatives, and DeFi trading for global users.",
    "Gate.io": "Gate.io is a crypto exchange app by Gate Technology Inc. It offers spot and margin trading for a wide range of digital assets.",
    "KuCoin": "KuCoin is a crypto exchange app by KuCoin Global. It provides spot, futures, and lending services for cryptocurrencies.",
    "CoinMarketCap": "CoinMarketCap is a crypto data and tracking app by CoinMarketCap (Binance). It provides real-time prices, charts, and portfolio tracking for thousands of cryptocurrencies.",
    "MetaMask": "MetaMask is a crypto wallet app by ConsenSys. It provides a self-custody Ethereum wallet, token management, and dApp browser for Web3 interactions.",
    # Removed 'OJK-regulated' language — regulator status unverified
    "Tokocrypto": "Tokocrypto is a crypto exchange app by PT Crypto Indonesia Berkat. It offers cryptocurrency trading for Indonesian users.",
    "Crypto.com": "Crypto.com is a crypto exchange and wallet app by Crypto.com. It provides crypto trading, staking, Visa card integration, and DeFi access.",
    "OctaFX": "OctaFX is a crypto and forex trading app by Octa Markets. It provides CFD trading on forex, commodities, and crypto pairs.",
    "IQ Option": "IQ Option is a crypto and forex trading app by IQOption LLC. It offers options, forex, and crypto CFD trading with educational tools.",
    "Olymp Trade": "Olymp Trade is a crypto and forex trading app by OlympTrade. It provides fixed-time trades and forex CFDs for emerging market traders.",
    "Nanovest": "Nanovest is a crypto and investment app by PT Tumbuh Bersama Nano. It provides fractional US stock and cryptocurrency investing for Indonesian users.",
    "ICAP Trade": "ICAP Trade is a crypto trading app. It offers cryptocurrency and digital asset trading services.",
    # Insurance
    "AIA": "AIA is an insurance app by AIA Group. It provides life and health insurance policy management, claims, and wellness rewards.",
    "Allianz": "Allianz is an insurance app by Allianz SE. It offers insurance policy management, claims submission, and roadside assistance.",
    "AXA": "AXA is an insurance app by AXA Group. It provides insurance policy access, claims tracking, and health services.",
    "ACA Insurance": "ACA Insurance is an insurance app by PT Asuransi Central Asia. It offers motor and property insurance management.",
    "ABDA Insurance": "ABDA Insurance is an insurance app by PT Asuransi Bina Dana Arta. It provides motor and general insurance services.",
    "Autocillin": "Autocillin is an insurance app by Adira Insurance. It offers vehicle insurance management, claims, and roadside assistance.",
    "BNI Life": "BNI Life is an insurance app by BNI Life Insurance. It provides life and health insurance policy management.",
    "BRI Life": "BRI Life is an insurance app by BRI Life. It offers bancassurance products and claims management for BRI customers.",
    "Capital Life": "Capital Life is an insurance app by Capital Life Indonesia. It provides life insurance and unit-linked investment products.",
    "Duha Syariah": "Duha Syariah is an insurance app by PT Asuransi Jiwa Syariah Jasa Mitra Abadi. It offers sharia-compliant life insurance.",
    "Ezurance": "Ezurance is an insurance app. It provides digital insurance comparison and purchasing services.",
    "FWD": "FWD is an insurance app by FWD Group. It offers life and health insurance with digital-first policy management and claims.",
    "Garda Mobile": "Garda Mobile is an insurance app by Asuransi Astra. It provides motor insurance, e-policy, emergency roadside assistance, and claims tracking.",
    "Indolife Pensiontama": "Indolife Pensiontama is an insurance app by PT Indolife Pensiontama. It offers life insurance and pension fund products.",
    "Manulife": "Manulife is an insurance app by Manulife Financial. It provides life, health insurance policy access, and wealth management.",
    "Prufast": "Prufast is an insurance app by Prudential Indonesia. It offers quick insurance policy purchase and claims processing.",
    "Sinarmas Insurance": "Sinarmas Insurance is an insurance app by PT Asuransi Sinar Mas. It provides motor, health, and property insurance services.",
    "Wahana Tata": "Wahana Tata is an insurance app by PT Asuransi Wahana Tata. It offers general insurance including motor and property coverage.",
    # P2P Lending
    "Ammana": "Ammana is an investment app by PT Ammana Fintek Syariah. It provides sharia-compliant P2P lending connecting funders with halal business projects.",
    "Asetku": "Asetku is an investment app by PT Pintar Inovasi Digital. It provides P2P lending connecting retail investors with consumer credit borrowers.",
    "Cermati": "Cermati is a P2P lending app by PT Cermati Fintek Indonesia. It provides financial product comparisons and personal loan marketplace.",
    "KlikUMKM": "KlikUMKM is a P2P lending app. It offers P2P lending services connecting funders with micro and small business borrowers in Indonesia.",
    "Klika2c": "Klika2c is a P2P lending app. It provides P2P lending services for Indonesian consumers and small businesses.",
    "PinjamModal": "PinjamModal is a P2P lending app by PT Finansial Integrasi Teknologi (subsidiary of BFI Finance). It provides working capital loans for MSMEs and micro-retailers.",
    "Restock.id": "Restock.id is a P2P lending app by PT Restock.id. It provides asset-backed P2P lending for creative retail MSMEs, using inventory as collateral.",
    "TokoModal": "TokoModal is a P2P lending app by PT Toko Modal Mitra Usaha. It provides micro-financing for small shop owners through agent-based P2P lending.",
    # Crowdfunding
    "Kitabisa": "Kitabisa is a crowdfunding app by PT Kita Bisa Indonesia. It is Indonesia's largest crowdfunding platform for social causes, medical bills, and disaster relief.",
    "Crowde": "Crowde is a crowdfunding app by PT Crowde Membangun Bangsa. It connects investors with agricultural projects for social-impact crowdfunding.",
    "Crowdo": "Crowdo is a crowdfunding app by Crowdo Holdings. It provides equity and debt crowdfunding for startups and SMEs in Southeast Asia.",
    "GandengTangan": "GandengTangan is a crowdfunding app by PT Gandeng Tangan Indonesia. It offers micro-lending crowdfunding for small business empowerment.",
}

print(f"DESC_FINANCE: {len(DESC_FINANCE)} entries")


DESC_FINANCE: 213 entries


In [122]:
# =============================================================================
# DESCRIPTIONS DICTIONARY — Commerce
# All corrections and patches are applied directly below.
# =============================================================================
DESC_COMMERCE = {
    # Marketplace
    "AliExpress": "AliExpress is a marketplace app by Alibaba Group. It connects international buyers with Chinese manufacturers for affordable cross-border shopping.",
    "Amazon": "Amazon is a marketplace app by Amazon.com Inc. It offers global e-commerce, digital content, and same-day delivery services.",
    "Banggood": "Banggood is a marketplace app by Banggood. It offers cross-border shopping with electronics, gadgets, and lifestyle products from Chinese suppliers.",
    "Bayleaf": "Bayleaf is a shopping app by Bayleaf. It provides online retail shopping services through the Bayleaf e-commerce platform.",
    "Blibli": "Blibli is a marketplace app by PT Global Digital Niaga (Djarum Group). It offers curated e-commerce with genuine product guarantees and same-day delivery in Indonesia.",
    "Bukalapak": "Bukalapak is a marketplace app by PT Bukalapak.com Tbk. It provides C2C and B2C e-commerce with a focus on empowering warung micro-retailers across Indonesia.",
    "Buttonscarves": "Buttonscarves is a fashion shopping app by PT Buttonscarves Indonesia. It sells premium hijabs, scarves, bags, and fashion accessories through its official direct-to-consumer channel.",
    "Casio Watches": "Casio Watches is a lifestyle and connected-watch companion app by Casio. It supports Casio watch connectivity, settings, activity data, and related brand features.",
    "CASIO_WATCH": "CASIO_WATCH is a lifestyle and connected-watch companion app by Casio. It supports Casio watch connectivity, settings, activity data, and related brand features.",
    "DHgate": "DHgate is a marketplace app by DHgate.com. It connects international buyers with Chinese wholesalers for B2B and B2C cross-border trade.",
    "DangDang": "DangDang is a marketplace app by Dangdang (China). It offers books, media, and general merchandise e-commerce in the Chinese market.",
    "ECI": "ECI is a shopping app by ECI. It provides marketplace-related online retail services through the ECI platform.",
    "Elevenia": "Elevenia was an Indonesian marketplace app by PT XL Planet that operated as a C2C and B2C e-commerce platform.",
    "Etsy": "Etsy is a marketplace app by Etsy Inc. It provides a marketplace for handmade, vintage, and unique creative goods from independent sellers.",
    "Flipkart": "Flipkart is a marketplace app by Flipkart (Walmart). It is India's leading e-commerce platform offering electronics, fashion, and groceries.",
    "Hartono Elektronika": "Hartono Elektronika is a marketplace app by Hartono Elektronika. It offers consumer electronics and home appliances retail in Indonesia.",
    "Itemku": "Itemku is a marketplace app by PT Five Jack. It is Indonesia's marketplace for game vouchers, in-game items, and digital goods trading.",
    "JD.ID": "JD.ID was an Indonesian e-commerce app by JD.com that operated a B2C online retail platform with local warehousing and fulfillment.",
    "Jualo": "Jualo is a marketplace app by PT Jualo. It provides classified listings for secondhand and new goods in Indonesia.",
    "Klik Indogrosir": "Klik Indogrosir is a marketplace app by Indomarco (Salim Group). It provides wholesale-to-retail online ordering for grocery and FMCG products.",
    "Lazada": "Lazada is a marketplace app by Lazada Group (Alibaba). It offers B2C and marketplace e-commerce across Southeast Asia with integrated logistics.",
    "MAPCLUB": "MAPCLUB is a shopping and loyalty app by MAP Group (PT Mitra Adiperkasa). It combines rewards, promotions, store discovery, and online shopping across MAP retail brands in Indonesia.",
    "Matahari": "Matahari is a marketplace app by PT Matahari Department Store Tbk. It offers fashion, beauty, and lifestyle products through online and offline channels.",
    "Mercari": "Mercari is a marketplace app by Mercari Inc (Japan). It provides a C2C marketplace for buying and selling secondhand items.",
    "Noon": "Noon is a marketplace app by Noon.com (Middle East). It offers e-commerce for electronics, fashion, and daily essentials across the Gulf region.",
    "OLX": "OLX is a marketplace app by OLX Group. It provides classified ads for buying and selling secondhand goods, vehicles, and property.",
    "Orami": "Orami is a parenting and shopping app by PT Orami Indonesia. It combines family-focused commerce with parenting content and community features.",
    "Poizon": "Poizon is a marketplace app by Dewu (China). It provides an authentication-verified marketplace for sneakers, streetwear, and luxury goods.",
    "Rakuten": "Rakuten is a marketplace app by Rakuten Group (Japan). It offers e-commerce, cashback, and digital services globally.",
    "RupaRupa": "RupaRupa is a marketplace app by PT Ruparupa (Kawan Lama Group). It offers home improvement, furniture, and electronics through an omnichannel platform.",
    "Shop App": "Shop App is a shopping app by Shopify. It provides order tracking, product discovery, and Shop Pay-enabled checkout across participating stores.",
    "Shopee": "Shopee is a marketplace app by Sea Group. It is Southeast Asia's leading e-commerce platform with integrated payments, logistics, and social commerce features.",
    "Taobao": "Taobao is a marketplace app by Alibaba Group. It is China's largest C2C marketplace offering billions of product listings.",
    "Temu": "Temu is a marketplace app by PDD Holdings. It offers ultra-low-price cross-border e-commerce shipping directly from Chinese factories.",
    "Tmall": "Tmall is a marketplace app by Alibaba Group. It provides a premium B2C e-commerce platform for branded goods in China.",
    "Tokopedia": "Tokopedia is a marketplace app by PT Tokopedia (GoTo Group). It is one of Indonesia's largest C2C/B2C marketplaces with integrated digital goods and financial services.",
    "Weverse Shop": "Weverse Shop is a marketplace app by HYBE Corp. It sells official K-pop merchandise and artist goods tied to the Weverse fan platform.",
    "Xiaomi Store": "Xiaomi Store is a marketplace app by Xiaomi Corp. It offers official hardware, accessories, and smart home products from Xiaomi's ecosystem.",
    "Zalando": "Zalando is a marketplace app by Zalando SE (Germany). It offers fashion and lifestyle e-commerce across European markets.",
    "Zalora": "Zalora is a marketplace app by Zalora Group (Global Fashion Group). It specializes in fashion e-commerce across Southeast Asia.",
    "eBay": "eBay is a marketplace app by eBay Inc. It provides auction-style and fixed-price e-commerce for new and secondhand goods globally.",
    # Fashion
    "ASOS": "ASOS is a fashion app by ASOS plc. It offers fast-fashion e-commerce for young adults with global shipping from the UK.",
    "Adidas": "Adidas is a fashion app by Adidas AG. It offers sportswear shopping, product launches, and membership rewards.",
    "Adidas Running": "Adidas Running is a fitness app by Adidas (Runtastic). It provides GPS activity tracking, training plans, and workout logging.",
    "Berrybenka": "Berrybenka is a fashion app by PT Berrybenka. It offers Indonesian women's fashion and beauty e-commerce.",
    "Bershka": "Bershka is a fashion app by Inditex. It provides fast fashion shopping for young adults.",
    "Bossini": "Bossini is a fashion app by Bossini International. It offers casual fashion retail from the Hong Kong-based brand.",
    "Casetify": "Casetify is a fashion app by Casetify. It sells custom-designed phone cases, Apple Watch bands, and tech accessories.",
    "Chanel": "Chanel is a fashion app by Chanel S.A. It showcases luxury fashion collections, beauty products, and boutique locations.",
    "Elbina Hijab": "Elbina Hijab is a fashion app. It offers Muslim women's hijab and modest fashion retail in Indonesia.",
    "Elzatta": "Elzatta is a fashion app by PT Elzatta Hijab Indonesia. It provides modest fashion and hijab collections for Indonesian women.",
    "Fadkhera": "Fadkhera is a fashion app. It offers fashion and modest wear products in Indonesia.",
    "Farfetch": "Farfetch is a fashion app by Farfetch. It provides luxury fashion e-commerce connecting boutiques and designers worldwide.",
    "Foot Locker": "Foot Locker is a fashion app by Foot Locker Inc. It offers athletic footwear, apparel, and sneaker release reservations.",
    "Guardian": "Guardian is a health and beauty retail app by Guardian Health & Beauty. It offers personal care, wellness, and beauty product shopping.",
    "H&M": "H&M is a fashion app by H&M Group. It offers affordable fashion shopping with loyalty rewards and in-store integration.",
    "Keen": "Keen is a fashion app by KEEN Footwear. It offers outdoor and casual footwear direct-to-consumer.",
    "Li-Ning": "Li-Ning is a fashion app by Li-Ning Co (China). It offers sportswear and athletic footwear shopping.",
    "Lotte": "Lotte is a fashion app by Lotte Group (South Korea). It offers department store and duty-free shopping across fashion and lifestyle categories.",
    "Louis Vuitton": "Louis Vuitton is a fashion app by LVMH. It showcases luxury fashion, leather goods, and accessories with e-commerce and boutique locator.",
    "LuisaViaRoma": "LuisaViaRoma is a fashion app by LuisaViaRoma (Italy). It offers luxury designer fashion e-commerce.",
    "Maybelline": "Maybelline is a fashion app by L'Oréal (Maybelline New York). It offers cosmetics browsing, virtual try-on, and beauty tips.",
    "Mytheresa": "Mytheresa is a fashion app by Mytheresa (Germany). It provides luxury fashion e-commerce for women, men, and kids.",
    "New Balance": "New Balance is a fashion app by New Balance Athletics. It offers athletic footwear and apparel shopping.",
    "Nike": "Nike is a fashion app by Nike Inc. It offers sportswear shopping, sneaker launches (SNKRS), and Nike membership perks.",
    "Pull&Bear": "Pull&Bear is a fashion app by Inditex. It offers casual and youthful fashion retail.",
    "Rabbani": "Rabbani is a fashion app by PT Rabbani Asysa. It offers kerudung (hijab) and modest fashion retail in Indonesia.",
    "Rings": "Rings is too generic as an app name. Confirm the exact publisher or brand before assigning a jewelry-commerce description.",
    "Saturdays": "Saturdays is a fashion app by SATURDAYS (PT Saturdays Pagi Indonesia). It offers affordable premium eyewear and sunglasses D2C in Indonesia.",
    "Shein": "Shein is a fashion app by Roadget Business Pte. It offers ultra-fast fashion e-commerce with low-cost global shipping.",
    "Tuneeca": "Tuneeca is a fashion app by PT Tuneeca Indonesia. It offers modest fashion and Muslim wear retail.",
    "Under Armour": "Under Armour is a fashion app by Under Armour Inc. It offers athletic performance apparel and footwear shopping.",
    "Uniqlo": "Uniqlo is a fashion app by Fast Retailing (Japan). It offers LifeWear basics and seasonal fashion with in-store pickup.",
    "Zara": "Zara is a fashion app by Inditex. It offers fast fashion shopping with new collections, in-store stock checking, and click-and-collect.",
    "Zizara": "Zizara is a fashion app. It offers modest Muslim fashion retail in Indonesia.",
    "Zoya": "Zoya is a fashion app by PT Shafco Multi Trading. It offers hijab, modest fashion, and cosmetics retail in Indonesia.",
    "iStyle": "iStyle is a fashion app by iStyle. It offers beauty and cosmetics e-commerce.",
    # Grocery
    "7-Eleven": "7-Eleven is a grocery app by Seven & i Holdings. It provides convenience store product browsing, delivery, and loyalty features.",
    "AlloFresh": "AlloFresh is a grocery app by AlloFresh (CT Corp and Bukalapak). It offers online grocery and fresh produce delivery in Indonesia.",
    "HappyFresh": "HappyFresh is a grocery app by HappyFresh. It provides on-demand grocery delivery from supermarkets and specialty stores in Southeast Asia.",
    "Indomaret Poinku": "Indomaret Poinku is a grocery app by PT Indomarco Prismatama. It offers loyalty rewards, promotions, and digital coupons for Indomaret convenience stores.",
    "Klik Indomaret": "Klik Indomaret is a grocery app by PT Indomarco Prismatama. It provides online ordering and delivery from Indomaret stores.",
    "Sayurbox": "Sayurbox is a grocery app by PT Sayurbox Teknologi Indonesia. It offers farm-to-table fresh produce and grocery delivery.",
    "Segari": "Segari is a grocery app by PT Segari. It provides next-day fresh produce and grocery delivery in Indonesia.",
    "Walmart": "Walmart is a grocery app by Walmart Inc. It offers grocery pickup, delivery, and general merchandise e-commerce.",
    # Home & Living
    "Dekoruma": "Dekoruma is a home & living app by PT Dekoruma Inovasi Lestari. It offers furniture, home decor, and interior design services in Indonesia.",
    "Fabelio": "Fabelio was a home & living e-commerce brand in Indonesia focused on modern furniture and home goods.",
    "IKEA": "IKEA is a home & living app by Inter IKEA Group. It offers furniture and home furnishing browsing, room planning, and online ordering.",
    "Informa": "Informa is a home & living app by PT Informa Furnishings (Kawan Lama Group). It offers furniture and home living products.",
    "Lotte Home Shopping": "Lotte Home Shopping is a home & living app by Lotte Group. It provides TV and online home shopping for household goods and lifestyle products.",
    "Mitra10": "Mitra10 is a home & living app by PT Catur Mitra Sejati Sentosa (CSAP). It offers building materials, tools, and home improvement products.",
    "Nitori": "Nitori is a home & living app by Nitori Holdings (Japan). It offers affordable furniture and home decor retail.",
    "Olympic Furniture": "Olympic Furniture is a home & living app by PT Cahaya Sakti Furintraco. It offers ready-to-assemble furniture in Indonesia.",
    "Pepperfry": "Pepperfry is a home & living app by Pepperfry (India). It offers furniture and home decor e-commerce.",
    "Philips": "Philips is a brand app by Philips. Depending on the product line, it may provide smart-device control, product services, or brand-related consumer utilities.",
    # Automotive Market
    "BCA Merchant": "BCA Merchant is a merchant operations app by Bank Central Asia. It helps merchants manage onboarding, QRIS/EDC services, and transaction-related merchant activities.",
    "Carmudi": "Carmudi is an automotive market app by Carmudi. It offers new and used car listings and dealer connections in Indonesia.",
    "Carsome": "Carsome is an automotive market app by Carsome Group. It provides inspected and certified used car buying and selling across Southeast Asia.",
    "Garasi.id": "Garasi.id is an automotive market app by PT Garasi Digital Indonesia. It offers used car marketplace with inspection and financing.",
    "IMOVE": "IMOVE is an app name that may refer to different services by market. Use a safer app-specific description only after confirming the exact publisher or package/domain.",
    "MobilBekas": "MobilBekas is an automotive market app by MobilBekas.com. It provides classified listings for secondhand vehicles in Indonesia.",
    "Moladin": "Moladin is an automotive market app by PT Moladin Digital Indonesia. It offers used motorcycle marketplace with credit and inspection services.",
    "Momobil": "Momobil is an automotive market app by PT Adi Mobil Indonesia. It provides used car buying and selling with inspection services.",
    "Momotor": "Momotor is an automotive market app. It provides motorcycle marketplace listings in Indonesia.",
    "QQCar": "QQCar is a game app. It refers to a car-racing style mobile game rather than an automotive marketplace service.",
    "RajaMobil": "RajaMobil is an automotive market app by PT Raja Mobil Indonesia. It offers new and used car listings with dealer comparison.",
    # Seller Tools
    "Big Cartel": "Big Cartel is a seller tools app by Big Cartel. It provides simple e-commerce store building for independent creators and artists.",
    "GoBiz": "GoBiz is a seller tools app by Gojek (GoTo Group). It offers merchant management for restaurants and shops on the Gojek platform.",
    "Mitra Bukalapak": "Mitra Bukalapak is a seller tools app by PT Bukalapak.com Tbk. It empowers warung owners with digital product selling and wholesale purchasing.",
    "Mitra Tokopedia": "Mitra Tokopedia is a seller tools app by PT Tokopedia (GoTo Group). It provides kiosk and warung digitization with PPOB and wholesale services.",
    "Shopify": "Shopify is a seller tools app by Shopify Inc. It provides e-commerce platform for merchants to build online stores with payment and shipping management.",
    "TikTok Seller": "TikTok Seller is a seller tools app by ByteDance. It provides merchant management tools for selling products via TikTok Shop.",
    "iSeller": "iSeller is a seller tools app by PT iSeller Commerce. It offers omnichannel POS and e-commerce management for Indonesian retailers.",
    # B2B
    "GudangAda": "GudangAda is a B2B marketplace app by PT GudangAda Globalindo. It connects FMCG brands, distributors, and warung retailers for wholesale procurement.",
    "Indotrading": "Indotrading is a B2B marketplace app by PT Indotrading. It provides B2B supplier directory and product sourcing for Indonesian businesses.",
    "Ralali": "Ralali is a B2B marketplace app by PT Raksasa Laju Lintang. It offers industrial and business supplies procurement in Indonesia.",
    # Property
    "Lamudi": "Lamudi is a property marketplace app by Lamudi. It provides property listings for buying and renting homes and commercial spaces in Indonesia.",
    "Rumah123": "Rumah123 is a property marketplace app by PT Web Marketing Indonesia (99 Group). It offers residential property listings for sale and rent across Indonesia.",
    # Review Platform
    "IMDb": "IMDb is a review platform app by Amazon (IMDb.com). It provides movie and TV show ratings, reviews, cast information, and watchlists.",
    "Tripadvisor": "Tripadvisor is a review platform app by Tripadvisor Inc. It offers user reviews and bookings for hotels, restaurants, and attractions worldwide.",
}

print(f"DESC_COMMERCE: {len(DESC_COMMERCE)} entries")


DESC_COMMERCE: 120 entries


In [123]:
# =============================================================================
# DESCRIPTIONS DICTIONARY — Communication
# =============================================================================
DESC_COMMUNICATION = {
    # Instant Messaging
    "WhatsApp": "WhatsApp is an instant messaging app by Meta Platforms. It provides end-to-end encrypted text messaging, voice calls, video calls, and group chats.",
    "LINE": "LINE is an instant messaging app by LY Corporation. It offers messaging, voice and video calls, stickers, and integrated lifestyle services.",
    "WeChat": "WeChat is an instant messaging app by Tencent. It combines messaging, social networking, payments, and mini-programs in one platform.",
    "Discord": "Discord is an instant messaging app by Discord Inc. It provides text, voice, and video chat organized into servers and channels.",
    "Signal": "Signal is an instant messaging app by Signal Foundation. It offers privacy-focused messaging and calls with end-to-end encryption.",
    "Viber": "Viber is an instant messaging app by Rakuten Viber. It provides messaging, voice calls, video calls, and group chats.",
    "KakaoTalk": "KakaoTalk is an instant messaging app by Kakao Corp. It offers messaging, voice calls, and integrated digital services.",
    "QQ": "QQ is an instant messaging app by Tencent. It provides messaging, file sharing, and social communication features.",
    "Zalo": "Zalo is an instant messaging app by VNG Corporation. It offers messaging, voice and video calls, and social communication features.",
    "Zello": "Zello is an instant messaging app by Zello Inc. It provides push-to-talk walkie-talkie communication over mobile data and Wi-Fi.",
    "BOTIM": "BOTIM is an instant messaging app by Algento Cloud Computing FZ-LLC. It provides messaging and internet voice and video calling.",
    "BiP": "BiP is an instant messaging app by Turkcell. It offers messaging, voice and video calls, and additional communication features.",
    "FaceTime": "FaceTime is a communication app by Apple. It provides audio and video calling across Apple devices.",
    "Google Chat": "Google Chat is an instant messaging app by Google. It offers team messaging and collaboration integrated with Google Workspace.",
    "IMO": "IMO is an instant messaging app by imo.im. It provides messaging and voice and video calls optimized for mobile use.",
    "MiChat": "MiChat is an instant messaging app by MiChat Pte. Ltd. It offers messaging, group chats, and nearby-user discovery features.",
    "iMessage": "iMessage is an instant messaging service by Apple. It provides encrypted messaging, media sharing, and communication features across Apple devices.",

    # Social Network
    "Facebook": "Facebook is a social network app by Meta Platforms. It provides social networking with feeds, groups, marketplace, and events.",
    "Instagram": "Instagram is a social network app by Meta Platforms. It offers photo and video sharing, Stories, Reels, and direct messaging.",
    "LinkedIn": "LinkedIn is a social network app by LinkedIn Corporation, owned by Microsoft. It provides professional networking, job discovery, and career development tools.",
    "X (Twitter)": "X (Twitter) is a social network app by X Corp. It provides real-time posting, replies, communities, and live conversation features.",
    "Threads": "Threads is a social network app by Meta Platforms. It offers text-based social posting linked to Instagram accounts.",
    "Snapchat": "Snapchat is a social network app by Snap Inc. It provides disappearing photo and video messaging, Stories, AR lenses, and creator content discovery.",
    "Reddit": "Reddit is a social network app by Reddit Inc. It offers community-driven discussion forums across a wide range of topics.",
    "Quora": "Quora is a social network app by Quora Inc. It provides a question-and-answer platform where users share knowledge and opinions.",
    "Kaskus": "Kaskus is a social network app by PT Darta Media Indonesia. It is an Indonesian online forum platform for discussions and community interaction.",
    "Bluesky": "Bluesky is a social network app by Bluesky PBLLC. It provides decentralized microblogging built on the AT Protocol.",
    "Disqus": "Disqus is a social network app by Disqus. It provides website comment hosting and online discussion community features.",
    "Dubsmash": "Dubsmash was a social video app. It offered short lip-sync and dance video creation and sharing.",
    "Facebook Messenger": "Facebook Messenger is a messaging app by Meta Platforms. It provides text messaging, voice and video calls, and chat features connected to Facebook accounts.",
    "Helo": "Helo was a regional social media and content aggregation platform owned by ByteDance, designed for short-form videos, photo sharing, and news, similar to a localized Facebook or Twitter. Launched in 2018 in India and later in Indonesia, it focused on local content and community engagement, but ultimately shut down operations around mid-2023 due to lack of market fit.",
    "MICO": "MICO is a social network app by Mico World. It provides stranger chat, live streaming, party rooms, and social discovery features.",
    "MeetMe": "MeetMe is a social network app by The Meet Group. It offers social discovery through chat and live streaming.",
    "Meipai": "Meipai is a social video community app by Meitu. It provides short video creation and sharing, with a community focused on women users.",
    "OmeTV": "OmeTV is a social discovery app by OmeTV. It provides random video chat with strangers worldwide.",
    "Smule": "Smule is a social music app by Smule Inc. It offers karaoke singing, duets, and collaborative music creation.",
    "Wattpad": "Wattpad is a social storytelling app by Wattpad WEBTOON Studios. It provides a community for reading and writing user-generated stories.",
    "Weverse": "Weverse is a social network app by WEVERSE COMPANY, part of HYBE. It provides artist-fan community interaction and exclusive fan content.",
    "YY": "YY is a social entertainment app by JOYY Inc. It provides live streaming and interactive online community features.",

    # Dating & Discovery
    "Tinder": "Tinder is a dating and discovery app by Match Group. It provides swipe-based matching for dating and social connections.",
    "Bumble": "Bumble is a dating and discovery app by Bumble Inc. It offers dating, friendship, and professional networking with a women-first messaging model.",
    "Badoo": "Badoo is a dating and discovery app by Badoo. It offers profile-based social discovery and dating features.",
    "Coffee Meets Bagel": "Coffee Meets Bagel is a dating and discovery app by Coffee Meets Bagel. It provides curated daily match suggestions for relationship-focused dating.",
    "AsianDate": "AsianDate is a dating and discovery app. It provides international dating and chat focused on connecting with Asian singles.",
    "DOWN Dating": "DOWN Dating is a dating and discovery app by DOWN. It offers matching for casual dating and relationships.",
    "Dating.com": "Dating.com is a dating and discovery app by Dating.com. It provides international online dating and matchmaking features.",
    "Litmatch": "Litmatch is a dating and discovery app by CONSTRUCT TECHNOLOGY. It offers voice-based social matching and chat features popular with younger users.",
    "LovePlanet": "LovePlanet is a dating and discovery app by LovePlanet LLC. It provides dating, chat, and live video discovery features.",
    "Lovoo": "Lovoo is a dating and discovery app by LOVOO GmbH. It provides swipe-based dating with chat, video, and livestream features.",
    "Meetic": "Meetic is a dating and discovery app by Meetic. It provides online dating focused on serious relationships in European markets.",
    "Omi": "Omi is a dating and discovery app by Omi. It provides swipe-based dating, blind-date style matching, and personality-based discovery features.",
    "Pinterest": "Pinterest is a visual discovery app by Pinterest. It provides idea discovery, saving, and inspiration boards for fashion, recipes, home, and projects.",
    "TanTan": "TanTan is a dating and discovery app by Tantan. It provides swipe-based dating and social matching.",
    "Zoosk": "Zoosk is a dating and discovery app by Zoosk. It offers online dating with behavioral matchmaking features.",

    # Short Video & Live
    "TikTok": "TikTok is a short video and live app by ByteDance. It provides short-form video creation, sharing, live streaming, and algorithm-driven content discovery.",
    "SnackVideo": "SnackVideo is a short video and live app by Kuaishou Technology. It offers short video creation and browsing.",
    "BIGO Live": "BIGO Live is a short video and live app by BIGO Technology. It provides live streaming, virtual gifting, and social interaction.",
    "Likee": "Likee is a short video and live app by Likee Technology. It offers short video creation with effects, music, and live streaming features.",
    "Nimo TV": "Nimo TV is a live streaming app by HUYA. It provides game live streaming and creator broadcasting features.",
    "BOOYAH!": "BOOYAH! is a gaming video and live app by Garena. It provides game clip sharing, highlights, and live streaming for gamers.",
    "ReelShort": "ReelShort is a short drama video app by Crazy Maple Studio. It offers bite-sized vertical drama episodes for mobile viewing.",
    "ShortTV": "ShortTV is a short drama video app. It provides vertical mini-dramas (often called micro-dramas or duanju) designed specifically for smartphones. Unlike traditional TV, these series are filmed in a vertical 9:16 format so you can watch them without rotating your phone.",
    "FlickReels": "FlickReels is a short drama video app. It provides vertical short drama series and mini-shows for mobile viewers.",
    "GoodShort": "GoodShort is a short drama video app by GoodNovel. It provides short original dramas and movie-style vertical episodes.",
    "MoboReels": "MoboReels is a short drama video app by MOBOREADER TECHNOLOGY USA CO LTD. It provides short movies, microdramas, and vertical drama content.",
    "NetShort": "NetShort is a short drama video app by NETSTORY PTE. LTD. It provides vertically formatted mobile drama series and mini-episodes.",
    "Vigo Video": "Vigo Video (formerly known as Hypstar), was a popular short-form video platform similar to TikTok that allowed users to create and share 15-second clips. However, it is important to note that its original global version was officially shut down by its parent company, ByteDance, in October 2020.",
}

print(f"DESC_COMMUNICATION: {len(DESC_COMMUNICATION)} entries")

DESC_COMMUNICATION: 67 entries


In [124]:
# =============================================================================
# DESCRIPTIONS DICTIONARY — Entertainment
# =============================================================================
DESC_ENTERTAINMENT = {
    # Video Streaming
    "YouTube": "YouTube is a video streaming app by Google (Alphabet). It provides user-generated and professional video content, live streaming, and Shorts.",
    "Netflix": "Netflix is a video streaming app by Netflix Inc. It offers subscription-based on-demand movies, series, and original productions worldwide.",
    "Disney+": "Disney+ is a video streaming app by The Walt Disney Company. It streams Disney, Pixar, Marvel, Star Wars, and National Geographic content.",
    "Amazon Prime Video": "Amazon Prime Video is a video streaming app by Amazon. It offers on-demand movies, series, and originals with Prime membership.",
    "Vidio": "Vidio is a video streaming app by PT Vidio Dot Com (Emtek Group). It streams Indonesian dramas, sports (Liga 1), and live TV.",
    "HBO Max": "HBO Max is a video streaming app by Warner Bros. Discovery. It offers HBO originals, movies, and curated content.",
    "Apple TV": "Apple TV is a video streaming app by Apple. It provides Apple TV+ originals and a hub for purchasing/renting movies and shows.",
    "Viu": "Viu is a video streaming app by PCCW (Hong Kong). It offers Asian dramas, Korean content, and local originals in Southeast Asia.",
    "WeTV": "WeTV is a video streaming app by Tencent. It streams Chinese and Asian dramas, anime, and variety shows for Southeast Asian audiences.",
    "iQIYI": "iQIYI is a video streaming app by Baidu (iQIYI Inc). It provides Chinese dramas, movies, and variety shows with subtitles for global audiences.",
    "Bilibili": "Bilibili is a video streaming app by Bilibili Inc (China). It provides anime, gaming, and user-generated content with a bullet-comment system.",
    "Twitch": "Twitch is a video streaming app by Amazon (Twitch Interactive). It provides live game streaming, esports, and creator community content.",
    "RCTI+": "RCTI+ is a video streaming app by PT MNC Digital Entertainment. It offers live TV, catch-up content from RCTI, MNCTV, and GTV channels.",
    "Vision+": "Vision+ is a video streaming app by MNC Vision Networks. It provides live TV channels and on-demand content from MNC Group.",
    "Hotstar": "Hotstar is a video streaming app by Disney+ Hotstar (Disney Star, India). It offers cricket streaming, Bollywood, and Disney content.",
    "Viki": "Viki is a video streaming app by Rakuten Viki. It provides Asian dramas and movies with community-contributed subtitles in 200+ languages.",
    "MOLA TV": "MOLA TV is a video streaming app by PT Mola Television Indonesia. It offers premium sports (Premier League) and entertainment content.",
    "KlikFilm": "KlikFilm is a video streaming app by PT Klikfilm Indonesia. It streams curated independent Indonesian and international films.",
    "Genflix": "Genflix is a video streaming app by PT Mediatama Anugrah Citra. It offers Indonesian movies, series, and live TV streaming.",
    "Sushiroll": "Sushiroll is a video streaming app by PT Sushiroll Media. It streams anime content for Indonesian audiences.",
    "DramaBox": "DramaBox is a video streaming app by Storymatrix. It offers short-form episodic drama content in bite-sized vertical format.",
    "DramaWave": "DramaWave is a video streaming app. It provides short-form drama series for mobile streaming.",
    "Dailymotion": "Dailymotion is a video streaming app by Vivendi. It offers user-uploaded and publisher video content.",
    "Tubi": "Tubi is a video streaming app by Fox Corporation. It provides free ad-supported movies and TV shows.",
    "Plex": "Plex is a video streaming app by Plex Inc. It provides personal media server streaming and free ad-supported movies/TV.",
    "Vimeo": "Vimeo is a video streaming app by Vimeo Inc. It provides high-quality video hosting for creators and businesses.",
    "VidMate": "VidMate is a video streaming app by VidMate Studio. It provides video discovery and download from multiple streaming sources.",
    "Vidiostream": "Vidiostream is a video streaming app. It provides video streaming services.",
    "Hooq": "Hooq is a video streaming app by Hooq Digital (former Singtel/Sony/Warner Bros JV). It offered Asian and Hollywood content streaming in Southeast Asia.",
    "Insert Live": "Insert Live is a video streaming app by Trans Media Group. It provides celebrity news and entertainment content from Insert program.",
    "Kick": "Kick is a video streaming app by Kick.com. It provides live streaming for gamers and creators with a creator-friendly revenue model.",
    "Loklok": "Loklok is a video streaming app by Loklok. It provides free Asian drama, anime, and movie streaming with subtitles.",
    "Mango TV": "Mango TV is a video streaming app by Hunan Broadcasting (China). It offers Chinese variety shows, dramas, and reality content.",
    "MonoMax": "MonoMax is a video streaming app by Mono Group (Thailand). It streams Thai and international movies and series.",
    "MovieBox": "MovieBox is a video streaming app. It provides movie and TV show streaming via a mobile application.",
    "Ocean of Movies": "Ocean of Movies is a video streaming app. It provides movie streaming content.",
    "Periscope": "Periscope is a video streaming app by Twitter (X Corp). It provided mobile live video broadcasting.",
    "RapidTV": "RapidTV is a video streaming app. It provides video streaming services.",
    "Sony LIV": "Sony LIV is a video streaming app by Sony Pictures Networks India. It streams sports (cricket), Hindi originals, and movies.",
    "Stardust TV": "Stardust TV is a video streaming app. It provides entertainment and celebrity content streaming.",
    "Tango": "Tango is a video streaming app by Tango (TangoMe). It provides live streaming with virtual gifts and social features.",
    "ViuTV": "ViuTV is a video streaming app by PCCW (Hong Kong). It offers live and on-demand Hong Kong free-to-air TV content.",
    "Vuclip": "Vuclip is a video streaming app by Vuclip (PCCW). It provides mobile video streaming optimized for emerging markets.",
    "Yandex Video": "Yandex Video is a video streaming app by Yandex (Russia). It provides video search and streaming aggregation.",
    "YouTube Kids": "YouTube Kids is a video streaming app by Google. It provides a curated, child-safe video experience with parental controls.",
    "Youku": "Youku is a video streaming app by Alibaba Group. It offers Chinese dramas, movies, and variety shows.",
    "beIN SPORTS": "beIN SPORTS is a video streaming app by beIN Media Group. It provides live sports coverage including football, tennis, and motorsport.",
    "Google Play Movies": "Google Play Movies is a video streaming app by Google. It offers movie and TV show rental and purchase.",
    "iflix": "iflix is a video streaming app by iflix (now part of WeTV). It offered free and premium Asian entertainment streaming.",
    # Music Streaming
    "Spotify": "Spotify is a music streaming app by Spotify Technology SA. It provides on-demand music, podcasts, and personalized playlists.",
    "YouTube Music": "YouTube Music is a music streaming app by Google. It offers music streaming with official songs, albums, music videos, and remixes.",
    "JOOX": "JOOX is a music streaming app by Tencent. It offers free and premium music streaming popular in Southeast Asia.",
    "SoundCloud": "SoundCloud is a music streaming app by SoundCloud. It provides a platform for independent artists to upload and share music.",
    "Deezer": "Deezer is a music streaming app by Deezer S.A. It provides music streaming with Hi-Fi audio and Flow personalized playlist.",
    "TIDAL": "TIDAL is a music streaming app by Block Inc (formerly Square). It offers high-fidelity lossless music streaming and exclusive content.",
    "Apple Music": "Apple Music is a music streaming app by Apple. It provides music streaming, curated playlists, radio, and spatial audio.",
    "Boomplay": "Boomplay is a music streaming app by Transsnet Music (Transsion). It provides music streaming focused on African and emerging market content.",
    "QQ Music": "QQ Music is a music streaming app by Tencent Music Entertainment. It offers music streaming with one of the largest Chinese music libraries.",
    "NOICE": "NOICE is a music streaming app by PT NOICE Teknologi Indonesia. It provides Indonesian podcasts, audio dramas, and audiobook streaming.",
    "Google Play Music": "Google Play Music is a music streaming app by Google. It offered music purchase, upload, and subscription streaming (now replaced by YouTube Music).",
    "AZLyrics": "AZLyrics is a music streaming app by AZLyrics.com. It provides a searchable database of song lyrics.",
    "Audible": "Audible is a music streaming app by Amazon (Audible). It provides audiobook and podcast listening with subscription and purchase options.",
    "Genius": "Genius is a music streaming app by Genius Media Group. It provides annotated song lyrics, music news, and artist interviews.",
    "Kuwo Music": "Kuwo Music is a music streaming app by Kuwo (Tencent Music). It offers music streaming services in China.",
    "Melon": "Melon is a music streaming app by Kakao Entertainment (South Korea). It is South Korea's leading music streaming platform.",
    "Musixmatch": "Musixmatch is a music streaming app by Musixmatch. It provides synchronized real-time lyrics for songs across streaming services.",
    "Podbean": "Podbean is a music streaming app by Podbean Tech. It provides podcast hosting, distribution, and listening services.",
    "Radio FM": "Radio FM is a music streaming app. It provides FM radio station streaming via mobile.",
    "Simfy Africa": "Simfy Africa is a music streaming app by Simfy Africa. It offers music streaming services in African markets.",
    "Suno": "Suno is a music streaming app by Suno Inc. It provides AI-powered music generation from text prompts.",
    "Trebel": "Trebel is a music streaming app by Trebel Music. It provides free ad-supported offline music downloads for emerging markets.",
    "Yandex Music": "Yandex Music is a music streaming app by Yandex (Russia). It offers music streaming and personalized recommendations.",
    "Zing MP3": "Zing MP3 is a music streaming app by VNG Corporation (Vietnam). It provides music streaming for Vietnamese audiences.",
    "iHeartRadio": "iHeartRadio is a music streaming app by iHeartMedia. It provides live radio, podcasts, and custom playlists.",
    # Comics & Webtoon
    "LINE Webtoon": "LINE Webtoon is a comics & webtoon app by Naver/LY Corporation. It provides free vertical-scroll webcomics across genres worldwide.",
    "MangaToon": "MangaToon is a comics & webtoon app by MangaToon. It provides free manga, manhua, and novel reading with daily updates.",
    "WebComics": "WebComics is a comics & webtoon app by WebComics. It provides manga and webtoon reading with a community feature.",
    "Manga UP!": "Manga UP! is a comics & webtoon app by Square Enix. It provides free manga reading from Square Enix's catalog.",
    "MangaFox": "MangaFox is a comics & webtoon app. It provides online manga reading across various genres.",
    "MangaTown": "MangaTown is a comics & webtoon app. It provides free online manga reading with a community forum.",
    "Mangaku": "Mangaku is a comics & webtoon app. It provides Indonesian-translated manga reading.",
    # Comedy & Memes
    "9GAG": "9GAG is a comedy & memes app by 9GAG. It provides user-generated memes, funny images, and viral content sharing.",
    "1Cak": "1Cak is a comedy & memes app by PT 1Cak Media. It is Indonesia's meme and humor sharing community, inspired by 9GAG.",
    "iFunny": "iFunny is a comedy & memes app by iFunny Inc. It provides a meme and funny content sharing community.",
    # Mobile Games
    "Mobile Legends": "Mobile Legends is a mobile game by Moonton (ByteDance). It is Southeast Asia's most popular 5v5 MOBA game.",
    "PUBG": "PUBG is a mobile game by KRAFTON/Tencent. It is a battle royale shooter where 100 players compete on a shrinking map.",
    "Free Fire": "Free Fire is a mobile game by Garena (Sea Group). It is a fast-paced battle royale shooter popular in emerging markets.",
    "Genshin Impact": "Genshin Impact is a mobile game by miHoYo/HoYoverse. It is an open-world action RPG with gacha mechanics and cross-platform play.",
    "Call of Duty": "Call of Duty is a mobile game by Activision (Microsoft). It offers first-person shooter multiplayer and battle royale modes.",
    "Roblox": "Roblox is a mobile game by Roblox Corporation. It is a user-generated gaming platform where players create and play community-built games.",
    "Minecraft": "Minecraft is a mobile game by Mojang Studios (Microsoft). It is an open-world sandbox game for building, exploration, and survival.",
    "Clash of Clans": "Clash of Clans is a mobile game by Supercell. It is a strategy game where players build villages, train troops, and raid other players.",
    "Candy Crush Saga": "Candy Crush Saga is a mobile game by King (Activision Blizzard). It is a match-three puzzle game with thousands of levels.",
    "Among Us": "Among Us is a mobile game by Innersloth. It is a social deduction party game where crewmates find impostors.",
    "Honkai Star Rail": "Honkai Star Rail is a mobile game by miHoYo/HoYoverse. It is a turn-based space fantasy RPG with gacha character collection.",
    "Honor of Kings": "Honor of Kings is a mobile game by TiMi Studio (Tencent). It is a 5v5 MOBA game, one of the world's highest-grossing mobile titles.",
    "Valorant": "Valorant is a mobile game by Riot Games. It is a tactical first-person shooter with character-based abilities.",
    "Fortnite": "Fortnite is a mobile game by Epic Games. It is a battle royale and creative sandbox game with cultural crossover events.",
    "League of Legends": "League of Legends is a mobile game by Riot Games. It is a 5v5 MOBA game with over 160 champions.",
    "Clash Royale": "Clash Royale is a mobile game by Supercell. It is a real-time strategy card game with tower defense combat.",
    "Brawl Stars": "Brawl Stars is a mobile game by Supercell. It is a fast-paced multiplayer brawler with various game modes.",
    "8 Ball Pool": "8 Ball Pool is a mobile game by Miniclip. It is an online multiplayer billiards simulation game.",
    "Subway Surfers": "Subway Surfers is a mobile game by SYBO Games. It is an endless runner where players dash through subway tracks.",
    "Stumble Guys": "Stumble Guys is a mobile game by Kitka Games (Scopely). It is a multiplayer party knockout game with obstacle courses.",
    "Dream League Soccer": "Dream League Soccer is a mobile game by First Touch Games. It is a football management and gameplay simulation.",
    "Angry Birds 2": "Angry Birds 2 is a mobile game by Rovio (SEGA). It is a physics-based puzzle game where birds are launched at pig structures.",
    "Hay Day": "Hay Day is a mobile game by Supercell. It is a farming simulation where players grow crops and trade with neighbors.",
    "FIFA Mobile": "FIFA Mobile is a mobile game by EA Sports. It offers licensed football gameplay with team building and live events.",
    "eFootball": "eFootball is a mobile game by Konami. It is a free-to-play football simulation with licensed teams and real-time PvP.",
    "Dota 2": "Dota 2 is a mobile game by Valve Corporation. It is a complex 5v5 MOBA with deep strategy and esports scene.",
    "Ludo King": "Ludo King is a mobile game by Gametion Technologies. It is a digital board game based on the classic Ludo.",
    "Ragnarok Online": "Ragnarok Online is a mobile game by Gravity Co (South Korea). It is an MMORPG based on the classic Ragnarok franchise.",
    "Rise of Kingdoms": "Rise of Kingdoms is a mobile game by Lilith Games. It is a real-time strategy civilization-building game with PvP warfare.",
    "Top War": "Top War is a mobile game by Topwar Studio. It is a merge-to-upgrade strategy war game.",
    "Hearthstone": "Hearthstone is a mobile game by Blizzard Entertainment. It is a digital collectible card game set in the Warcraft universe.",
    "Pokemon Go": "Pokemon Go is a mobile game by Niantic. It is an augmented-reality game where players catch Pokemon in real-world locations.",
    "SimCity BuildIt": "SimCity BuildIt is a mobile game by EA. It is a city-building simulation where players design and manage a metropolis.",
    "Cookie Run": "Cookie Run is a mobile game by Devsisters. It is an endless runner and kingdom-building game featuring cookie characters.",
    "Auto Chess": "Auto Chess is a mobile game by Dragonest. It is an auto-battler strategy game with chess-piece character placement.",
    "Black Myth: Wukong": "Black Myth: Wukong is a mobile game by Game Science. It is an action RPG based on the Chinese classic Journey to the West.",
    "Blood Strike": "Blood Strike is a mobile game by NetEase. It is an FPS battle royale game with fast-paced combat.",
    "Blockman Go": "Blockman Go is a mobile game by Blockman Go Studio. It provides a platform of mini-games including bed wars and skyblock.",
    "Brawlhalla": "Brawlhalla is a mobile game by Blue Mammoth (Ubisoft). It is a free-to-play platform fighting game.",
    "Chess.com": "Chess.com is a mobile game by Chess.com LLC. It provides online chess matches, puzzles, lessons, and tournaments.",
    "Clash of Kings": "Clash of Kings is a mobile game by Elex Tech. It is an MMO strategy game with castle building and alliance warfare.",
    "Dragon Ball Legends": "Dragon Ball Legends is a mobile game by Bandai Namco. It is an action fighting game featuring Dragon Ball characters.",
    "Dragon Nest": "Dragon Nest is a mobile game by Eyedentity Games. It is an action MMORPG with fast-paced combo combat.",
    "Epic Seven": "Epic Seven is a mobile game by Smilegate/Super Creative. It is an anime-style turn-based RPG with gacha mechanics.",
    "Family Island": "Family Island is a mobile game by Melsoft Games. It is a farming adventure game set on a prehistoric island.",
    "Fishdom": "Fishdom is a mobile game by Playrix. It is a match-three puzzle game with aquarium decoration.",
    "Football Strike": "Football Strike is a mobile game by Miniclip. It is a multiplayer free-kick and goalkeeper football game.",
    "Gardenscapes": "Gardenscapes is a mobile game by Playrix. It is a match-three puzzle game with garden renovation storyline.",
    "Hago": "Hago is a mobile game by Hago (YY/JOYY). It is a social gaming app offering mini-games with voice chat.",
    "Happy Color": "Happy Color is a mobile game by X-Flow. It is a paint-by-numbers coloring game.",
    "Hero Wars": "Hero Wars is a mobile game by Nexters. It is an idle RPG hero-collection battler.",
    "Homescapes": "Homescapes is a mobile game by Playrix. It is a match-three puzzle game with home renovation storyline.",
    "Hungry Shark World": "Hungry Shark World is a mobile game by Ubisoft. It is an arcade game where players control a shark in an open ocean.",
    "Injustice 2": "Injustice 2 is a mobile game by Warner Bros. Games (NetherRealm). It is a fighting game featuring DC Comics superheroes.",
    "Kahoot!": "Kahoot! is a mobile game by Kahoot! ASA. It is a gamified quiz and learning platform for education and trivia.",
    "Mario Kart Tour": "Mario Kart Tour is a mobile game by Nintendo. It is a kart racing game featuring Mario characters and real-world city tracks.",
    "One Piece Bounty Rush": "One Piece Bounty Rush is a mobile game by Bandai Namco. It is a 4v4 treasure-looting action game with One Piece characters.",
    "Plants vs Zombies 2": "Plants vs Zombies 2 is a mobile game by PopCap (EA). It is a tower defense game where plants defend against zombie waves.",
    "Plato": "Plato is a mobile game by Plato Team. It provides multiplayer social games including Werewolf, Draw Together, and card games.",
    "Pokemon": "Pokemon is a mobile game by The Pokemon Company. It provides various Pokemon game experiences on mobile.",
    "Pokemon Unite": "Pokemon Unite is a mobile game by TiMi Studios (Tencent) and The Pokemon Company. It is a 5v5 MOBA featuring Pokemon.",
    "Slither.io": "Slither.io is a mobile game by Steve Howse. It is a multiplayer snake-style io game where players grow by consuming orbs.",
    "Sniper 3D Assassin": "Sniper 3D Assassin is a mobile game by Wildlife Studios. It is a first-person sniper shooting game with mission-based gameplay.",
    "Solo Leveling: Arise": "Solo Leveling: Arise is a mobile game by Netmarble. It is an action RPG based on the Solo Leveling manhwa franchise.",
    "Squad Busters": "Squad Busters is a mobile game by Supercell. It is a multiplayer action game featuring characters from Supercell's game universe.",
    "Super Mario Run": "Super Mario Run is a mobile game by Nintendo. It is a side-scrolling runner featuring Mario with one-handed gameplay.",
    "Tennis Clash": "Tennis Clash is a mobile game by Wildlife Studios. It is a real-time multiplayer tennis sports game.",
    "The Spike": "The Spike is a mobile game by HIGH-X. It is a volleyball simulation game with spike timing mechanics.",
    "Toon Blast": "Toon Blast is a mobile game by Peak Games (Zynga). It is a cartoon-themed puzzle game with cube-matching mechanics.",
    "Top Eleven": "Top Eleven is a mobile game by Nordeus. It is a football manager simulation game with real-time multiplayer.",
    "Township": "Township is a mobile game by Playrix. It combines city-building with farming and resource management.",
    "Toy Blast": "Toy Blast is a mobile game by Peak Games (Zynga). It is a puzzle game with toy-themed block matching.",
    "UNO!": "UNO! is a mobile game by Mattel163. It is a digital version of the classic UNO card game with online multiplayer.",
    "Akinator": "Akinator is a mobile game by Elokence. It is a guessing game where an AI genie tries to identify a character the player is thinking of.",
    "Blizzard": "Blizzard is a mobile game by Blizzard Entertainment (Microsoft). It covers Blizzard's mobile ports including Diablo Immortal and Hearthstone.",
    "Gameloft": "Gameloft is a mobile game by Gameloft (Vivendi). It develops and publishes mobile games including Asphalt and Dungeon Hunter series.",
    # Gaming Platform
    "Steam": "Steam is a gaming platform app by Valve Corporation. It is the world's largest PC game distribution platform with community and workshop features.",
    "PlayStation": "PlayStation is a gaming platform app by Sony Interactive Entertainment. It provides PS Store access, friend management, and Remote Play from PlayStation consoles.",
    "Xbox": "Xbox is a gaming platform app by Microsoft. It provides Xbox Game Pass, social features, and remote play from Xbox consoles.",
    "Epic Games": "Epic Games is a gaming platform app by Epic Games. It provides the Epic Games Store, Fortnite launcher, and Unreal Engine access.",
    "Garena": "Garena is a gaming platform app by Garena (Sea Group). It serves as a game distribution and community platform for Southeast Asia.",
    "Riot Games": "Riot Games is a gaming platform app by Riot Games (Tencent). It provides game launching and management for League of Legends, Valorant, and more.",
    "TapTap": "TapTap is a gaming platform app by TapTap. It is an independent game discovery and distribution platform for mobile games globally.",
    "Apple Game Center": "Apple Game Center is a gaming platform app by Apple. It provides game achievements, leaderboards, and multiplayer matchmaking for iOS games.",
    "Tencent Games": "Tencent Games is a gaming platform app by Tencent. It provides game distribution and management for Tencent's portfolio of mobile games.",
    "NetEase Games": "NetEase Games is a gaming platform app by NetEase. It provides game distribution for NetEase titles including Identity V and Naraka.",
    "Supercell": "Supercell is a gaming platform app by Supercell (Tencent). It provides access to Supercell games including Clash of Clans, Brawl Stars, and Hay Day.",
    "Netmarble": "Netmarble is a gaming platform app by Netmarble Corp. It distributes mobile games including Ni no Kuni, Marvel Future Fight, and more.",
    "BlueStacks": "BlueStacks is a gaming platform app by BlueStacks Inc. It is an Android emulator that allows mobile game play on PC and Mac.",
    "Nox Player": "Nox Player is a gaming platform app by Nox Limited. It is an Android emulator for running mobile games on desktop.",
    "AHAGames": "AHAGames is a gaming platform app. It provides mobile game distribution and launcher services.",
    "Bandai Namco": "Bandai Namco is a gaming platform app by Bandai Namco Entertainment. It distributes anime-based games including Dragon Ball and One Piece titles.",
    "EA Games": "EA Games is a gaming platform app by Electronic Arts. It provides game distribution for EA titles including FIFA, Madden, and Apex Legends.",
    "Fun Games For Free": "Fun Games For Free is a gaming platform app. It distributes casual free-to-play mobile games.",
    "G2A": "G2A is a gaming platform app by G2A.com. It is a marketplace for game keys, software, and digital goods.",
    "GamesBean": "GamesBean is a gaming platform app. It provides mobile game distribution.",
    "Huawei GameCenter": "Huawei GameCenter is a gaming platform app by Huawei. It distributes mobile games for Huawei device users.",
    "King": "King is a gaming platform app by King (Activision Blizzard). It develops and distributes casual games including Candy Crush and Farm Heroes.",
    "KingsGroup": "KingsGroup is a gaming platform app by KingsGroup Holdings. It develops strategy mobile games including Z Day and World War Rising.",
    "Konami": "Konami is a gaming platform app by Konami. It distributes mobile games including eFootball, Yu-Gi-Oh!, and Castlevania.",
    "Kuro Games": "Kuro Games is a gaming platform app by Kuro Games. It develops anime-style action RPGs including Punishing: Gray Raven and Wuthering Waves.",
    "Lilith Games": "Lilith Games is a gaming platform app by Lilith Games (China). It develops Rise of Kingdoms and AFK Arena.",
    "Ludia": "Ludia is a gaming platform app by Ludia (Jam City). It develops licensed mobile games including Jurassic World Alive.",
    "MY.GAMES": "MY.GAMES is a gaming platform app by MY.GAMES (VK Company). It distributes PC and mobile games from Russian and global studios.",
    "Miniclip": "Miniclip is a gaming platform app by Miniclip (Tencent). It develops casual mobile games including 8 Ball Pool and Subway Surfers.",
    "Moon Active": "Moon Active is a gaming platform app by Moon Active. It develops Coin Master, one of the top-grossing casual mobile games.",
    "Nexon": "Nexon is a gaming platform app by Nexon. It distributes online and mobile games including MapleStory and KartRider.",
    "Niantic": "Niantic is a gaming platform app by Niantic. It develops AR location-based games including Pokemon Go and Peridot.",
    "Nintendo Wii": "Nintendo Wii is a gaming platform app by Nintendo. It provides Wii-related game services and content.",
    "Nordeus": "Nordeus is a gaming platform app by Nordeus (Take-Two). It develops Top Eleven football manager.",
    "Paper Games": "Paper Games is a gaming platform app by Papergames (China). It develops story-driven mobile games including Shining Nikki.",
    "Peak Games": "Peak Games is a gaming platform app by Peak Games (Zynga). It develops puzzle games including Toon Blast and Toy Blast.",
    "Poki": "Poki is a gaming platform app by Poki BV. It provides a browser-based platform for free online casual games.",
    "QQ Games": "QQ Games is a gaming platform app by Tencent. It provides casual online games integrated with the QQ platform.",
    "Rockstar Games": "Rockstar Games is a gaming platform app by Rockstar Games (Take-Two). It distributes GTA, Red Dead Redemption, and other titles.",
    "Rovio": "Rovio is a gaming platform app by Rovio Entertainment (SEGA). It develops Angry Birds franchise games.",
    "SEGA": "SEGA is a gaming platform app by SEGA Corp. It distributes Sonic, Yakuza, and other SEGA franchise mobile games.",
    "Toca Boca": "Toca Boca is a gaming platform app by Toca Boca (Spin Master). It develops creative play apps for children.",
    "Ubisoft": "Ubisoft is a gaming platform app by Ubisoft. It distributes mobile versions of Assassin's Creed, Just Dance, and more.",
    "Vivo GameCenter": "Vivo GameCenter is a gaming platform app by Vivo. It provides game distribution for Vivo device users.",
    "Voodoo": "Voodoo is a gaming platform app by Voodoo (France). It develops hyper-casual mobile games.",
    "WithBuddies": "WithBuddies is a gaming platform app. It provides social multiplayer gaming experiences.",
    "Zynga": "Zynga is a gaming platform app by Zynga (Take-Two). It develops social games including FarmVille, Words With Friends, and Zynga Poker.",
    "miHoYo": "miHoYo is a gaming platform app by miHoYo/HoYoverse. It develops Genshin Impact, Honkai Star Rail, and Honkai Impact 3rd.",
}

print(f"DESC_ENTERTAINMENT: {len(DESC_ENTERTAINMENT)} entries")

DESC_ENTERTAINMENT: 208 entries


In [125]:
# =============================================================================
# DESCRIPTIONS DICTIONARY — Health & Wellness
# =============================================================================
DESC_HEALTH = {
    # Healthcare & Telemedicine
    "Halodoc": "Halodoc is a healthcare & telemedicine app by PT Media Dokter Investama. It provides doctor consultations, medicine delivery, lab bookings, and other health services in Indonesia.",
    "SehatQ": "SehatQ is a healthcare & telemedicine app by PT SehatQ Harsana Emedika. It offers doctor consultations, health articles, and healthcare provider discovery.",
    "Alomedika": "Alomedika is a healthcare & telemedicine app by Alodokter Group. It provides medical education, clinical references, and discussion tools for healthcare professionals in Indonesia.",
    "K24 Klik Apotek": "K24 Klik Apotek is a healthcare & telemedicine app by PT K-24 Indonesia. It provides online pharmacy ordering and medicine delivery services.",
    "HelloSehat": "HelloSehat is a healthcare & telemedicine app by Hello Health Group. It offers health articles, wellness information, and educational health content.",
    "Buku Bumil": "Buku Bumil is a healthcare & telemedicine app. It provides pregnancy tracking, fetal development information, maternal health tools, and community features for Indonesian mothers.",
    "Clue": "Clue is a healthcare & telemedicine app by BioWink GmbH. It provides menstrual cycle tracking and reproductive health insights.",
    "FamilyDoctor": "FamilyDoctor is a healthcare & telemedicine app. It provides doctor appointment booking, clinic access, and telehealth-related services.",
    "FatSecret": "FatSecret is a healthcare & telemedicine app by FatSecret. It provides calorie counting, food diary, nutrition tracking, and weight management tools.",
    "Flo Health": "Flo Health is a healthcare & telemedicine app by Flo Health Inc. It offers period tracking, ovulation prediction, and reproductive health insights.",
    "Healthline": "Healthline is a healthcare & telemedicine app by Healthline Media. It provides evidence-based health information and wellness articles.",
    "Lifesum": "Lifesum is a healthcare & telemedicine app by Lifesum AB. It provides personalized nutrition planning, calorie tracking, and healthy eating guidance.",
    "Mannings": "Mannings is a healthcare & telemedicine app by Dairy Farm Company Limited. It offers health and beauty product shopping, promotions, and retail services.",
    "MeiYou": "MeiYou is a healthcare & telemedicine app. It provides period tracking and women's health management features.",
    "MySejahtera": "MySejahtera is a healthcare & telemedicine app by the Government of Malaysia. It provides public health services, health records, and government-linked health management features.",
    "Ovia Health": "Ovia Health is a healthcare & telemedicine app by Ovia Health. It offers fertility, pregnancy, and parenting tracking tools.",
    "Pregnancy Tracker": "Pregnancy Tracker is a healthcare & telemedicine app. It provides week-by-week pregnancy monitoring and fetal development information.",
    "Teman Bumil": "Teman Bumil is a healthcare & telemedicine app by Teman Bumil. It provides pregnancy guidance, maternal health content, and parenting support for Indonesian mothers.",
    "Tencent Health": "Tencent Health is a healthcare & telemedicine app by Tencent. It provides online medical consultations, health records management, and integration with public health services in China.",
    "iHerb": "iHerb is a healthcare & telemedicine app by iHerb. It offers online shopping for vitamins, supplements, and natural health products.",

    # Fitness & Sport
    "Strava": "Strava is a fitness & sport app by Strava Inc. It provides GPS activity tracking for running, cycling, and other sports with social and leaderboard features.",
    "MyFitnessPal": "MyFitnessPal is a fitness & sport app by MyFitnessPal. It offers calorie counting, food logging, and exercise tracking.",
    "DAZN": "DAZN is a fitness & sport app by DAZN Group. It provides live and on-demand sports streaming.",
    "ClassPass": "ClassPass is a fitness & sport app by ClassPass. It provides access to fitness classes, gyms, and wellness services through membership plans.",
    "FitHub": "FitHub is a fitness & sport app by PT Jaya Digital Properti. It provides gym membership access, class booking, and workout-related services.",
    "Bodybuilding.com": "Bodybuilding.com is a fitness & sport app by Bodybuilding.com. It provides workout plans, fitness programs, and nutrition guidance.",
    "Borobudur Marathon": "Borobudur Marathon is a fitness & sport app. It provides event registration, race information, activity tracking, and live tracking for the Borobudur Marathon.",
    "FIFA": "FIFA is a fitness & sport app by FIFA. It provides football news, live match information, fantasy features, and FIFA tournament content.",
    "FTL Gym": "FTL Gym is a fitness & sport app. It provides gym membership access, class booking, and personal training management.",
    "FitBest": "FitBest is a fitness & sport app. It provides online fitness programs and wellness guidance.",
    "Fitness": "Fitness is a fitness & sport app by Apple Inc. (or Google, depending on OS). It tracks daily activity, workout metrics, and rings-based progress to encourage physical movement.",
    "FlashScore": "FlashScore is a fitness & sport app by Livesport. It provides real-time sports scores, fixtures, standings, and match updates.",
    "FotMob": "FotMob is a fitness & sport app by FotMob. It provides live football scores, news, statistics, and match analysis.",
    "Fox Sports": "Fox Sports is a fitness & sport app by Fox Corporation. It provides sports news, scores, and live sports-related content.",
    "LiveScore": "LiveScore is a fitness & sport app by LiveScore Group. It provides live sports scores, fixtures, and match updates.",
    "LiveScore Group": "LiveScore Group is a fitness & sport organization that manages the LiveScore, LiveScore Bet, and Virgin Bet brands, providing real-time sports data and betting services.",
    "Meet You": "Meet You is a health tracking app. It provides menstrual cycle tracking, fertility prediction, and women's health management features.",
    "MyFitApp": "MyFitApp is a fitness & sport app by Innovatise. It provides gym information, class schedules, bookings, and facility updates.",
    "NBA": "NBA is a fitness & sport app by the National Basketball Association. It provides live games, scores, highlights, and basketball news.",
    "OW Health": "OW Health is the developer behind the 'Flo' period and ovulation tracker, focusing on data-driven women's health and reproductive wellness.",
    "Premier League": "Premier League is a fitness & sport app by the Premier League. It provides fixtures, scores, fantasy features, and football news.",
    "Pulselive": "Pulselive is a digital sports technology provider (part of Sony) that develops official apps and digital platforms for major sports organizations like the Premier League and ICC.",
    "Sky Sports": "Sky Sports is a fitness & sport app by Sky. It provides sports news, live scores, highlights, and video content.",
    "SofaScore": "SofaScore is a fitness & sport app by SofaScore. It provides live scores, player ratings, and match statistics across many sports.",
    "Sportradar": "Sportradar is a sports technology and data company that provides real-time sports scores, statistics, and integrity services to media companies and bookmakers.",
    "Sweatcoin": "Sweatcoin is a fitness & sport app by Sweatco Ltd. It converts walking activity into in-app rewards and wellness incentives.",
    "UEFA": "UEFA is a fitness & sport app by UEFA. It provides match schedules, scores, and content for UEFA competitions.",

    # Health Wearables
    "Fitbit": "Fitbit is a health wearables app by Google. It syncs with Fitbit devices for activity, sleep, heart rate, and health monitoring.",
    "Garmin": "Garmin is a health wearables app by Garmin. It syncs with Garmin devices for GPS tracking, fitness metrics, and health monitoring.",
    "Huawei Health": "Huawei Health is a health wearables app by Huawei. It syncs with Huawei wearables for activity, sleep, heart rate, and wellness tracking.",
    "Mi Watch": "Mi Watch is a health wearables app by Xiaomi. It syncs with Xiaomi smartwatches to monitor heart rate, sleep, workouts, and other fitness metrics.",
    "Google Pixel Watch": "Google Pixel Watch is a health wearables app by Google. It sets up and manages Pixel Watch devices, including fitness, notifications, and watch settings.",
    "Allview Watch": "Allview Watch is a health wearables app by Allview. It manages Allview smart devices, tracking daily steps, sleep patterns, and health notifications.",
    "Doogee Watch": "Doogee Watch is a health wearables app by Doogee. It connects to Doogee wearables to record sports data, heart rate, and sleep quality.",
    "HeyTap Watch": "HeyTap Watch is a health wearables app by OPPO. It is used to manage and sync data from OPPO watches, focusing on fitness tracking and health monitoring.",
    "Huawei Vassistant": "Huawei Vassistant is a health wearables support tool by Huawei. It serves as a voice assistant and utility interface for Huawei smart devices and health services.",
    "Huawei Watch": "Huawei Watch refers to the wearable device ecosystem managed via the Huawei Health app, tracking comprehensive fitness, SPO2, and cardiovascular metrics.",
    "Infinix Watch": "Infinix Watch is a health wearables app by Infinix. It tracks exercise data and physiological health indicators for users of Infinix wearable devices.",
    "Lenovo Watch": "Lenovo Watch is a health wearables app by Lenovo. It connects to Lenovo smartwatches to monitor activity, sleep, and heart rate.",
    "TCL Watch": "TCL Watch is a health wearables app by TCL. It provides device management and health data visualization for TCL’s range of smart wearables.",
    "Wiko Watch": "Wiko Watch is a health wearables app by Wiko. It tracks physical activity and health stats for users of Wiko branded smart bands and watches.",

    # Maternal & Family
    "The Asian Parent": "The Asian Parent is a maternal & family app by theAsianparent. It provides pregnancy tracking, parenting content, and family-focused community features.",
    "BabyCentre": "BabyCentre is a maternal & family app by BabyCenter. It provides pregnancy tracking, baby development guidance, and parenting resources.",
    "BabyTree": "BabyTree is a maternal & family app by Babytree Group. It is one of China’s largest parenting platforms, offering pregnancy tracking, early childhood education, and community support.",
    "Belajar Parenting": "Belajar Parenting is a maternal & family app. It provides parenting education and psychological well-being content for parents.",
    "Diary Bunda": "Diary Bunda is a maternal & family app by Alodokter Group. It provides pregnancy tracking and maternal and child health guidance for Indonesian mothers.",
    "Find My Kids": "Find My Kids is a maternal & family app by GEO TRACK TECHNOLOGIES INC. It provides GPS child location tracking, alerts, and parental control features.",
    "Health Parenting": "Health Parenting is a maternal & family app that provides resources for child health management, nutritional guidance, and developmental milestones tracking.",
    "Life360": "Life360 is a maternal & family app by Life360. It provides family location sharing, driving safety features, and emergency alerts.",
    "MamaLyfe": "MamaLyfe is a maternal & family app by PT Mamalyfe Selalu Diberkati. It provides parenting content, family discussions, and child development information.",
    "Parent Childcare": "Parent Childcare is a maternal & family app by Parent ApS. It provides childcare management, parent communication, scheduling, billing, and child development updates.",
    "Parentune": "Parentune is a maternal & family app by Parentune. It provides parenting advice, expert consultations, and community support for child health, growth, and development.",
    "Qustodio": "Qustodio is a maternal & family app by Qustodio. It provides parental controls including screen time management, content filtering, and location tracking.",
    "Tentang Anak": "Tentang Anak is a maternal & family app by Tentang Anak. It provides pregnancy support, child development tracking, and parenting education from trusted experts.",
}

print(f"DESC_HEALTH: {len(DESC_HEALTH)} entries")

DESC_HEALTH: 74 entries


In [126]:
# =============================================================================
# DESCRIPTIONS DICTIONARY — Information & Education
# =============================================================================
DESC_INFO_EDU = {
    # News & Media
    "Detik News": "Detik News is a news & media app by detikcom (Trans Digital Media). It provides Indonesian breaking news, features, and lifestyle content across multiple categories.",
    "Kompas": "Kompas is a news & media app by Kompas Gramedia. It provides Indonesian and international news coverage, analysis, and feature journalism.",
    "Kumparan": "Kumparan is a news & media app by PT Kumparan Harapan Mulia. It provides Indonesian news, community content, and personalized topic feeds.",
    "Liputan6": "Liputan6 is a news & media app by PT Kreatif Media Karya (Emtek Group). It provides Indonesian news, video, and live coverage across multiple topics.",
    "Tribunnews": "Tribunnews is a news & media app by Tribun Network, part of Kompas Gramedia. It provides Indonesian national and regional news coverage.",
    "CNN": "CNN is a news & media app by CNN. It provides breaking news, live video, and analysis on world events.",
    "BBC": "BBC is a news & media app by the British Broadcasting Corporation. It provides global news, analysis, and multimedia coverage.",
    "Google News": "Google News is a news & media app by Google. It provides aggregated and personalized news coverage from multiple publishers.",
    "Al Jazeera": "Al Jazeera is a news & media app by Al Jazeera Media Network. It provides international news and analysis with strong coverage of the Middle East and global affairs.",
    "Bloomberg": "Bloomberg is a news & media app by Bloomberg. It provides business, finance, markets, and global economic news.",
    "CNBC": "CNBC is a news & media app by CNBC. It provides business news, market updates, and financial analysis.",
    "CNET": "CNET is a news & media app by CNET. It provides technology news, product reviews, and buying advice.",
    "ESPN": "ESPN is a news & media app by ESPN. It provides sports news, scores, highlights, and live event coverage.",
    "Forbes": "Forbes is a news & media app by Forbes Media. It provides business, investing, leadership, and entrepreneurship coverage.",
    "The New York Times": "The New York Times is a news & media app by The New York Times Company. It provides news, analysis, opinion, and culture coverage.",
    "The Wall Street Journal": "The Wall Street Journal is a news & media app by Dow Jones. It provides business, financial, and general news coverage.",
    "Financial Times": "Financial Times is a news & media app by the Financial Times. It provides global business, markets, and economic journalism.",
    "ABC News": "ABC News is a news & media app by ABC News. It provides US and international news coverage, live reporting, and video.",
    "AP News": "AP News is a news & media app by The Associated Press. It provides breaking news and reporting across world, politics, business, and sports.",
    "Fox News": "Fox News is a news & media app by Fox News Media. It provides US and international news, opinion, and live programming.",
    "Channel NewsAsia": "Channel NewsAsia is a news & media app by CNA, part of Mediacorp. It provides Asian and international news coverage.",
    "Dream.co.id": "Dream.co.id is a news & media app by PT Dream Sentosa Indonesia. It provides Indonesian lifestyle, entertainment, and news content.",
    "Female Daily": "Female Daily is a news & media app by Female Daily Network. It provides beauty product reviews, recommendations, trends, and community content in Indonesia.",
    "Femina": "Femina is a news & media app by Femina Group. It provides women's lifestyle, fashion, relationship, and health content in Indonesia.",
    "Flipboard": "Flipboard is a news & media app by Flipboard. It provides personalized news and magazine-style content curation from multiple publishers.",
    "IGN": "IGN is a news & media app by IGN Entertainment. It provides video game, movie, TV, and tech news and reviews.",
    "India Times": "India Times is a news & media app by Times Internet. It provides Indian news, entertainment, and lifestyle content.",
    "La Repubblica": "La Repubblica is a news & media app by GEDI Gruppo Editoriale. It provides Italian and international news coverage.",
    "Mail Online": "Mail Online is a news & media app by dmg media. It provides news, entertainment, celebrity, and lifestyle coverage.",
    "Manchester United": "Manchester United is a news & media app by Manchester United Football Club. It provides official club news, match updates, and fan content.",
    "Opera News": "Opera News is a news & media app by Opera. It provides personalized news aggregation and local content feeds.",
    "Scooper News": "Scooper News is a news & media app by Transsnet. It provides personalized news and entertainment content with a regional focus.",
    "Sina News": "Sina News is a news & media app by Sina. It provides Chinese-language news and media coverage.",
    "Soompi": "Soompi is a news & media app by Soompi, part of Rakuten Viki. It provides K-pop, K-drama, and Korean entertainment news.",
    "SportFeeds": "SportFeeds is a news & media app that aggregates real-time sports headlines, scores, and breaking news from various global sports outlets.",
    "Tencent News": "Tencent News is a news & media app by Tencent. It provides Chinese-language news, video, and topic feeds within the Tencent ecosystem.",
    "The Economist Espresso": "The Economist Espresso is a news & media app by The Economist. It provides concise daily briefings on world affairs, business, and economics.",
    "The Independent": "The Independent is a news & media app by The Independent. It provides UK and international news, opinion, and feature journalism.",
    "AllFootball": "AllFootball is a news & media app by All Football Inc. It provides football news, scores, stats, highlights, and fan community features.",
    "Baidu News": "Baidu News is a news & media app by Baidu. It provides Chinese-language news aggregation from multiple sources.",
    "CBC News": "CBC News is a news & media app by the Canadian Broadcasting Corporation. It provides Canadian and international news coverage.",
    "OneFootball": "OneFootball is a news & media app by OneFootball GmbH. It provides football news, live scores, highlights, and transfer updates.",
    "TopBuzz": "TopBuzz was a news & media app by ByteDance. It provided AI-curated news and viral content feeds before being discontinued in many markets.",
    "VOA News": "VOA News is a news & media app by Voice of America. It provides international news and multimedia reporting in multiple languages.",

    # General Education
    "Ruangguru": "Ruangguru is a general education app by PT Ruang Raya Indonesia. It provides online tutoring, video lessons, and test preparation for Indonesian students.",
    "Zenius": "Zenius is a general education app by Zenius Education. It provides recorded lessons, exercises, and learning support for Indonesian students.",
    "Duolingo": "Duolingo is a general education app by Duolingo. It provides gamified language learning across many languages.",
    "Coursera": "Coursera is a general education app by Coursera. It provides online courses, certificates, and degree-related learning from universities and companies.",
    "Brainly": "Brainly is a general education app by Brainly. It provides student Q&A and homework help through a community-based platform.",
    "Google Classroom": "Google Classroom is a general education app by Google. It provides assignment management, classroom communication, and workflow tools for schools.",
    "Cakap": "Cakap is a general education app by PT Cakap Digital Teknologi. It provides live online classes for languages and other upskilling subjects.",
    "Kelas Pintar": "Kelas Pintar is a general education app by PT Extramarks Education Indonesia. It provides K-12 digital learning content and school support tools.",
    "Sekolahmu": "Sekolahmu is a general education app by PT Sekolah.mu Indonesia Cipta. It provides online learning programs and educational pathways for Indonesian learners.",
    "Rumah Belajar": "Rumah Belajar is a general education app by the Indonesian Ministry of Education. It provides free digital learning resources for Indonesian students and teachers.",
    "ELSA Speak": "ELSA Speak is a general education app by ELSA. It provides AI-powered English pronunciation and speaking practice.",
    "Photomath": "Photomath is a general education app by Photomath, part of Google. It solves math problems with step-by-step explanations.",
    "TED": "TED is a general education app by TED Conferences. It provides talks and educational videos from experts across many fields.",
    "Academia": "Academia is a general education app by Academia.edu. It provides access to academic papers and researcher profiles.",
    "Apple Books": "Apple Books is a general education app by Apple. It provides ebook and audiobook reading and purchasing for Apple users.",
    "Kindle": "Kindle is a general education app by Amazon. It provides ebook reading, syncing, and access to Amazon's digital book store.",
    "Gramedia Digital": "Gramedia Digital is a general education app by Kompas Gramedia. It provides digital books, magazines, and newspapers for Indonesian readers.",
    "Yousician": "Yousician is a general education app by Yousician. It provides interactive music learning for instruments and singing.",
    "Lingokids": "Lingokids is a general education app by Lingokids. It provides educational games and English learning activities for children.",
    "Cake Learn English": "Cake Learn English is a general education app by Cake. It provides bite-sized English learning lessons using real-life video clips.",
    "GauthMath": "GauthMath is a general education app, now branded as Gauth. It provides AI-powered homework help and step-by-step explanations across multiple subjects.",
    "Mathpresso": "Mathpresso is a general education app by Mathpresso. It provides study tools including QANDA for photo-based problem solving and tutoring support.",
    "WordBit English": "WordBit English is a general education app by WordBit. It provides vocabulary learning through lock-screen flashcard-style lessons.",
    "EJOY": "EJOY is a general education app by eJOY. It provides English learning through videos, subtitles, and vocabulary tools.",
    "BabyBus": "BabyBus is a general education app by BabyBus. It provides educational games, songs, and early learning content for children.",
    "Abjad": "Abjad is a general education app providing interactive lessons and tools for learning the Arabic script and basic language skills.",
    "ATI": "ATI is a general education app by Assessment Technologies Institute. It provides study materials, practice exams, and nursing education resources.",
    "Ayo Belajar": "Ayo Belajar is a general education app designed for Indonesian students, providing K-12 curriculum-based learning materials.",
    "CNKI": "CNKI is a general education app by China National Knowledge Infrastructure. It provides access to Chinese academic literature and research databases.",
    "CUHK": "CUHK is a general education app by The Chinese University of Hong Kong. It provides campus services and academic information for university users.",
    "Cambridge": "Cambridge is a general education app by Cambridge University Press & Assessment. It provides English learning materials, dictionaries, and exam preparation resources.",
    "DUMI": "DUMI is a general education app that provides language learning and vocabulary building tools through gamified lessons.",
    "Harvard": "Harvard is a general education app by Harvard University. It provides campus news, academic resources, and mobile access to university services.",
    "KUPU": "KUPU is a general education and career platform app. It provides skill-based learning, professional courses, and job matching services.",
    "Merriam-Webster": "Merriam-Webster is a general education app by Merriam-Webster. It provides dictionary definitions, thesaurus features, and vocabulary tools.",
    "Moodle": "Moodle is a general education app by Moodle. It provides access to course materials, assignments, and learning management systems.",
    "Springer": "Springer is a general education app by Springer Nature. It provides access to academic books, journals, and research content.",
    "UCAN": "UCAN is a general education app providing test preparation and assessment tools for competitive exams and academic certifications.",

    # Campus & LMS
    "BINUS University": "BINUS University is a campus & LMS app by Bina Nusantara University. It provides student academic services, schedules, and campus information.",
    "UIN Maulana Malik Ibrahim Malang": "UIN Maulana Malik Ibrahim Malang is a campus & LMS app by UIN Maulana Malik Ibrahim Malang. It provides academic services and campus information for students.",
    "Universitas Brawijaya": "Universitas Brawijaya is a campus & LMS app by Universitas Brawijaya. It provides academic information and student service access.",
    "Universitas Diponegoro": "Universitas Diponegoro is a campus & LMS app by Universitas Diponegoro. It provides student academic and campus service features.",
    "Universitas Gadjah Mada": "Universitas Gadjah Mada is a campus & LMS app by Universitas Gadjah Mada. It provides student academic information and campus services.",
    "Universitas Gunadarma": "Universitas Gunadarma is a campus & LMS app by Universitas Gunadarma. It provides academic services and student portal access.",
    "Universitas Indonesia": "Universitas Indonesia is a campus & LMS app by Universitas Indonesia. It provides student academic information and campus services.",
    "Universitas Islam Indonesia": "Universitas Islam Indonesia is a campus & LMS app by Universitas Islam Indonesia. It provides academic and student service information.",
    "Universitas Negeri Semarang": "Universitas Negeri Semarang is a campus & LMS app by Universitas Negeri Semarang. It provides student academic services and campus information.",
    "Universitas Negeri Yogyakarta": "Universitas Negeri Yogyakarta is a campus & LMS app by Universitas Negeri Yogyakarta. It provides student academic services and university information.",
    "Universitas Padjadjaran": "Universitas Padjadjaran is a campus & LMS app by Universitas Padjadjaran. It provides student academic portal access and campus information.",
    "Universitas Pendidikan Indonesia": "Universitas Pendidikan Indonesia is a campus & LMS app by Universitas Pendidikan Indonesia. It provides academic services and student portal features.",
    "Universitas Sebelas Maret": "Universitas Sebelas Maret is a campus & LMS app by Universitas Sebelas Maret. It provides student academic information and campus services.",
    "Universitas Terbuka": "Universitas Terbuka is a campus & LMS app by Universitas Terbuka. It provides distance learning access and student academic services.",
    "Universitas Udayana": "Universitas Udayana is a campus & LMS app by Universitas Udayana. It provides student academic services and campus information.",

    # Religious
    "Muslim Pro": "Muslim Pro is a religious app by Bitsmedia. It provides prayer times, Quran reading, Qibla direction, and Islamic calendar tools.",
    "Al Quran Indonesia": "Al Quran Indonesia is a religious app. It provides Quran reading with Indonesian translation and audio recitation.",
    "Bible": "Bible is a religious app by YouVersion. It provides Bible reading plans, audio, devotionals, and multiple translations.",
    "Islam21c": "Islam21c is a religious app by Islam21c. It provides Islamic articles, commentary, and current affairs content.",
    "IslamHouse": "IslamHouse is a religious app by IslamHouse. It provides Islamic educational materials in multiple languages.",
    "IslamWay": "IslamWay is a religious app by IslamWay. It provides Islamic lectures, Quran recitations, and religious audio content.",
    "IslamWeb": "IslamWeb is a religious app by IslamWeb. It provides fatwas, Islamic articles, and educational resources.",
    "Nabawi": "Nabawi is a religious app providing Islamic content, prayer times, and educational resources related to the Sunnah.",
    "Nusuk": "Nusuk is a religious app by the Saudi Ministry of Hajj and Umrah. It provides pilgrimage planning, permit-related services, and travel support for Umrah and Hajj.",
    "PDF Quran": "PDF Quran is a religious app providing digital, high-quality scans of the Holy Quran for offline reading and recitation.",
    "Saqina": "Saqina is a religious app focused on spiritual well-being, providing Islamic mindfulness content and Quranic recitations.",
    "TV Quran": "TV Quran is a religious app. It provides Quran recitation audio and related Islamic listening content.",
    "WeMuslim": "WeMuslim is a religious app. It provides prayer times, Quran tools, and Muslim lifestyle features.",

    # Cooking & Recipes
    "Dapur Umami": "Dapur Umami is a cooking & recipes app by Ajinomoto Indonesia. It provides Indonesian recipes, cooking inspiration, and food tips.",
    "Douguo Food": "Douguo Food is a cooking & recipes app by Douguo. It provides recipe sharing, meal ideas, and cooking community features.",
    "Food Network": "Food Network is a cooking & recipes app by Food Network. It provides recipes, cooking videos, and food inspiration.",
    "Masak Apa Ya": "Masak Apa Ya is a cooking & recipes app. It provides Indonesian recipe ideas and meal inspiration.",

    # Reference & Wiki
    "Wikipedia": "Wikipedia is a reference & wiki app by the Wikimedia Foundation. It provides free encyclopedia articles across many languages.",
    "wikiHow": "wikiHow is a reference & wiki app by wikiHow. It provides step-by-step how-to guides across a wide range of topics.",
}

print(f"DESC_INFO_EDU: {len(DESC_INFO_EDU)} entries")

DESC_INFO_EDU: 116 entries


In [127]:
# =============================================================================
# DESCRIPTIONS DICTIONARY — Lifestyle
# =============================================================================
DESC_LIFESTYLE = {
    # Food Delivery
    "Ajinomoto": "Ajinomoto is a food delivery app by PT Ajinomoto Indonesia. It provides online ordering for Ajinomoto seasoning and food products through its shopping app ecosystem.",
    "Bango": "Bango is a lifestyle app by Unilever Indonesia focusing on culinary heritage, recipes, and a directory of local food stalls featuring Bango products.",
    "Bolt Food": "Bolt Food is a food delivery app by Bolt Technology. It provides on-demand restaurant food delivery and related local commerce services.",
    "Ele.me": "Ele.me is a food delivery app by Alibaba Group. It provides on-demand food delivery and local lifestyle services in China.",
    "Foodpanda": "Foodpanda is a food delivery app by Delivery Hero. It provides food and grocery delivery across Asia and other markets.",
    "Glovo": "Glovo is a food delivery app by Glovo. It provides on-demand delivery for food, groceries, and other local commerce items.",
    "Pho": "Pho is a dining & FnB app providing menu browsing and restaurant information for Vietnamese street food chains.",
    "Talabat": "Talabat is a food delivery app by Talabat, part of Delivery Hero. It provides restaurant food delivery and grocery services across the Middle East.",
    "Wolt": "Wolt is a food delivery app by Wolt, part of DoorDash. It provides food and retail delivery services in multiple markets.",

    # Dining & FnB
    "Akindo Sushiro": "Akindo Sushiro is a dining & FnB app by Akindo Sushiro. It provides restaurant information, reservations, and menu-related services for Sushiro outlets.",
    "AlBaik": "AlBaik is a dining & FnB app by AlBaik. It provides menu browsing, branch information, and ordering-related services for the AlBaik restaurant chain.",
    "Burger King": "Burger King is a dining & FnB app by Burger King. It provides mobile ordering, offers, loyalty features, and restaurant information.",
    "Choco Chips": "Choco Chips is a lifestyle app by Chocochips Boutique, offering a mobile shopping platform for women's fashion and apparel.",
    "Domino's Pizza": "Domino's Pizza is a dining & FnB app by Domino's. It provides pizza ordering, delivery tracking, offers, and store services.",
    "Doyan Kuliner": "Doyan Kuliner is a dining & FnB app serving as a discovery and recommendation platform for Indonesian local food and restaurants.",
    "Eatigo": "Eatigo is a dining & FnB app by Eatigo. It provides restaurant discovery, reservations, and time-based dining discounts.",
    "Fore Coffee": "Fore Coffee is a dining & FnB app by PT Fore Kopi Indonesia. It provides mobile ordering, pickup, delivery, and loyalty features for Fore Coffee customers.",
    "Garudafood": "Garudafood is a lifestyle app by PT Garudafood providing digital ordering and distribution services for its snack and beverage products.",
    "Hangry": "Hangry is a dining & FnB app by Hangry. It provides ordering and brand access for Hangry's multi-brand food and beverage business.",
    "KFC": "KFC is a dining & FnB app by KFC. It provides food ordering, delivery, promotions, and restaurant services.",
    "Kopi Kenangan": "Kopi Kenangan is a dining & FnB app by Kenangan Brands. It provides mobile ordering, loyalty rewards, and store services for Kopi Kenangan customers.",
    "Luckin Coffee": "Luckin Coffee is a dining & FnB app by Luckin Coffee. It provides app-based coffee ordering, pickup, and delivery services.",
    "Maxx Coffee": "Maxx Coffee is a dining & FnB app by PT Maxx Coffee Prima. It provides coffee ordering and brand-related customer services.",
    "McDonald's": "McDonald's is a dining & FnB app by McDonald's. It provides ordering, McDelivery access, deals, rewards, and restaurant information.",
    "Ongkowidjojo": "Ongkowidjojo is a lifestyle and enterprise app used for distribution management and business services within the Ongkowidjojo group.",
    "Otten Coffee": "Otten Coffee is a dining & FnB app by PT Otten Coffee Indonesia. It provides specialty coffee bean, machine, and brewing equipment shopping.",
    "Pepsi": "Pepsi is a lifestyle app by PepsiCo providing brand engagement, promotional offers, and loyalty rewards for consumers.",
    "PergiKuliner": "PergiKuliner is a dining & FnB app by PergiKuliner. It provides restaurant discovery, reviews, ratings, and dining recommendations in Indonesia.",
    "Pizza Hut": "Pizza Hut is a dining & FnB app by Pizza Hut. It provides food ordering, delivery, promotions, and restaurant services.",
    "Richeese Factory": "Richeese Factory is a dining & FnB app by PT Richeese Kuliner Indonesia. It provides food ordering and related customer services for Richeese Factory outlets.",
    "Starbucks": "Starbucks is a dining & FnB app by Starbucks. It provides ordering, payment, loyalty rewards, and store locator services.",
    "Waiterio": "Waiterio is a dining & FnB app by Waiterio. It provides POS, ordering, and restaurant management tools for food businesses.",
    "Zomato": "Zomato is a dining & FnB app by Zomato. It provides restaurant discovery, menus, reviews, and dining-related services.",

    # Automotive Owner Service
    "AUTO2000": "AUTO2000 is an automotive owner service app by AUTO2000. It provides Toyota service booking, maintenance information, and owner support services.",
    "AUTOBILD": "AUTOBILD is an automotive owner service app by AUTO BILD. It provides automotive news, reviews, and car-related editorial content.",
    "Astra Motor": "Astra Motor is an automotive owner service app by Astra Motor. It provides Honda motorcycle service booking, owner information, and related services.",
    "Audi": "Audi is an automotive owner service app by Audi. It provides connected vehicle services, vehicle management, and owner support features.",
    "AutoHome": "AutoHome is an automotive owner service app by Autohome. It provides automotive news, reviews, pricing, and dealer information in China.",
    "Automagz": "Automagz is an automotive owner service app. It provides automotive news, reviews, and enthusiast content.",
    "BMW": "BMW is an automotive owner service app by BMW. It provides connected-car controls, vehicle information, and owner service features.",
    "BYD": "BYD is an automotive owner service app by BYD. It provides EV owner services, vehicle management, and connected-car features.",
    "DAYA AUTO": "DAYA AUTO is an automotive owner service app by PT Daya Adicipta Motora for Honda motorcycle service booking and parts in Indonesia.",
    "Daihatsu": "Daihatsu is an automotive owner service app by Astra Daihatsu Motor. It provides service booking, maintenance information, and owner support.",
    "Ford": "Ford is an automotive owner service app by Ford. It provides connected vehicle controls, service scheduling, and owner support features.",
    "Honda": "Honda is an automotive owner service app by Honda. It provides vehicle information, maintenance scheduling, and owner-related services.",
    "Mazda": "Mazda is an automotive owner service app by Mazda. It provides connected-car and owner service features, including maintenance-related support.",
    "Mercedes-Benz": "Mercedes-Benz is an automotive owner service app by Mercedes-Benz. It provides connected vehicle controls, remote features, and service support.",
    "Mitsubishi": "Mitsubishi is an automotive owner service app by Mitsubishi Motors. It provides owner information, maintenance support, and service-related features.",
    "Modifikasi": "Modifikasi is an automotive owner service app providing a community platform and news for vehicle modification enthusiasts.",
    "Montir ID": "Montir ID is an automotive owner service app by Montir.id. It provides mechanic booking and vehicle service assistance.",
    "Motorplus": "Motorplus is an automotive owner service app by Motorplus. It provides motorcycle news, reviews, and modification-related content.",
    "OtoDetik": "OtoDetik is an automotive owner service app by detikcom. It provides automotive news, reviews, and vehicle-related editorial content.",
    "Otomotif Kompas": "Otomotif Kompas is an automotive owner service app by Kompas. It provides automotive news, reviews, and car or motorcycle content.",
    "Otomotif Tempo": "Otomotif Tempo is an automotive owner service app by Tempo. It provides automotive news and review content.",
    "Otosia": "Otosia is an automotive owner service app by Otosia. It provides automotive news, reviews, and vehicle-related content.",
    "Porsche": "Porsche is an automotive owner service app by Porsche. It provides connected-car features, charging support, and owner services.",
    "Skoda": "Skoda is an automotive owner service app by Skoda Auto. It provides connected-car services and owner support features.",
    "StarCharge": "StarCharge is an automotive owner service app by Star Charge. It provides EV charging-related services such as station access and charging management.",
    "Tesla": "Tesla is an automotive owner service app by Tesla. It provides remote vehicle controls, charging management, and service support.",
    "Toyota": "Toyota is an automotive owner service app by Toyota. It provides owner information, service booking, and connected-car features.",
    "Volkswagen": "Volkswagen is an automotive owner service app by Volkswagen. It provides connected vehicle services, maintenance support, and owner features.",
    "Wuling": "Wuling is an automotive owner service app by Wuling. It provides owner support, vehicle information, and service-related features.",
    "Yamaha": "Yamaha is an automotive owner service app by Yamaha Motor. It provides motorcycle-related owner services, maintenance information, and brand support.",
    "iOtomotif": "iOtomotif is an automotive owner service app providing digital news, reviews, and market price information for vehicles.",

    # Beauty & Personal Care
    "Beautynesia": "Beautynesia is a beauty & personal care app by Beautynesia. It provides beauty news, product reviews, tutorials, and skincare content.",
    "Hot Pepper Beauty": "Hot Pepper Beauty is a beauty & personal care app by Recruit. It provides salon, spa, and beauty reservation services in Japan.",
    "Perfect Corp": "Perfect Corp is a beauty & personal care app by Perfect Corp. It provides AI-powered virtual try-on and beauty-tech experiences for makeup and skincare.",
    "Sephora": "Sephora is a beauty & personal care app by Sephora. It provides beauty product shopping, recommendations, and loyalty features.",
    "Watsons": "Watsons is a beauty & personal care app by A.S. Watson Group. It provides health and beauty shopping, promotions, and loyalty services.",

    # Job & Freelance
    "Fastwork": "Fastwork is a job & freelance app by Fastwork. It connects clients with freelancers for digital and professional services.",
    "Fiverr": "Fiverr is a job & freelance app by Fiverr. It provides a marketplace for freelance services across creative, technical, and business categories.",
    "Glassdoor": "Glassdoor is a job & freelance app by Glassdoor. It provides job listings, company reviews, salary insights, and workplace information.",
    "Glints": "Glints is a job & freelance app by Glints. It provides job search, hiring, and career development services for Southeast Asia.",
    "HeadHunter Russia": "HeadHunter Russia is a job & freelance app by HeadHunter. It provides job search and recruitment services in the Russian-speaking market.",
    "Indeed": "Indeed is a job & freelance app by Indeed. It provides job search and listing aggregation across many industries and locations.",
    "JobStreet": "JobStreet is a job & freelance app by JobStreet, part of SEEK. It provides job search and recruitment services in Southeast Asia.",
    "JobsDB": "JobsDB is a job & freelance app by JobsDB, part of SEEK. It provides job search and career services in Hong Kong and other Asian markets.",
    "KitaLulus": "KitaLulus is a job & freelance app by PT Kita Lulus Indonesia. It provides job search and career support, especially for entry-level workers in Indonesia.",
    "Seek": "Seek is a job & freelance app by SEEK. It provides job search and recruitment services across Asia-Pacific markets.",
    "Upwork": "Upwork is a job & freelance app by Upwork. It provides a global freelancing marketplace for remote work and project-based hiring.",

    # Smart Home
    "SmartThings": "SmartThings is a smart home app by Samsung. It provides centralized control, automation, and monitoring for compatible smart home devices.",
}

print(f"DESC_LIFESTYLE: {len(DESC_LIFESTYLE)} entries")

DESC_LIFESTYLE: 81 entries


In [128]:
# =============================================================================
# DESCRIPTIONS DICTIONARY — Transportation
# =============================================================================
DESC_TRANSPORTATION = {
    # Travel Booking
    "Traveloka": "Traveloka is a travel booking app by PT Traveloka Indonesia. It provides flights, hotels, trains, attractions, airport transfers, and other travel bookings across Southeast Asia.",
    "Tiket.com": "Tiket.com is a travel booking app by PT Global Tiket Network. It provides flights, hotels, trains, car rentals, and event ticket bookings in Indonesia.",
    "Agoda": "Agoda is a travel booking app by Agoda, part of Booking Holdings. It offers hotel, accommodation, and travel bookings across global destinations.",
    "Booking.com": "Booking.com is a travel booking app by Booking.com, part of Booking Holdings. It provides hotel, apartment, and vacation rental booking worldwide.",
    "Expedia": "Expedia is a travel booking app by Expedia Group. It provides flights, hotels, car rentals, and vacation package booking globally.",
    "Airbnb": "Airbnb is a travel booking app by Airbnb. It connects travelers with stays, homes, and local experiences hosted by individuals and businesses.",
    "Skyscanner": "Skyscanner is a travel booking app by Skyscanner, part of Trip.com Group. It provides flight, hotel, and car rental search and price comparison.",
    "Trivago": "Trivago is a travel booking app by trivago. It compares hotel prices across multiple booking websites.",
    "Klook": "Klook is a travel booking app by Klook Travel Technology. It provides attractions, tours, transport passes, and travel experience booking.",
    "Pegipegi": "Pegipegi was a travel booking app by PT Go Online Destinations. It provided hotel, flight, and train booking services in Indonesia before ceasing operations.",
    "Mister Aladin": "Mister Aladin is a travel booking app by PT Mister Mobile Indonesia. It provides hotel, flight, train, and travel package booking for Indonesian travelers.",
    "Nusatrip": "Nusatrip is a travel booking app by PT Nusatrip Internasional. It provides flight and hotel booking services focused on Indonesian and regional travelers.",
    "Travelio": "Travelio is a travel booking app by PT Travelio Group. It provides apartment, house, and short- or long-stay property rental booking in Indonesia.",
    "Ctrip": "Ctrip is a travel booking app by Trip.com Group. It provides flights, hotels, trains, tours, and other travel bookings.",
    "Kayak": "Kayak is a travel booking app by KAYAK, part of Booking Holdings. It provides travel search and price comparison for flights, hotels, and car rentals.",
    "Hotels.com": "Hotels.com is a travel booking app by Expedia Group. It provides hotel booking and loyalty-related accommodation services.",
    "OYO Rooms": "OYO Rooms is a travel booking app by OYO. It provides budget and standardized hotel booking across multiple markets.",
    "Bobobox": "Bobobox is a travel booking app by PT Bobobox Mitra Indonesia. It provides smart capsule hotel and compact-stay booking in Indonesia.",
    "Airy Rooms": "Airy Rooms was a travel booking app by PT Airy Rooms Indonesia. It provided budget accommodation booking with standardized amenities in Indonesia.",
    "HotelMurah": "HotelMurah is a travel booking and digital payment app by PT Hotel Murah Travelindo. It provides hotel, flight, and train bookings, alongside digital top-ups and bill payment services in Indonesia.",
    "Hilton": "Hilton is a travel booking app by Hilton. It provides hotel booking, digital check-in, digital key, and Hilton Honors account services.",
    "Hyatt": "Hyatt is a travel booking app by Hyatt. It provides hotel booking, loyalty services, and digital room access features.",
    "AirAsia": "AirAsia is a travel booking app by Capital A. It provides flight booking and other travel-related services including hotels and transport.",
    "Garuda Indonesia": "Garuda Indonesia is a travel booking app by PT Garuda Indonesia. It provides flight booking, check-in, and GarudaMiles-related services.",
    "Sriwijaya Air": "Sriwijaya Air is a travel booking app by PT Sriwijaya Air. It provides flight booking and airline-related passenger services.",
    "Air France": "Air France is a travel booking app by Air France. It provides flight booking, check-in, boarding passes, and Flying Blue services.",
    "British Airways": "British Airways is a travel booking app by British Airways. It provides flight booking, check-in, and Executive Club account services.",
    "Cathay Pacific": "Cathay Pacific is a travel booking app by Cathay Pacific. It provides flight booking, check-in, and loyalty-related travel services.",
    "Emirates": "Emirates is a travel booking app by Emirates. It provides flight booking, check-in, and Emirates Skywards account services.",
    "Lufthansa": "Lufthansa is a travel booking app by Lufthansa Group. It provides flight booking, check-in, and Miles & More-related travel services.",
    "Qatar Airways": "Qatar Airways is a travel booking app by Qatar Airways. It provides flight booking, check-in, and Privilege Club services.",
    "Turkish Airlines": "Turkish Airlines is a travel booking app by Turkish Airlines. It provides flight booking, check-in, and Miles&Smiles account services.",
    "Ryanair": "Ryanair is a travel booking app by Ryanair. It provides low-cost flight booking, check-in, and mobile boarding pass features.",
    "EasyJet": "EasyJet is a travel booking app by easyJet. It provides budget flight booking, check-in, and trip management for European routes.",
    "Malaysia Airlines": "Malaysia Airlines is a travel booking app by Malaysia Airlines. It provides flight booking, check-in, and Enrich loyalty services.",
    "Air Arabia": "Air Arabia is a travel booking app by Air Arabia. It provides low-cost airline booking and passenger services across regional routes.",
    "Air Berlin": "Air Berlin was a travel booking app by Air Berlin. It provided flight booking and airline passenger services before the airline ceased operations.",
    "Avianca": "Avianca is a travel booking app by Avianca. It provides flight booking, check-in, and loyalty-related airline services.",
    "EgyptAir": "EgyptAir is a travel booking app by EgyptAir. It provides flight booking and passenger travel services for the airline.",
    "Flynas": "Flynas is a travel booking app by flynas. It provides low-cost flight booking and airline passenger services.",
    "HK Airlines": "HK Airlines is a travel booking app by Hong Kong Airlines. It provides flight booking and airline passenger services.",
    "Jet Airways": "Jet Airways was a travel booking app by Jet Airways. It provided flight booking and passenger services before the airline entered insolvency and ceased operations.",
    "Kuwait Airways": "Kuwait Airways is a travel booking app by Kuwait Airways. It provides flight booking and airline passenger services.",
    "Oman Air": "Oman Air is a travel booking app by Oman Air. It provides flight booking, check-in, and airline passenger services.",
    "Royal Jordanian": "Royal Jordanian is a travel booking app by Royal Jordanian. It provides flight booking and airline passenger services.",
    "flyadeal": "flyadeal is a travel booking app by flyadeal. It provides low-cost flight booking and passenger services.",
    "flydubai": "flydubai is a travel booking app by flydubai. It provides flight booking, check-in, and airline passenger services.",
    "MyICA Mobile": "MyICA Mobile is a government travel document and border-services app by Singapore's Immigration & Checkpoints Authority. It provides ICA e-services such as arrival card submission and immigration-related services.",
    "Bioskop Online": "Bioskop Online is an entertainment streaming app by PT Bioskop Online Indonesia. It provides transactional online movie viewing for Indonesian films rather than travel booking.",
    "CGV Cinemas": "CGV Cinemas is a cinema ticketing app by CGV. It provides movie schedules, seat selection, snack ordering, and ticket booking for CGV theaters.",
    "Cinema XXI": "Cinema XXI is a cinema ticketing app by PT Nusantara Sejahtera Raya. It provides movie schedules, seat booking, and ticket purchase for Cinema XXI theaters.",
    "Cineplex": "Cineplex is a cinema ticketing app by Cineplex Entertainment. It provides movie schedules, cinema ticket booking, and theater-related services.",
    "Cinepolis": "Cinepolis is a cinema ticketing app by Cinépolis. It provides movie schedules and ticket booking for Cinépolis cinemas.",
    "Cinestar": "Cinestar is a cinema ticketing app by CineStar. It provides movie schedules and ticket booking for CineStar cinemas.",
    "TIX ID": "TIX ID is an entertainment ticketing app by PT Nusantara Elang Sejahtera. It provides movie ticket booking and broader entertainment-related ticketing and promotions in Indonesia.",
    "BookMyShow": "BookMyShow is a ticketing app by BookMyShow. It provides movie, event, and live entertainment ticket booking.",
    "Tokyo Disney Resort": "Tokyo Disney Resort is a destination planning and ticketing app by Oriental Land Co. It provides park ticketing, attraction planning, and resort information for Tokyo Disney Resort.",
    "Eventim": "Eventim is an event ticketing app by CTS Eventim. It provides concert, festival, and event ticket booking.",
    "Reservix": "Reservix is an event ticketing app by Reservix. It provides concert, cultural, and live event ticket booking.",
    "Ticketmaster": "Ticketmaster is an event ticketing app by Ticketmaster, part of Live Nation Entertainment. It provides event, concert, and sports ticket booking worldwide.",

    # Ride Hailing
    "Gojek": "Gojek is a ride hailing app by GoTo Group. It provides ride-hailing, food delivery, payments, and other on-demand services.",
    "Grab": "Grab is a ride hailing app by Grab Holdings. It provides ride-hailing, delivery, payments, and other on-demand services across Southeast Asia.",
    "Uber": "Uber is a ride hailing app by Uber Technologies. It provides on-demand ride-hailing and related mobility services globally.",
    "Bluebird": "Bluebird is a ride hailing app by PT Blue Bird Tbk. It provides taxi booking and ride services in Indonesia.",
    "Maxim": "Maxim is a ride hailing app by Maxim. It provides ride-hailing, delivery, and related mobility services in multiple markets.",
    "inDrive": "inDrive is a ride hailing app by inDrive. It provides ride-hailing with negotiated fares between passengers and drivers.",
    "Bolt": "Bolt is a ride hailing app by Bolt Technology. It provides ride-hailing and other local mobility services.",
    "Anterin": "Anterin is a ride hailing app by PT Anterin Lintas Nusantara. It provides ride-hailing and delivery services in Indonesia.",
    "Careem": "Careem is a ride hailing app by Careem. It provides ride-hailing and related local services across the Middle East and nearby regions.",
    "DiDi": "DiDi is a ride hailing app by DiDi Global. It provides ride-hailing and mobility services in multiple international markets.",
    "Free Now": "Free Now is a ride hailing app by FREENOW. It provides taxi, ride-hailing, and mobility services in Europe.",
    "Taxsee": "Taxsee is a ride hailing app by Taxsee. It provides taxi dispatch and ride-hailing services.",

    # Ride Hailing Driver
    "Gojek Driver": "Gojek Driver is a ride hailing driver app by GoTo Group. It provides order, earnings, and service management for Gojek driver-partners.",
    "Bolt Driver": "Bolt Driver is a ride hailing driver app by Bolt Technology. It provides order and earnings management for Bolt driver-partners.",
    "Jeeny Driver": "Jeeny Driver is a ride hailing driver app by Jeeny. It provides ride order management for Jeeny driver-partners.",

    # Logistics & Delivery
    "AnterAja": "AnterAja is a logistics & delivery app by PT Tri Adi Bersama. It provides parcel delivery, courier booking, and shipment tracking in Indonesia.",
    "DHL": "DHL is a logistics & delivery app by DHL. It provides international shipping, parcel tracking, and logistics-related services.",
    "Deliveree": "Deliveree is a logistics & delivery app by Deliveree. It provides on-demand trucking, van, and logistics booking for businesses and individuals.",
    "Lalamove": "Lalamove is a logistics & delivery app by Lalamove. It provides on-demand same-day delivery and courier or vehicle booking.",
    "Paxel": "Paxel is a logistics & delivery app by PT Paxel Algorita Unggul. It provides same-day delivery and parcel shipping services in Indonesia.",
    "Titipku": "Titipku is a logistics & delivery app by PT Titipku Teknologi Indonesia. It provides market shopping, local commerce, and delivery services.",

    # Navigation & Maps
    "Google Maps": "Google Maps is a navigation & maps app by Google. It provides turn-by-turn navigation, live traffic, transit information, and place discovery.",
    "Waze": "Waze is a navigation & maps app by Google. It provides community-driven navigation with real-time traffic and road incident updates.",
    "Apple Maps": "Apple Maps is a navigation & maps app by Apple. It provides navigation, directions, and place discovery on Apple devices.",
    "HERE WeGo": "HERE WeGo is a navigation & maps app by HERE Technologies. It provides maps, route planning, and navigation with offline support.",
    "Naver Map": "Naver Map is a navigation & maps app by Naver. It provides mapping and navigation services focused on South Korea.",
    "Petal Maps": "Petal Maps is a navigation & maps app by Huawei. It provides navigation, mapping, and route planning services.",
    "AutoNavi (Amap)": "AutoNavi (Amap) is a navigation & maps app by AutoNavi. It provides mapping and navigation services in China.",
    "Bing Maps": "Bing Maps is a navigation & maps app by Microsoft. It provides mapping, directions, and location search services.",
    "Google Earth": "Google Earth is a navigation & maps app by Google. It provides satellite imagery, 3D terrain, and virtual globe exploration.",
    "OpenStreetMap": "OpenStreetMap is a navigation & maps platform by the OpenStreetMap Foundation. It provides free and collaboratively built map data.",
    "SoSo Map": "SoSo Map is a navigation & maps app by Tencent. It provides mapping and navigation services within the Tencent ecosystem.",

    # Public Transport & Ticketing
    "KAI Access": "KAI Access is a public transport & ticketing app by PT Kereta Api Indonesia. It provides train ticket booking, schedules, and passenger services for Indonesian railways.",
    "KRL Access": "KRL Access is a public transport & ticketing app by PT KAI Commuter. It provides commuter line schedules, station information, and passenger tools for the Jakarta metropolitan area.",
    "Angkasa Pura I": "Angkasa Pura I is a public transport & ticketing app by PT Angkasa Pura I. It provides airport information and service features for airports under Angkasa Pura I.",
    "Angkasa Pura II": "Angkasa Pura II is a public transport & ticketing app by PT Angkasa Pura II. It provides airport information and service features for airports under Angkasa Pura II.",
    "PELNI": "PELNI is a public transport & ticketing app by PT PELNI. It provides passenger ship and ferry ticket booking services in Indonesia.",
    "Rosalia Indah": "Rosalia Indah is a public transport & ticketing app by PO Rosalia Indah. It provides intercity bus ticket booking and passenger services.",
    "TractoGo": "TractoGo is a transport rental app by TRAC, part of Astra. It provides car rental, bus rental, and airport transfer booking services.",

    # Parking
    "Parkee": "Parkee is a parking app by PT Parkee Digital Indonesia. It provides digital parking payment and parking management services in Indonesia.",
}

print(f"DESC_TRANSPORTATION: {len(DESC_TRANSPORTATION)} entries")

DESC_TRANSPORTATION: 100 entries


In [129]:
# =============================================================================
# DESCRIPTIONS DICTIONARY — Productivity & Tools
# =============================================================================
DESC_PRODUCTIVITY = {
    # AI Assistant
    "ChatGPT": "ChatGPT is an AI assistant app by OpenAI. It provides conversational AI for question answering, writing, coding, and reasoning tasks.",
    "Google Gemini": "Google Gemini is an AI assistant app by Google. It provides multimodal AI assistance integrated with Google products and services.",
    "Microsoft Copilot": "Microsoft Copilot is an AI assistant app by Microsoft. It provides AI-powered assistance for writing, research, productivity, and coding tasks.",
    "Claude": "Claude is an AI assistant app by Anthropic. It provides conversational AI for writing, analysis, coding, and reasoning tasks.",
    "DeepSeek": "DeepSeek is an AI assistant app by DeepSeek. It provides large language model-based AI assistance focused on reasoning and coding.",
    "Perplexity": "Perplexity is an AI assistant app by Perplexity. It provides AI-powered search and answer generation with cited web results.",
    "Amazon Alexa": "Amazon Alexa is an AI assistant app by Amazon. It provides voice-based assistance for smart home control, information, music, and automation.",
    "Apple Siri": "Apple Siri is an AI assistant by Apple. It provides voice-based assistance for device control, queries, messaging, and smart actions across Apple devices.",
    "Character AI": "Character AI is an AI assistant app by Character.AI. It provides conversational AI characters for entertainment, companionship, and roleplay.",
    "DouBao": "DouBao is an AI assistant app by ByteDance. It provides a Chinese-language conversational AI assistant for general tasks.",
    "Grok": "Grok is an AI assistant app by xAI. It provides conversational AI with live information integration from the X ecosystem.",
    "KiMi Moonshot": "KiMi Moonshot is an AI assistant app by Moonshot AI. It provides a long-context AI assistant for chat, search, and document understanding.",
    "PixVerse": "PixVerse is an AI creation app by PixVerse. It provides AI video generation from text prompts and images.",
    "QuillBot": "QuillBot is an AI assistant app by QuillBot. It provides paraphrasing, grammar checking, summarization, and writing assistance.",
    "Qwen": "Qwen is an AI assistant app by Alibaba Cloud. It provides large language model-based AI assistance for multilingual and multimodal tasks.",
    "Talkie AI": "Talkie AI is an AI assistant app by Talkie. It provides AI character chat and roleplay-style conversations.",

    # Office & Collaboration
    "Microsoft Teams": "Microsoft Teams is an office & collaboration app by Microsoft. It provides video meetings, chat, calling, and file collaboration for teams.",
    "Zoom": "Zoom is an office & collaboration app by Zoom. It provides video meetings, webinars, chat, and team collaboration features.",
    "Google Meet": "Google Meet is an office & collaboration app by Google. It provides video conferencing integrated with Google Workspace.",
    "Slack": "Slack is an office & collaboration app by Slack, part of Salesforce. It provides team messaging, channels, workflows, and app integrations.",
    "Notion": "Notion is an office & collaboration app by Notion Labs. It provides notes, documents, databases, wikis, and project collaboration.",
    "WPS Office": "WPS Office is an office & collaboration app by Kingsoft. It provides document, spreadsheet, PDF, and presentation editing tools.",
    "Outlook": "Outlook is an office & collaboration app by Microsoft. It provides email, calendar, contacts, and scheduling features.",
    "Evernote": "Evernote is an office & collaboration app by Evernote. It provides note-taking, organization, scanning, and sync across devices.",
    "Google Translate": "Google Translate is an office & collaboration app by Google. It provides text, image, conversation, and voice translation across many languages.",
    "Scribd": "Scribd is an office & collaboration app by Scribd. It provides access to ebooks, audiobooks, documents, and reading content.",
    "Monday.com": "Monday.com is an office & collaboration app by Monday.com. It provides project management, workflows, and team coordination tools.",
    "Microsoft Loop": "Microsoft Loop is an office & collaboration app by Microsoft. It provides collaborative workspaces and reusable components across Microsoft 365 apps.",
    "Microsoft OneNote": "Microsoft OneNote is an office & collaboration app by Microsoft. It provides digital note-taking with notebooks, sections, and media support.",
    "DingTalk": "DingTalk is an office & collaboration app by Alibaba. It provides enterprise chat, meetings, workflow, and attendance tools.",
    "Behance": "Behance is a creative portfolio app by Adobe. It provides a platform for showcasing and discovering design and creative work.",
    "QQ Reader": "QQ Reader is a reading app by Tencent. It provides ebook reading and access to Chinese digital literature.",
    "iReader": "iReader is a reading app by iReader Technology. It provides ebook reading and digital publishing services.",

    # Design & Editing
    "Canva": "Canva is a design & editing app by Canva. It provides drag-and-drop graphic design for social media, presentations, and marketing materials.",
    "CapCut": "CapCut is a design & editing app by ByteDance. It provides video editing with templates, effects, and AI-powered creative tools.",
    "Adobe": "Adobe is a creative software brand by Adobe Inc. On mobile, it typically refers to the Adobe Creative Cloud app for managing creative assets and fonts.",
    "Adobe Lightroom": "Adobe Lightroom is a design & editing app by Adobe. It provides photo editing, RAW processing, and photo library management.",
    "PicsArt": "PicsArt is a design & editing app by PicsArt. It provides photo and video editing with AI tools, stickers, and collage features.",
    "InShot": "InShot is a design & editing app by InShot. It provides mobile video and photo editing for social media content.",
    "KineMaster": "KineMaster is a design & editing app by KineMaster. It provides multi-layer video editing with timeline-based controls.",
    "VSCO": "VSCO is a design & editing app by VSCO. It provides photo and video editing with film-style presets and a creative community.",
    "VivaVideo": "VivaVideo is a design & editing app by QuVideo. It provides video editing with transitions, effects, and music tools.",
    "B612": "B612 is a design & editing app by SNOW. It provides beauty camera, filters, AR effects, and photo or video editing.",
    "BeautyCam": "BeautyCam is a design & editing app by Meitu. It provides beauty filters, retouching, and selfie editing tools.",
    "BeautyPlus": "BeautyPlus is a design & editing app by Pixocial. It provides photo retouching, beauty filters, and editing tools.",
    "Camera360": "Camera360 is a design & editing app by PinGuo. It provides camera filters, beauty effects, and photo editing.",
    "FaceApp": "FaceApp is a design & editing app by FaceApp Technology. It provides AI-based face editing and transformation effects.",
    "FaceU": "FaceU is a design & editing app. It provides beauty camera filters, stickers, and AR face effects.",
    "Happy Color": "Happy Color is a coloring app by X-Flow. It provides a paint-by-numbers coloring experience.",
    "Perfect365": "Perfect365 is a design & editing app by Perfect365. It provides virtual makeup simulation and beauty photo editing.",
    "PhotoGrid": "PhotoGrid is a design & editing app by PhotoGrid. It provides photo collage, slideshow, and video editing tools.",
    "PhotoLab": "PhotoLab is a design & editing app by Linerock Investments. It provides AI photo effects, templates, and artistic edits.",
    "PhotoRoom": "PhotoRoom is a design & editing app by PhotoRoom. It provides AI background removal and product photo editing.",
    "PicCollage": "PicCollage is a design & editing app by Cardinal Blue. It provides collage creation with templates, stickers, and layouts.",
    "Pixelcut": "Pixelcut is a design & editing app by Pixelcut. It provides AI product photo editing and background removal.",
    "Polarr": "Polarr is a design & editing app by Polarr. It provides advanced photo editing with filters and AI enhancements.",
    "Prequel": "Prequel is a design & editing app by Prequel. It provides aesthetic photo and video filters and effects.",
    "QuVideo": "QuVideo is a video editing app publisher brand. It develops editing apps including VivaVideo.",
    "Remini": "Remini is a design & editing app by Bending Spoons. It provides AI photo enhancement, restoration, and upscaling.",
    "SnapEdit": "SnapEdit is a design & editing app by SnapEdit. It provides AI-powered object removal and photo cleanup.",
    "Ulike": "Ulike is a design & editing app. It provides beauty camera and selfie enhancement tools.",
    "UpFoto": "UpFoto is a design & editing app. It provides photo editing and enhancement tools.",
    "VlogNow": "VlogNow is a design & editing app, now commonly branded as VN Video Editor. It provides mobile video editing with templates, effects, and timeline controls.",
    "XingTu": "XingTu is a design & editing app by ByteDance. It provides photo editing and beauty enhancement tools for the Chinese market.",

    # Browser & Search
    "Google Search": "Google Search is a browser & search app by Google. It provides web search, voice search, and AI-assisted answer features.",
    "Microsoft Bing": "Microsoft Bing is a browser & search app by Microsoft. It provides web search with AI chat and image generation features.",
    "UC Browser": "UC Browser is a browser & search app by UCWeb. It provides mobile web browsing with data compression and content discovery.",
    "Opera": "Opera is a browser & search app by Opera. It provides web browsing with built-in privacy, ad blocking, and browser tools.",
    "Brave Browser": "Brave Browser is a browser & search app by Brave Software. It provides privacy-focused browsing with built-in ad and tracker blocking.",
    "Mozilla Firefox": "Mozilla Firefox is a browser & search app by Mozilla. It provides open-source web browsing with privacy-focused features.",
    "DuckDuckGo": "DuckDuckGo is a browser & search app by DuckDuckGo. It provides privacy-focused search and browsing tools.",
    "Baidu Browser": "Baidu Browser is a browser & search app by Baidu. It provides mobile web browsing with AI-powered search and content discovery.",
    "QQ Browser": "QQ Browser is a browser & search app by Tencent. It provides web browsing and search services for Chinese users.",
    "Quark Browser": "Quark Browser is a browser & search app by UCWeb. It provides a minimalist browser with search and AI-assisted features.",
    "Yahoo Search": "Yahoo Search is a browser & search app by Yahoo. It provides web search and personalized news, featuring Yahoo Scout for AI-powered answers.",
    "Yandex Search": "Yandex Search is a search app by Yandex. It provides web search and related information services for Russian-language users.",

    # Cloud Storage & File Sharing
    "Google Drive": "Google Drive is a cloud storage app by Google. It provides file storage, sharing, and collaboration across Google Workspace files.",
    "Dropbox": "Dropbox is a cloud storage app by Dropbox. It provides file storage, sync, sharing, and collaboration across devices.",
    "Microsoft OneDrive": "Microsoft OneDrive is a cloud storage app by Microsoft. It provides file storage and sync integrated with Microsoft 365.",
    "Apple iCloud": "Apple iCloud is a cloud storage service by Apple. It provides file, photo, backup, and sync services across Apple devices.",
    "MEGA": "MEGA is a cloud storage app by MEGA. It provides encrypted cloud storage and file sharing.",
    "MediaFire": "MediaFire is a cloud storage app by MediaFire. It provides file hosting, storage, and direct-link sharing.",
    "4shared": "4shared is a cloud storage app by 4shared. It provides file storage, sharing, and media access features.",
    "Samsung Cloud": "Samsung Cloud is a cloud storage service by Samsung. It provides backup and sync for supported Samsung device data.",
    "SHAREit": "SHAREit is a file sharing app by Smart Media4U Technology. It provides peer-to-peer offline file transfer between devices.",
    "Baidu Cloud": "Baidu Cloud is a cloud storage service by Baidu. It provides cloud storage and file synchronization for Chinese users.",
    "Go2Cloud": "Go2Cloud is a cloud infrastructure service app providing management tools for IaaS solutions, VPS servers, and cloud instances.",
    "Honor Cloud": "Honor Cloud is a cloud storage service by Honor. It provides device backup and cloud sync for Honor users.",
    "Huawei Cloud": "Huawei Cloud is a cloud storage service by Huawei. It provides backup and sync services for Huawei devices.",
    "Photobucket": "Photobucket is a cloud storage app by Photobucket. It provides photo and video hosting and sharing.",
    "Yandex Disk": "Yandex Disk is a cloud storage app by Yandex. It provides cloud storage and file synchronization.",

    # System & Utility
    "Adobe Acrobat Reader": "Adobe Acrobat Reader is a system & utility app by Adobe. It provides PDF viewing, annotation, signing, and form filling.",
    "NordVPN": "NordVPN is a system & utility app by Nord Security. It provides VPN encryption and privacy protection for internet traffic.",
    "Truecaller": "Truecaller is a system & utility app by True Software Scandinavia. It provides caller ID, spam blocking, and phone number lookup.",
    "GetContact": "GetContact is a system & utility app by GetContact. It provides caller identification and spam detection based on tagged phone numbers.",
    "CamScanner": "CamScanner is a system & utility app by IntSig. It provides document scanning, OCR, and PDF creation.",
    "AnyDesk": "AnyDesk is a system & utility app by AnyDesk. It provides remote desktop access and device control.",
    "Microsoft SwiftKey": "Microsoft SwiftKey is a system & utility app by Microsoft. It provides a smart keyboard with swipe typing and multilingual prediction.",
    "Google Lens": "Google Lens is a system & utility app by Google. It provides visual search, OCR, and translation using the camera.",
    "AnchorFree": "AnchorFree is a system & utility brand formerly associated with Hotspot Shield VPN, now under Aura. It provides VPN-related privacy services.",
    "DroidVPN": "DroidVPN is a system & utility app by DroidVPN. It provides VPN proxy services for Android devices.",
    "GOOSE VPN": "GOOSE VPN is a system & utility app by GOOSE VPN. It provides VPN privacy and encrypted browsing services.",
    "Hamachi": "Hamachi is a networking utility by LogMeIn. It provides virtual private network connectivity for remote access and private networks.",
    "Hola VPN": "Hola VPN is a system & utility app by Hola. It provides VPN and proxy-based access services.",
    "Online Video Converter": "Online Video Converter is a system & utility app. It provides media conversion and video download-related tools.",
    "OpenVPN": "OpenVPN is a system & utility app by OpenVPN. It provides an open-source VPN client for secure networking.",
    "Opera VPN": "Opera VPN is a system & utility feature by Opera that provides built-in virtual private network services for secure and private web browsing.",
    "PDF Scanner": "PDF Scanner is a system & utility app. It provides document scanning and PDF conversion tools.",
    "Period Calendar": "Period Calendar is a health tracking utility app. It provides menstrual cycle tracking and fertility prediction.",
    "PrivateVPN": "PrivateVPN is a system & utility app by PrivateVPN. It provides encrypted VPN tunneling services.",
    "PureVPN": "PureVPN is a system & utility app by PureVPN. It provides VPN privacy and secure internet access services.",
    "Screen Recorder & Video Recorder": "Screen Recorder & Video Recorder is a system & utility app. It provides screen recording and video capture tools on mobile devices.",
    "Sticker.ly": "Sticker.ly is a system & utility app by Sticker.ly. It provides custom sticker creation and sharing for messaging apps.",
    "TLS ESNI": "TLS ESNI is a system & utility tool used to enable Encrypted Server Name Indication, helping users bypass SNI-based filtering and improve privacy.",
    "TunnelBear": "TunnelBear is a system & utility app by TunnelBear. It provides user-friendly VPN services for private browsing.",
    "Turbo VPN": "Turbo VPN is a system & utility app. It provides VPN proxy services for mobile devices.",
    "VPN Master": "VPN Master is a system & utility app. It provides VPN proxy services for mobile devices.",
    "VPN Super Unlimited Proxy": "VPN Super Unlimited Proxy is a system & utility app. It provides VPN and proxy services for mobile devices.",
    "VPN Unlimited": "VPN Unlimited is a system & utility app by KeepSolid. It provides VPN encryption and secure connection services.",

    # Business Operations
    "Moka POS": "Moka POS is a business operations app by PT Moka Teknologi Indonesia. It provides point-of-sale, inventory, and business reporting tools.",
    "Majoo": "Majoo is a business operations app by PT Majoo Teknologi Indonesia. It provides POS, CRM, inventory, and business management tools for MSMEs.",
    "BukuWarung": "BukuWarung is a business operations app by PT Buku Warung Digital Indonesia. It provides bookkeeping, invoicing, and payment tools for micro and small businesses.",
    "Kasir Pintar": "Kasir Pintar is a business operations app by Kasir Pintar. It provides cashier, POS, stock, and sales reporting tools for small businesses.",
    "Pawoon": "Pawoon is a business operations app by PT Pawoon Inspirasional Indonesia. It provides POS and business management tools for retail and restaurants.",
    "Qasir": "Qasir is a business operations app by PT Solusi Pasar Digital. It provides POS and business management tools for MSMEs.",
    "Payfazz": "Payfazz is a business operations app by Fazz Financial Group. It provides agent banking and digital financial service distribution.",
    "KasPro": "KasPro is a business operations app. It provides digital payment and financial service tools for businesses.",
    "Ameera Kasir": "Ameera Kasir is a business operations app. It provides point-of-sale and cashier management for small businesses.",
    "Cashlez": "Cashlez is a business operations app by PT Cashlez Worldwide Indonesia. It provides payment acceptance and merchant services for businesses.",
    "Cazh POS": "Cazh POS is a business operations app. It provides POS and cashier tools for small businesses.",
    "Clover": "Clover is a business operations app by Clover, part of Fiserv. It provides POS software and merchant tools for retailers and restaurants.",
    "Finata": "Finata is a business operations app by PT Finata Inovasi Keuangan. It provides bookkeeping and financial management tools for SMEs.",
    "Freshdesk": "Freshdesk is a business operations app by Freshworks. It provides helpdesk, ticketing, and customer support management.",
    "Laris POS": "Laris POS is a business operations app. It provides point-of-sale management for retail businesses.",
    "LiveAgent": "LiveAgent is a business operations app by Quality Unit. It provides helpdesk, ticketing, and live chat customer support tools.",
    "Raptor POS": "Raptor POS is a business operations app. It provides point-of-sale services for merchants.",
    "Restau POS": "Restau POS is a business operations app. It provides restaurant-focused POS and order management tools.",
    "SalesPlay POS": "SalesPlay POS is a business operations app by SalesPlay. It provides POS, inventory, and staff management for merchants.",
    "Sobot": "Sobot is a business operations app by Sobot. It provides AI-powered customer service and contact center solutions.",
    "Vireo POS": "Vireo POS is a business operations app. It provides point-of-sale management tools for small businesses.",
    "2DFire": "2DFire is a business operations app by 2DFire. It provides restaurant POS and kitchen or order management tools.",
    "South China Morning Post": "South China Morning Post is a news app by SCMP. It provides Hong Kong, China, and Asia-focused journalism.",
    "The Washington Post": "The Washington Post is a news app by The Washington Post. It provides American and international journalism, analysis, and opinion.",
    "Wix": "Wix is a business operations app by Wix. It provides website building, hosting, and online store management for businesses.",
    "Xfers": "Xfers is a business operations and payments infrastructure brand, now part of Fazz. It provides payment and financial API services for businesses.",

    # Developer Tools
    "GitHub": "GitHub is a developer tools app by GitHub, part of Microsoft. It provides repository hosting, code review, issues, and collaboration for developers.",
    "Visual Studio": "Visual Studio is a developer tool by Microsoft. It provides an integrated development environment for building software applications.",
    "TeamViewer": "TeamViewer is a remote access and support app by TeamViewer. It provides remote desktop, device control, and support tools.",
    "Salesforce": "Salesforce is a business platform app by Salesforce. It provides CRM, sales, service, and enterprise workflow tools.",
    "ArcGIS": "ArcGIS is a geospatial platform app by Esri. It provides GIS mapping, field data collection, and spatial analytics tools.",
    "Autodesk": "Autodesk is a design and engineering software brand by Autodesk. It provides CAD, 3D modeling, and construction or engineering tools.",
    "Mapbox": "Mapbox is a developer platform by Mapbox. It provides mapping and location APIs for building map-based applications.",
    "MathWorks": "MathWorks is a technical computing software brand by MathWorks. It provides MATLAB and Simulink tools for computing and simulation.",
    "Radmin": "Radmin is a remote administration tool by Famatech. It provides secure remote computer access and control.",
    "Xero": "Xero is an accounting app by Xero. It provides cloud-based bookkeeping, invoicing, and business finance management.",
    "Zahir": "Zahir is an accounting app by Zahir Internasional. It provides accounting and bookkeeping software for Indonesian businesses.",
    "ShareSDK": "ShareSDK is a developer SDK by MobTech. It provides social sharing and third-party login integration for app developers.",

    # File Viewer & Reader
    "Google Photos": "Google Photos is a file viewer & reader app by Google. It provides photo and video backup, organization, and AI-powered search.",
}

print(f"DESC_PRODUCTIVITY: {len(DESC_PRODUCTIVITY)} entries")

DESC_PRODUCTIVITY: 158 entries


In [130]:
# =============================================================================
# DESCRIPTIONS DICTIONARY — Platform & System + Other
# =============================================================================
DESC_PLATFORM = {
    # App Store
    "Google Play": "Google Play is an app store by Google. It distributes Android apps, games, books, and digital content.",
    "Apple App Store": "Apple App Store is an app store by Apple. It distributes iPhone, iPad, and other Apple-platform apps and subscriptions.",
    "Huawei AppGallery": "Huawei AppGallery is an app store by Huawei. It distributes apps for Huawei devices as an alternative to Google Play services.",
    "Samsung Galaxy Store": "Samsung Galaxy Store is an app store by Samsung. It distributes apps, themes, and device-specific content for Samsung Galaxy users.",
    "APKPure": "APKPure is an alternative Android app marketplace. It provides APK downloads and app distribution outside Google Play.",
    "APKMirror": "APKMirror is an alternative Android app repository by Illogical Robot. It provides verified APK downloads and version archives for Android apps.",
    "9Apps": "9Apps is an alternative Android app marketplace by UCWeb. It provides app and game downloads outside Google Play.",
    "OPPO App Market": "OPPO App Market is an app store by OPPO. It distributes apps for OPPO smartphone users.",
    "Vivo App Store": "Vivo App Store is an app store by Vivo. It distributes apps for Vivo smartphone users.",
    "Xiaomi App Store": "Xiaomi App Store is an app store by Xiaomi. It distributes apps for Xiaomi, Redmi, and related device users.",
    "Honor App Store": "Honor App Store is an app store by Honor. It distributes apps for Honor device users.",
    "Lenovo App Store": "Lenovo App Store is an app store by Lenovo. It distributes apps for Lenovo device users.",
    "Cafe Bazaar": "Cafe Bazaar is an app store by Cafe Bazaar. It is a major Android app marketplace in Iran.",
    "HappyMod": "HappyMod is an alternative app store. It provides modified Android apps and games.",
    "QooApp": "QooApp is an app store focused on Asian mobile games. It distributes Japanese, Korean, Chinese, and anime-related mobile game titles.",
    "iTunes": "iTunes is a digital media platform by Apple. It provides music, movie, and podcast purchasing and library management rather than a mobile app store.",
    "Google Play Games": "Google Play Games is a gaming platform app by Google. It provides game achievements, cloud saves, and access to supported Android games on mobile and PC.",
    "Carousell": "Carousell is a marketplace app by Carousell. It provides a platform for buying and selling secondhand goods.",
    "DeviantArt": "DeviantArt is a creative community platform by DeviantArt. It provides artwork sharing, discovery, and creator community features.",
    "Dreamstime": "Dreamstime is a stock media platform by Dreamstime. It provides royalty-free stock photography and image licensing.",
    "Envato": "Envato is a digital asset marketplace by Envato. It provides themes, templates, graphics, audio, and creative resources.",
    "Flickr": "Flickr is a photo platform by SmugMug. It provides photo hosting, sharing, and community features for photographers.",
    "Freepik": "Freepik is a digital asset platform by Freepik Company. It provides graphic resources, vectors, templates, and stock images.",
    "Pixabay": "Pixabay is a stock media platform by Canva. It provides free stock photos, videos, and illustrations.",
    "Shutterstock": "Shutterstock is a stock media platform by Shutterstock. It provides licensed stock photos, videos, music, and creative assets.",
    "Unsplash": "Unsplash is a stock photography platform by Getty Images. It provides free high-resolution photography.",

    # Telco Self-Care
    "Indosat Ooredoo Hutchison": "Indosat Ooredoo Hutchison is a telco self-care app by PT Indosat Ooredoo Hutchison, commonly through the myIM3 app. It provides account management, package purchases, and customer services for IM3 users.",
    "Telkomsel": "Telkomsel is a telco self-care app by PT Telekomunikasi Selular, commonly through the MyTelkomsel app. It provides account management, top-ups, package purchases, and digital services.",
    "XL Axiata": "XL Axiata is a telco self-care app by PT XL Axiata, commonly through the myXL app. It provides account management, top-ups, quota checks, and package purchases.",
    "Tri Indonesia": "Tri Indonesia is a telco self-care app by PT Hutchison 3 Indonesia, commonly through the bima+ app. It provides account management, package purchases, and usage tracking.",
    "AXIS": "AXIS is a telco self-care app by PT XL Axiata, commonly through the AXISnet app. It provides prepaid account management, quota checks, and data package purchases.",
    "Orbit": "Orbit is a telco self-care app by Telkomsel. It provides management for Orbit home internet devices, usage, and package services.",
    "MyTelkom": "MyTelkom is a telco self-care app by Telkom Indonesia. It provides account management and service access for Telkom consumer products.",
    "Biznet": "Biznet is a telco self-care app by Biznet Networks. It provides internet service account management, billing, and customer support tools.",
    "MyDigi": "MyDigi is a telco self-care app by Digi Telecommunications, now under CelcomDigi. It provides account management, bill payment, and internet plan purchases for Malaysian users.",
    "MyKorek": "MyKorek is a telco self-care app by Korek Telecom. It provides mobile account management and recharge services.",
    "RITA": "RITA is a telco self-care virtual assistant by Indosat Ooredoo Hutchison providing automated support for IM3 and Tri users.",
    "Simpel": "Simpel is a digital service utility used for account management and service registration within telco or government ecosystems.",

    # Government
    "Mobile JKN": "Mobile JKN is a government app by BPJS Kesehatan. It provides national health insurance membership services, facility access, and queue booking in Indonesia.",
    "BPJS Ketenagakerjaan": "BPJS Ketenagakerjaan is a government app by BPJS Ketenagakerjaan. It provides employment social security account management and claims-related services.",
    "PeduliLindungi": "PeduliLindungi was a government health app by the Indonesian government. It provided COVID-19 certificates, check-in features, and public health protocol services.",
    "DIKTI": "DIKTI is a government education app associated with Indonesia's higher education system. It provides higher education information and related academic services.",
    "M-Paspor": "M-Paspor is a government app by Indonesia's Directorate General of Immigration. It provides passport application and queue booking services.",
    "KlikPajak": "KlikPajak is a tax compliance app by PT Mekari Bayar Pajak. It provides Indonesian tax filing, e-invoicing, and business tax management services.",
    "Jasa Marga": "Jasa Marga is a transport services app by PT Jasa Marga. It provides toll road information, traffic updates, and related services.",
    "Embassies": "Embassies is a government and reference app providing directory information and contact details for diplomatic missions worldwide.",

    # Weather Service
    "AccuWeather": "AccuWeather is a weather service app by AccuWeather. It provides forecasts, radar, and severe weather alerts.",
    "The Weather Channel": "The Weather Channel is a weather service app by The Weather Company. It provides forecasts, radar, and severe weather updates.",
    "Windy": "Windy is a weather service app by Windy.com. It provides animated weather maps with wind, rain, temperature, and wave layers.",
    "Amber Weather": "Amber Weather is a weather service app. It provides forecasts, widgets, and weather alerts.",
    "MJ Weather": "MJ Weather is a weather service app, specifically the native weather application for Meizu smartphone users.",
    "NOAA Weather": "NOAA Weather is a weather service app providing forecasts and severe weather alerts based on National Oceanic and Atmospheric Administration data.",
    "OpenWeatherMap": "OpenWeatherMap is a weather data platform by OpenWeather. It provides weather forecasts, current conditions, and developer APIs.",
    "Weather Reader": "Weather Reader is a weather service utility that aggregates and reads out loud meteorological data and updates.",
    "Weather.com": "Weather.com is a weather service platform by The Weather Company. It provides forecasts, radar, and weather information.",
    "Weather.gov": "Weather.gov is the official weather platform of the US National Weather Service. It provides forecasts, alerts, and weather warnings.",
    "WeatherCN": "WeatherCN is a weather service app focused on China. It provides weather forecasts for Chinese locations.",

    # Energy & EV
    "Shell": "Shell is an energy & EV app by Shell. It provides fuel station locator, payment, rewards, and EV charging services through Shell and Shell Recharge.",
}

DESC_OTHER = {
    # Tobacco Device
    "IQOS": "IQOS is a tobacco device app by Philip Morris International. It connects to compatible IQOS devices for settings, personalization, firmware updates, and support.",
    "FOOM": "FOOM is a tobacco and lifestyle app for managing FOOM e-cigarette devices and accessing community loyalty rewards.",
    "Bentoel Group": "Bentoel Group is a business operations app providing B2B distribution and retailer support for their tobacco brands.",
    "Djarum": "Djarum is a business operations app for B2B logistics and retailer management within the Djarum distribution network.",
    "Gudang Garam": "Gudang Garam is a business operations app providing B2B supply chain and wholesale management for its products.",
    "Indonesian Tobacco": "Indonesian Tobacco is a business operations app focused on the distribution and supply chain management of tobacco products.",
    "Sampoerna": "Sampoerna is a business operations app providing B2B retail management and empowerment tools, such as the Ayo SRC platform.",
}

print(f"DESC_PLATFORM: {len(DESC_PLATFORM)} entries")
print(f"DESC_OTHER: {len(DESC_OTHER)} entries")

DESC_PLATFORM: 58 entries
DESC_OTHER: 7 entries


In [131]:
# =============================================================================
# Merge all description dicts and apply to output DataFrame
# =============================================================================
DESCRIPTIONS = {}
for d in [DESC_FINANCE, DESC_COMMERCE, DESC_COMMUNICATION, DESC_ENTERTAINMENT,
          DESC_HEALTH, DESC_INFO_EDU, DESC_LIFESTYLE, DESC_TRANSPORTATION,
          DESC_PRODUCTIVITY, DESC_PLATFORM, DESC_OTHER]:
    DESCRIPTIONS.update(d)

print(f"Total descriptions: {len(DESCRIPTIONS)}")

# Apply to output DataFrame
output["description"] = output["app_name"].map(DESCRIPTIONS).fillna("")

# ── Flag fallback_original rows in description ─────────────────────────────────
# Rows where no sig-library match was found use original source tags as-is.
# Append a note so downstream consumers know these are not confirmed matches.
_fallback_mask = mis["_match_source"] == "fallback_original"
_SIG_NOT_FOUND_NOTE = " (App not found in Signature Apps Library — tags reflect original source identifiers.)"

output.loc[_fallback_mask, "description"] = output.loc[_fallback_mask, "description"].apply(
    lambda d: (d.rstrip() + _SIG_NOT_FOUND_NOTE) if d else _SIG_NOT_FOUND_NOTE.lstrip()
)

fallback_count = _fallback_mask.sum()
print(f"\nFallback rows flagged with sig-library note: {fallback_count}")

filled = (output["description"] != "").sum()
empty  = (output["description"] == "").sum()
print(f"\nDescriptions filled: {filled}/{len(output)} ({filled/len(output)*100:.1f}%)")
print(f"Empty (needs manual review): {empty}")

if empty > 0:
    missing = output[output["description"] == ""][["app_name", "category", "subcategory"]]
    print(f"\n--- Apps without descriptions ({empty}) ---")
    for _, row in missing.iterrows():
        print(f"  {row['app_name']} | {row['category']} > {row['subcategory']}")


Total descriptions: 1201

Fallback rows flagged with sig-library note: 88

Descriptions filled: 1199/1199 (100.0%)
Empty (needs manual review): 0


In [103]:

# Ammana, Asetku, and Touch 'n Go descriptions are now inline in DESC_FINANCE above.


In [104]:

# Touch 'n Go unicode patch consolidated into DESC_FINANCE above.


In [105]:

# All description corrections are now inline in DESC_FINANCE above.
# This cell is intentionally empty.


In [132]:

# =============================================================================
# PRIMARY CATEGORY RECLASSIFICATIONS
# Fix wrong category/subcategory for 18 apps.
# All values are safe snake_case (labels already normalised at this point).
# =============================================================================

# (app_name, old_cat, old_sub, new_cat, new_sub)
# old_cat/old_sub are display-only; new_cat/new_sub are actually written.
CATEGORY_FIXES = [
    # ── finance > bnpl_pay_later (new subcategory) ────────────────────────────
    ("Kredivo",      "finance", "e_wallet",        "finance", "bnpl_pay_later"),
    ("Atome",        "finance", "e_wallet",        "finance", "bnpl_pay_later"),
    ("Akulaku",      "finance", "e_wallet",        "finance", "bnpl_pay_later"),
    ("Klarna",       "finance", "payment_gateway", "finance", "bnpl_pay_later"),
    ("YUP",          "finance", "payment_gateway", "finance", "bnpl_pay_later"),
    ("KawanCicil",   "finance", "payment_gateway", "finance", "bnpl_pay_later"),
    ("Home Credit",  "finance", "payment_gateway", "finance", "bnpl_pay_later"),

    # ── finance > p2p_lending ─────────────────────────────────────────────────
    ("Indosaku",     "finance", "e_wallet",        "finance", "p2p_lending"),
    ("AdaKami",      "finance", "payment_gateway", "finance", "p2p_lending"),
    ("KrediFazz",    "finance", "payment_gateway", "finance", "p2p_lending"),
    ("PinjamDuit",   "finance", "payment_gateway", "finance", "p2p_lending"),
    ("UKU",          "finance", "payment_gateway", "finance", "p2p_lending"),
    ("KlikKami",     "finance", "payment_gateway", "finance", "p2p_lending"),
    ("UangMe",       "finance", "payment_gateway", "finance", "p2p_lending"),
    ("Kredito",      "finance", "payment_gateway", "finance", "p2p_lending"),
    ("DanaCita",     "finance", "payment_gateway", "finance", "p2p_lending"),
    ("Findaya",      "finance", "payment_gateway", "finance", "p2p_lending"),

    # ── finance > payment_gateway (from e_wallet) ─────────────────────────────
    ("BestPay",      "finance", "e_wallet",        "finance", "payment_gateway"),
]

applied = 0
not_found = []
for app_name, old_cat, old_sub, new_cat, new_sub in CATEGORY_FIXES:
    mask = output["app_name"] == app_name
    if not mask.any():
        not_found.append(app_name)
        print(f"  not found: {app_name}")
        continue
    actual_cat = output.loc[mask, "category"].iloc[0]
    actual_sub = output.loc[mask, "subcategory"].iloc[0]
    output.loc[mask, "category"]    = new_cat
    output.loc[mask, "subcategory"] = new_sub
    applied += 1
    print(f"  {app_name}: {actual_cat} > {actual_sub}  →  {new_cat} > {new_sub}")

print(f"\nCategory fixes applied: {applied} / {len(CATEGORY_FIXES)}")
if not_found:
    print(f"Not found ({len(not_found)}): {not_found}")

bnpl_count = (output["subcategory"] == "bnpl_pay_later").sum()
print(f"\nfinance > bnpl_pay_later: {bnpl_count} apps")


  Kredivo: finance > e_wallet  →  finance > bnpl_pay_later
  Atome: finance > e_wallet  →  finance > bnpl_pay_later
  Akulaku: finance > e_wallet  →  finance > bnpl_pay_later
  Klarna: finance > payment_gateway  →  finance > bnpl_pay_later
  YUP: finance > payment_gateway  →  finance > bnpl_pay_later
  KawanCicil: finance > payment_gateway  →  finance > bnpl_pay_later
  Home Credit: finance > payment_gateway  →  finance > bnpl_pay_later
  Indosaku: finance > e_wallet  →  finance > p2p_lending
  AdaKami: finance > payment_gateway  →  finance > p2p_lending
  KrediFazz: finance > payment_gateway  →  finance > p2p_lending
  PinjamDuit: finance > payment_gateway  →  finance > p2p_lending
  UKU: finance > payment_gateway  →  finance > p2p_lending
  KlikKami: finance > payment_gateway  →  finance > p2p_lending
  UangMe: finance > payment_gateway  →  finance > p2p_lending
  Kredito: finance > payment_gateway  →  finance > p2p_lending
  DanaCita: finance > payment_gateway  →  finance > p2p_lend

In [ ]:

# =============================================================================
# SECONDARY CATEGORY + SUBCATEGORY LABELS  (Option B)
# All values are safe snake_case.
# =============================================================================

output["secondary_category"]    = ""
output["secondary_subcategory"] = ""

SECONDARY_LABELS = {
    # ── Finance super-apps ────────────────────────────────────────────────────
    "GoPay":           ("transportation",      "ride_hailing"),
    "OVO":             ("finance",             "investment"),
    "LinkAja":         ("transportation",      "public_transport_ticketing"),
    "Kredivo":         ("finance",             "e_wallet"),
    "Akulaku":         ("finance",             "e_wallet"),
    "Indodana":        ("finance",             "p2p_lending"),
    "KoinWorks":       ("finance",             "p2p_lending"),
    "Tanamduit":       ("finance",             "e_wallet"),
    "Traveloka":       ("finance",             "bnpl_pay_later"),
    "PayTren":         ("information_education", "religious"),

    # ── Commerce platforms with embedded wallets ──────────────────────────────
    "Shopee":          ("finance",             "e_wallet"),
    "Tokopedia":       ("finance",             "e_wallet"),
    "Lazada":          ("finance",             "e_wallet"),
    "Bukalapak":       ("commerce",            "seller_tools"),
    "TikTok":          ("commerce",            "marketplace"),
    "Shopify":         ("productivity_tools",  "business_operations"),
    "JD.ID":           ("lifestyle",           "food_delivery"),

    # ── Transportation super-apps ─────────────────────────────────────────────
    "Gojek":           ("lifestyle",           "food_delivery"),
    "Grab":            ("lifestyle",           "food_delivery"),
    "AirAsia":         ("lifestyle",           "food_delivery"),

    # ── Entertainment cross-category ──────────────────────────────────────────
    "YouTube":         ("entertainment",       "music_streaming"),
    "Twitch":          ("entertainment",       "gaming_platform"),
    "Discord":         ("entertainment",       "gaming_platform"),
    "BIGO Live":       ("communication",       "social_network"),
    "Nimo TV":         ("entertainment",       "gaming_platform"),
    "Smule":           ("entertainment",       "music_streaming"),
    "Kahoot!":         ("information_education", "general_education"),
    "TikTok Seller":   ("communication",       "short_video_live"),

    # ── Productivity ──────────────────────────────────────────────────────────
    "Canva":           ("productivity_tools",  "business_operations"),
    "Notion":          ("productivity_tools",  "business_operations"),
    "GitHub":          ("productivity_tools",  "office_collaboration"),
    "Salesforce":      ("productivity_tools",  "business_operations"),
    "Wix":             ("commerce",            "seller_tools"),

    # ── Health ────────────────────────────────────────────────────────────────
    "Strava":          ("communication",       "social_network"),
    "Sweatcoin":       ("finance",             "e_wallet"),
    "Halodoc":         ("commerce",            "marketplace"),
    "K24 Klik Apotek": ("commerce",            "marketplace"),

    # ─── Manual Assignment ────────────────────────────────────────────
    "BabyBus":         ("health_wellness",            "maternal_family"),
    "Orami":           ("health_wellness",            "maternal_family"),
    "Teman Bumil":     ("health_wellness",            "maternal_family"),
    "Pregnancy Tracker":     ("health_wellness",            "maternal_family"),
    "Buku Bumil":     ("health_wellness",            "maternal_family"),
}

sec_applied = 0
sec_not_found = []
for app_name, (sec_cat, sec_sub) in SECONDARY_LABELS.items():
    mask = output["app_name"] == app_name
    if not mask.any():
        sec_not_found.append(app_name)
        print(f"  not found: {app_name}")
        continue
    output.loc[mask, "secondary_category"]    = sec_cat
    output.loc[mask, "secondary_subcategory"] = sec_sub
    sec_applied += 1
    print(f"  {app_name}: secondary = {sec_cat} > {sec_sub}")

print(f"\nSecondary labels applied: {sec_applied} / {len(SECONDARY_LABELS)}")
if sec_not_found:
    print(f"Not found ({len(sec_not_found)}): {sec_not_found}")

print(f"\nColumns : {list(output.columns)}")
print(f"Rows    : {len(output)}")


  GoPay: secondary = transportation > ride_hailing
  OVO: secondary = finance > investment
  LinkAja: secondary = transportation > public_transport_ticketing
  Kredivo: secondary = finance > e_wallet
  Akulaku: secondary = finance > e_wallet
  Indodana: secondary = finance > p2p_lending
  KoinWorks: secondary = finance > p2p_lending
  Tanamduit: secondary = finance > e_wallet
  Traveloka: secondary = finance > bnpl_pay_later
  PayTren: secondary = information_education > religious
  Shopee: secondary = finance > e_wallet
  Tokopedia: secondary = finance > e_wallet
  Lazada: secondary = finance > e_wallet
  Bukalapak: secondary = commerce > seller_tools
  TikTok: secondary = commerce > marketplace
  Shopify: secondary = productivity_tools > business_operations
  JD.ID: secondary = lifestyle > food_delivery
  Gojek: secondary = lifestyle > food_delivery
  Grab: secondary = lifestyle > food_delivery
  AirAsia: secondary = lifestyle > food_delivery
  YouTube: secondary = entertainment > mu

In [134]:

# =============================================================================
# REGENERATE taxonomy_reference.csv
# Rebuilds counts from primary category/subcategory only (not secondary).
# =============================================================================

taxonomy_new = (
    output
    .groupby(["category", "subcategory"], sort=False)
    .size()
    .reset_index(name="app_count")
    .sort_values(["category", "app_count"], ascending=[True, False])
    .reset_index(drop=True)
)
taxonomy_new["taxonomy_version"] = TAXONOMY_VERSION

taxonomy_new.to_csv(TAXONOMY_REF, index=False, encoding="utf-8")
print(f"taxonomy_reference.csv saved — {len(taxonomy_new)} subcategories  →  {TAXONOMY_REF}")
print()
print(taxonomy_new.to_string(index=False))


taxonomy_reference.csv saved — 66 subcategories  →  /Users/mac/Documents/GitHub/DS-IOH-Application-Mapping/taxonomy_reference.csv

             category                subcategory  app_count taxonomy_version
             commerce                marketplace         40             v2.1
             commerce                    fashion         36             v2.1
             commerce          automotive_market         11             v2.1
             commerce                home_living         10             v2.1
             commerce                    grocery          8             v2.1
             commerce               seller_tools          7             v2.1
             commerce                        b2b          3             v2.1
             commerce            review_platform          2             v2.1
             commerce                   property          2             v2.1
        communication             social_network         22             v2.1
        communication 

In [135]:
output = output[["app_name", "source_app_names_old", "sig_app_tags",
                  "category", "subcategory",
                  "secondary_category", "secondary_subcategory",  "description"]]


In [136]:
# =============================================================================
# Re-save rnr_app_category_v2.csv with descriptions
# =============================================================================

output.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"✅ Saved with descriptions: {OUTPUT_CSV}")
print(f"   Rows: {len(output)} | Columns: {list(output.columns)}")

# Quick sample
output[output["description"] != ""].sample(5, random_state=42)[["app_name", "category", "subcategory", "description"]]

✅ Saved with descriptions: /Users/mac/Documents/GitHub/DS-IOH-Application-Mapping/rnr_app_category_v2.csv
   Rows: 1199 | Columns: ['app_name', 'source_app_names_old', 'sig_app_tags', 'category', 'subcategory', 'secondary_category', 'secondary_subcategory', 'description']


,app_name,category,subcategory,description
1177,beIN SPORTS,entertainment,video_streaming,beIN SPORTS is a video streaming app by beIN M...
864,Quora,communication,social_network,Quora is a social network app by Quora Inc. It...
101,B612,productivity_tools,design_editing,B612 is a design & editing app by SNOW. It pro...
439,HERE WeGo,transportation,navigation_maps,HERE WeGo is a navigation & maps app by HERE T...
58,Allo Bank,finance,mobile_banking,Allo Bank is a mobile banking app by Allo Bank...
